*[notebook only]*

# Block detection pipeline: LEVEL 1 notebook

**LVL0 → LVL1 → LVL2.** Block detection (16 change-point detectors plus fixed-block controls) and external validation. Set `LVL_1_CONFIG` (`[cell 9]`) from the LVL_1 sweep before the final run.

- **Conditions and cohorts:** the manuscript calls the two conditions cohorts: T2 = Cohort A (CA), T3 = Cohort B (CB). The code, file names, exports and the final report keep T2 and T3; only `make_replication_figures.ipynb` prints CA and CB.
- **Input:** the stamped LVL_0 handoff named by `LVL0_HANDOFF` (first code cell; no default). The handoff check stops unless it exists and has exactly one LVL_0 run log, which records its configuration.
- **Calibration:** `LVL1_SWEEP` (first code cell) names the LVL_1 sweep; the calibration check after `[cell 9]` stops unless `LVL_1_CONFIG` holds the LVL1 sweep report's selection (`<sweep>_selection.json`).
- **Output:** the stamped handoff `lv1_out_<ts>_{long,features,results}.csv` + `_meta.json` and its run log `lv1_out_<ts>_parameters.json`, in `LVL_OUTS` (`[cell 2]`). The lineage cell adds the LVL_0 handoff it was built from and the raw input's SHA-256 to the `_meta.json`. Pin the prefix in the LVL2 notebook.
- **Exports for the reports** (notebook only, after the lineage cell): H1 in T2+T3 calibration, T3 only and the pooled datasets (descriptives; every pair of detectors tested in T2+T3 calibration and in T3 only), `lv1_out_<ts>_h1_{desc_overall, desc_winner, desc_detector, tests, ranks, summary, method_association, mahalanobis}.csv`; the descriptives, `lv1_out_<ts>_report_{participant_metrics, age_continuous, age_groups, age_participants, session_features, table_nonparametric}.csv`.
- Every code cell starts with a label. A `# [cell N]` cell is a pipeline cell: N is its pipeline-wide number, also its `NOTEBOOK CELL n` banner in `lvl1.py`. A `# [notebook only]` cell is never copied into a script.
- `[cell 0]`–`[cell 2]` are shared copies; the build stops if they differ.

In [ ]:
# [notebook only] LVL_1 INPUT: THE LVL_0 HANDOFF THIS NOTEBOOK READS
# =============================================================================
# One LVL_0 handoff, lv0_out_<YYYYmmdd_HHMMSS>.csv, as written by the LVL0 notebook ([cell 7])
# or lvl0.py: its file name in LVL_OUTS ([cell 2]), e.g. "lv0_out_20261005_101500.csv".
# There is NO default: the handoff check below stops until this names an existing handoff. Pinning lets several pipeline branches coexist.
# The run log beside it (lv0_out_<ts>_parameters.json) holds that branch's LVL_0 configuration.
LVL0_HANDOFF = "lv0_out_20261106_180438.csv"
# The LVL_1 sweep (its folder in the sweeps directory) whose calibration LVL_1_CONFIG ([cell 9]) holds: the calibration
# check compares LVL_1_CONFIG with the LVL1 sweep report's <sweep>_selection.json, and the H1 tests read the sweep's
# sessions (T2 and T3) at each detector's best configuration. No default.
LVL1_SWEEP = "_sweep_lvl1_all"
# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))

In [ ]:
# [cell 0]
# =============================================================================
# SET BLAS/LAPACK THREAD LIMITS
# =============================================================================
# CRITICAL: These env vars MUST be set BEFORE importing numpy/scipy.
# They control multi-threaded BLAS (OpenBLAS, Intel MKL, etc.) so that
# joblib parallelism doesn't oversubscribe each worker with BLAS threads.
# Setting to '1' means joblib controls ALL parallelization via n_jobs.
# =============================================================================

import os
import multiprocessing as mp

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

# joblib controls all parallelism via n_jobs
n_jobs = 31

print(f"Available CPU cores: {mp.cpu_count()}")
print(f"CPU cores to use: {n_jobs}")

In [ ]:
# [cell 1]
# =============================================================================
# IMPORTS, WARNING CONTROL
# =============================================================================

# -----------------------------
# Standard library
# -----------------------------
from datetime import datetime
import hashlib
import json
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import warnings

# -----------------------------
# Third-party
# -----------------------------
from joblib import Parallel, delayed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import scipy.stats as stats
from scipy.optimize import curve_fit
import scipy.sparse as sp_sparse
from scipy.sparse.linalg import splu as sp_splu

from hmmlearn import hmm

import ruptures as rpt


from sklearn.cluster import KMeans
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from statsmodels.stats.multitest import fdrcorrection

# -----------------------------
# Warnings (logged to WARNINGS_OUTS/suppressed_warnings.log, set up in [cell 2])
# -----------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("default", category=ConvergenceWarning)

warnings.filterwarnings("default", message=".*convergence.*", category=UserWarning)
warnings.filterwarnings("default", message=".*did not converge.*", category=UserWarning)
warnings.filterwarnings("default", message=".*singular.*", category=UserWarning)
warnings.filterwarnings("default", message=".*Hessian.*", category=UserWarning)
warnings.filterwarnings("default", message=".*MLE.*boundary.*", category=UserWarning)

warnings.filterwarnings("ignore", message=".*KMeans.*", category=UserWarning)
warnings.filterwarnings("ignore", message=".*Fitting.*", category=UserWarning)

In [ ]:
# [cell 2]
# =============================================================================
# SET DIRECTORIES
# =============================================================================
# Everything the pipeline reads and writes is inside /home/sjsuss626/EMST_files/sjs_dissertation:
#   INPUT_LONG_PATH  the raw long-format data, read by LVL_0
#   LVL_OUTS         every output: handoffs, run logs, exports, the warnings log
#   FIG_DIR          figures, written only when the level's save_figures is True
#   CODE_DIR         this folder (the LVL2 notebook loads LVL0 / LVL1 cells from it)
INPUT_LONG_PATH = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dset/all_subjects_long.csv")
LVL_OUTS        = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/lvl_outs")
WARNINGS_OUTS   = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/lvl_outs/warnings")
FIG_DIR         = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/figures")
CODE_DIR        = Path("/home/sjsuss626/EMST_files/sjs_dissertation/code/practical_application_emst")
print("input  :", INPUT_LONG_PATH)
print("outputs:", LVL_OUTS)

# Warnings go to a file so they aren't silently lost.
LVL_OUTS.mkdir(parents=True, exist_ok=True)
WARNINGS_OUTS.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    filename=str(WARNINGS_OUTS / "suppressed_warnings.log"),
    filemode="w",  # overwrite each run
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logging.captureWarnings(True)

# =============================================================================
# PIPELINE VERSION AND SOFTWARE (recorded in every run log)
# =============================================================================
# Development series 0.0.x (1.0.1 will be the general tool). The same literal stands in the three level notebooks'
# [cell 2] (identical copies) and in Replication_and_final_report [cell 1]; the report's [cell 3] checks it against
# the run logs.
PIPELINE_VERSION = "0.0.1"
SOFTWARE_PACKAGES = ("numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib",
                     "matplotlib")


def software_versions():
    """Interpreter and package versions of this process: the "software" entry of every run log."""
    import platform as _platform
    import sys as _sys
    from importlib import metadata as _meta
    out = {"executable": _sys.executable, "python": _platform.python_version()}
    for p in SOFTWARE_PACKAGES:
        try:
            out[p] = _meta.version(p)
        except _meta.PackageNotFoundError:
            out[p] = None
    return out


# =============================================================================
# FIGURE SAVING + PER-LEVEL PARAMETER/RESULTS JSON
# =============================================================================
# Run logs sit in LVL_OUTS beside the outputs they describe. Figures are written only
# when the level's config sets save_figures (SAVE_FIGURES is set in each config cell).
RUN_LOG_DIR = LVL_OUTS
FIG_DIR.mkdir(parents=True, exist_ok=True)

FIG_PREFIX = "LVL"
SAVE_FIGURES = False
_FIG_SAVE_STATE = {"n": 0}

# Re-running this cell must not wrap our own wrapper. _ORIG_PLT_SHOW is a
# notebook GLOBAL and cell-level functions resolve globals at CALL time, so a
# second execution rebinds it to show_and_save itself -- the wrapper then calls
# itself forever (RecursionError inside plt.show()).
# Only ever cache a show() that genuinely comes from matplotlib.pyplot, and
# re-derive it if a previous run cached a wrapper.
def _is_real_show(fn):
    return getattr(fn, "__module__", "") == "matplotlib.pyplot"


if not _is_real_show(getattr(plt, "_orig_show_unpatched", None)):
    if _is_real_show(plt.show):
        plt._orig_show_unpatched = plt.show
    else:
        import importlib
        plt._orig_show_unpatched = importlib.reload(
            importlib.import_module("matplotlib.pyplot")).show
_ORIG_PLT_SHOW = plt._orig_show_unpatched

# Re-entrancy guard: even if some other wrapper is still installed from an
# earlier kernel state, this terminates instead of recursing.
_IN_SHOW = {"active": False}


def show_and_save(*args, **kwargs):
    if _IN_SHOW["active"]:
        return None
    _IN_SHOW["active"] = True
    try:
        return _show_and_save_inner(*args, **kwargs)
    finally:
        _IN_SHOW["active"] = False


def _show_and_save_inner(*args, **kwargs):
    for _num in (plt.get_fignums() if SAVE_FIGURES else []):
        _fig = plt.figure(_num)
        if getattr(_fig, "_saved_by_show", False):
            continue
        _FIG_SAVE_STATE["n"] += 1
        _fig.savefig(
            FIG_DIR / f"{FIG_PREFIX}_fig{_FIG_SAVE_STATE['n']:03d}.png",
            dpi=150, bbox_inches="tight",
        )
        _fig._saved_by_show = True
    return _ORIG_PLT_SHOW(*args, **kwargs)


plt.show = show_and_save


def _jsonable(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.astype(object).where(pd.notna(obj), None).to_dict(orient="records")
    if isinstance(obj, pd.Series):
        return obj.astype(object).where(pd.notna(obj), None).to_dict()
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def file_sha256(path, chunk=1 << 20):
    """SHA-256 of a file's bytes: the content identity of the raw input, recorded by LVL_0 and checked downstream."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def dump_level_json(level, config, results=None, extra=None, name=None):
    """Write the level's run log: <name>_parameters.json, where name is the stem of the
    output it describes (e.g. lv0_out_<ts>), or <level>_parameters_<ts>.json without one."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = RUN_LOG_DIR / (f"{name}_parameters.json" if name else f"{level}_parameters_{ts}.json")
    payload = {
        "level": level,
        "pipeline_version": PIPELINE_VERSION,
        "software": software_versions(),
        "timestamp": ts,
        "input_long_path": str(INPUT_LONG_PATH),
        "figures_dir": str(FIG_DIR),
        "figures_saved": int(_FIG_SAVE_STATE["n"]),
        "parameters": _jsonable(config),
        "statistical_results": _jsonable(results or {}),
        "outputs": _jsonable(extra or {}),
    }
    with open(path, "w") as f:
        json.dump(payload, f, indent=2, default=str)
    print(f"  {level} parameter/results JSON -> {path}")
    return path

In [ ]:
# [notebook only] LVL_0 HANDOFF CHECK
# =============================================================================
# [cell 20] reads LVL0_HANDOFF (first code cell). The file is read from LVL_OUTS
# ([cell 2]). Stops unless the pin names an existing LVL_0 handoff whose name starts
# with its stamp (lv0_out_<YYYYmmdd_HHMMSS>; descriptive text may follow), with exactly one
# run log of that stamp beside it (lv0_out_<stamp>*_parameters.json), built from the same raw
# input this notebook names (INPUT_LONG_PATH, which the LVL_1 handoff records as its input): same name and same
# content (the SHA-256 the LVL_0 run log records).
# LVL0_RECORD = that run log: the configuration of this branch's LVL_0. The pin is then set
# to the handoff's full path, so the later cells read exactly this file.
import re as _re
_avail0 = sorted(p.name for p in LVL_OUTS.glob("lv0_out_*.csv"))
if LVL0_HANDOFF is None:
    raise SystemExit(f"LVL0_HANDOFF is not set (first code cell). Name one LVL_0 handoff; "
                     f"available in {LVL_OUTS}: {_avail0}")
_h0 = LVL_OUTS / LVL0_HANDOFF
_m0 = _re.match(r"lv0_out_(\d{8}_\d{6})", _h0.name)
if not _m0 or _h0.suffix != ".csv":
    raise SystemExit(f"LVL0_HANDOFF = {LVL0_HANDOFF!r} is not a LVL_0 handoff "
                     "(lv0_out_<YYYYmmdd_HHMMSS>[...].csv)")
if not _h0.is_file():
    raise SystemExit(f"LVL0_HANDOFF = {LVL0_HANDOFF!r}: {_h0} not found; available in {LVL_OUTS}: {_avail0}")
_recs0 = sorted(_h0.parent.glob(f"lv0_out_{_m0.group(1)}*_parameters.json"))
if len(_recs0) != 1:
    raise SystemExit(f"expected exactly one LVL_0 run log lv0_out_{_m0.group(1)}*_parameters.json beside "
                     f"{_h0.name}, found {[p.name for p in _recs0]}: this handoff's LVL_0 configuration "
                     "cannot be traced")
LVL0_RECORD_PATH = _recs0[0]
with open(LVL0_RECORD_PATH) as _f:
    LVL0_RECORD = json.load(_f)
if Path(LVL0_RECORD["input_long_path"]).name != Path(INPUT_LONG_PATH).name:
    raise SystemExit(f"{_h0.name} was built from {Path(LVL0_RECORD['input_long_path']).name}, but this "
                     f"notebook's INPUT_LONG_PATH ([cell 2]) is {Path(INPUT_LONG_PATH).name}")
if LVL0_RECORD["outputs"].get("input_long_sha256") is None:
    raise SystemExit(f"{LVL0_RECORD_PATH.name} records no input_long_sha256: re-run LVL_0 with the current code")
if file_sha256(INPUT_LONG_PATH) != LVL0_RECORD["outputs"]["input_long_sha256"]:
    raise SystemExit(f"{INPUT_LONG_PATH} is not the file {_h0.name} was built from (content differs)")
LVL0_HANDOFF = str(_h0)
print(f"LVL_0 handoff : {_h0}")
print(f"LVL_0 record  : {LVL0_RECORD_PATH.name}  (raw input {Path(LVL0_RECORD['input_long_path']).name}, "
      f"{LVL0_RECORD['outputs'].get('n_rows')} rows, {LVL0_RECORD['outputs'].get('n_participants')} participants)")
for _k, _v in LVL0_RECORD["parameters"].items():
    print(f"    {_k:32s} {_v!r}")

*[cell 8]*

# LEVEL 1: COMPREHENSIVE BLOCK DETECTION WITH EXTERNAL VALIDATION

## Block Features and Validation

Every method is validated on **all three detection features** (Kruskal-Wallis, η²_H), including the
feature(s) it segmented on: effect size on those features *is* the test of whether segmentation worked. The set is
`LVL_1_CONFIG["validation_features"]`.

The `use_residuals` flag picks the column variant of each feature:
- **True** (current): residual columns (`aggregated_residual`, `local_residual_var`; entropy is response-based either way)
- **False**: raw-RT columns (`aggregated_rt`, `local_rt_var`)

Missing data: rolling-window edges and windows with fewer than
`min_periods` valid values are **NaN and never filled**; a rating rejected by `min_rt_thresh` / `max_rt_thresh`
reaches the RT features as `rt_threshold_na_handling` says (`partial` / `drop_stimulus` / `bfill` / `ffill`, no
default; cell 9); each detector runs on the stimuli where all its features are observed, and unobserved stimuli
inherit the preceding block (leading ones the first block).

`feature_rt_transform` selects which RT representation (raw / log / boxcox) those features are built from. Each signal is a composite/average of scale-level behaviour.

| Feature | Column (RT) | Column (Residual) | Calculation | Interpretation |
|---------|-------------|-------------------|-------------|----------------|
| **Adj RT Level (Stim)** ($AdjRT_{stim}$) | `aggregated_rt` | `aggregated_residual` | Mean or Median drift-adjusted RT/residual across scales per stimulus; defined by `rt_aggregation` | **Attentive: slow** (deliberate); **Inattentive: fast** (rushed) |
| **RT Variance** | `rt_variance` | `residual_variance` | Variance across scales per trial (cross-scale; computed but not used for detection or validation) | **Attentive: consistent**; **Inattentive: variable** |
| **Adj RT Var (win)** ($AdjVar_{win}$) | `local_rt_var` | `local_residual_var` | Temporal (sliding-window, Esterman VTC) variance of the adjusted RT over `derivative_filter_size` stimuli | **Attentive: stable**; **Inattentive: erratic** |
| **Coherence (RT)** *(computed, not used)* | `coherence_rt` | `coherence_residual` | Mean pairwise distance across scales (normalized by n_pairs) | **Attentive: high** (scale-differentiable); **Inattentive: low** (uniform) |
| **Coherence (Var)** *(computed, not used)* | `coherence_vars` | `coherence_residual_vars` | Mean pairwise distance of rolling per-scale variances | **Attentive: high** (scale-differentiable); **Inattentive: low** (uniform) |
| **Entropy (win)** ($Ent_{win}$) | `rating_entropy_norm` | `rating_entropy_norm` | Shannon entropy of responses over rolling window, normalized by log(k) per scale | **Low: straightlining**; **High: random responding**; **Shift in either direction suggests attention change** |

The three detection features are **Adj RT Level (Stim)**, **Adj RT Var (win)** and **Entropy (win)**. "Adj" = drift-adjusted RT (the residual under `use_residuals=True`); "(Stim)" = one value per stimulus; "(win)" = rolling/sliding-window statistic. They are distinct from the Level 2 block features (**Adj RT Level (Block)**, **Exp RT Var (Block)**, **Exp Entropy (Block)**).

---

## Signal Smoothing

Before changepoint detection, signals can be smoothed to reduce false positives from noise.

| Setting | Options | Description |
|---------|---------|-------------|
| `smoothing_method` | `"none"`, `"boxcar"`, `"gaussian"` | Type of smoothing kernel |
| `smoothing_window` | Integer (e.g., 10) | Window size in trials |

- **none**: No smoothing (raw signal, most sensitive to noise)
- **boxcar**: Uniform moving average (equal weights, simple denoising)
- **gaussian**: Gaussian-weighted (center-weighted, preserves peak shapes better)

---

## METHOD TAXONOMY

Three change-point families plus a fixed-trial baseline. Which families are
registered is controlled by `cpd_method_types` in `LVL_1_CONFIG`
(`["MAD", "HMM", "PELT"]`); each entry below is registered as both
per-feature single-signal variants AND a composite (`*_Attention_comp`)
variant that pools `cp_composite` / `hmm_composite` features. Block merging
applies to the MAD family only, on that method's own detection features.

### 1. CHANGEPOINT DETECTION (MAD-CPD)
**Method-name pattern**: `CPD_<Feature>` (per-feature) and `CPD_Attention_comp` (composite)
- **Definition**: Detect abrupt changes in a signal using a MAD-based threshold.
- **Rationale**: Identifies discrete shifts in participant behavior.
- **Features Used**:
  - Per-feature: one method per entry in `cp_features` (e.g., `CPD_Adj_RT_Level_Stim`, `CPD_Adj_RT_Var_win`)
  - Composite: pools `cp_composite` features via `cp_comp_method`:
    - `"average"` — normalize each signal and average
    - `"PCA"` — first principal component of the features (within-participant)
- **Key config**: `cp_k` (sensitivity; higher = fewer changepoints), `mad_window_size` (windowed vs global MAD)
- **Limitations**: Sensitive to noise if thresholding and smoothing aren't calibrated.

### 2. PELT (Pruned Exact Linear Time)
**Method-name pattern**: `PELT_<Feature>` and `PELT_Attention_comp`
- **Definition**: Exact changepoint detection by minimising a penalised cost. Implemented via the `ruptures` library.
- **Rationale**: Globally optimal under the chosen cost; tends to be more stable than MAD-CPD when signal is noisy but contains real regime shifts.
- **Features Used**: `pelt_features` / `pelt_composite` (currently the same three features as MAD-CPD).
- **Key config**: `pelt_pen_uni` / `pelt_pen_multi` (penalty; higher = fewer changepoints), `pelt_model` (e.g., `"rbf"`, `"l1"`, `"l2"`), `pelt_jump`, `min_block`.
- **Limitations**: Penalty choice is the key tuning knob; an under-penalised fit fragments aggressively.

### 3. HIDDEN MARKOV MODEL (HMM)
**Method-name pattern**: `HMM_<N>_<Feature>` and `HMM_Attention_comp_<N>` (where `<N>` is the state count)
- **Definition**: Detect probabilistic changes in latent state (e.g., attentive vs inattentive).
- **Rationale**: Less brittle than threshold-based methods; can integrate multiple features simultaneously.
- **Features Used**:
  - Per-feature: one method per entry in `hmm_features`
  - Composite: pools `hmm_composite` features
- **Key config**: `hmm_n_states` (typically 2 = attentive/inattentive; loops over a list to register multiple variants)
- **Limitations**: More parameters → can overfit on short sessions and over-fragment.

### 4. FIXED TRIAL-BASED (control / baseline)
**Method-name pattern**: `Fixed_<N>` (explicit sizes) and `Fixed_<DynamicMethod>_based` (matched-N control)
- **Definition**: Divides trials into equal-sized blocks of *N* trials, or into as many equal blocks as a chosen dynamic method found (per session under `control_block_count="per_session"`).
- **Rationale**: Simple baseline; assumes attention shifts occur at regular intervals; matched-N variant is the canonical control for the "would random removal at the same volume buy the same Δα?" diagnostics. Its granularity rule is `control_block_count`.
- **Limitations**: Ignores individual differences in attention dynamics.

In [ ]:
# [cell 9]
# =============================================================================
# CONFIGURATION
# =============================================================================

FIG_PREFIX = "LVL1"

LVL_1_CONFIG = {

    "save_figures": False,           # bool: default OFF. Turn on deliberately when you
                                 # want specific figures. A sweep must never render
                                 # PNGs -- 30 workers share one FIG_DIR and counter. FIG_DIR: [cell 2]

    # --------------------------------------------------------------------------
    # Inherited from LVL_0 (block detection re-reads these)
    # --------------------------------------------------------------------------
    # also set at LVL_2
    "scale_ranges": {         # dict: {scale_name: (min_value, max_value)} for each scale
        "valence":    (1, 9),
        "arousal":    (1, 9),
        "emotion":    (1, 7),
        "similarity": (1, 6),
        "threat":     (1, 5),
    },
    # also set at LVL_2
    "scale_types": {              # dict: {scale_name: "ordinal" | "nominal"}
        "valence":    "ordinal",
        "arousal":    "ordinal",
        "emotion":    "nominal",
        "similarity": "ordinal",
        "threat":     "ordinal",
    },
    # also set at LVL_0
    "pr_stim_drop": 0,                # Practice stimulus (number of stimuli that we define as practice) to drop at session start
    # also set at LVL_0
    "min_rt_thresh": 0,            # Minimum acceptable RT (seconds; default is 0.15)
    # also set at LVL_0
    "max_rt_thresh": 90,              # Maximum acceptable RT (seconds; default is 90)
    # How a scale-level rating REJECTED by min_rt_thresh / max_rt_thresh (an actual
    # violation; a threshold of 0 is off) reaches the Level-1 RT features (cell 19,
    # level1_features). NO DEFAULT: None stops the notebook at the knob check (cell 11),
    # and lvl1.py requires --rt_threshold_na_handling on its command line.
    #   "partial"       drop only that scale's RT value; Adj RT Level (Stim) averages the
    #                   remaining scales; that scale's rolling variance at the stimulus
    #                   is NaN, so Adj RT Var (win) averages the remaining scales too.
    #   "drop_stimulus" the whole stimulus is NaN in every Level-1 feature (and in no
    #                   neighbour's window); it inherits its neighbouring block.
    #   "bfill"/"ffill" that scale's RT value is taken from the next / previous valid
    #                   value of the same scale in the same session (NaN if there is
    #                   none). Its rolling variance there is the window's variance of the
    #                   remaining valid values; only if fewer than min_periods remain is
    #                   it filled the same way (never in a rolling-window edge).
    # In every mode the rejected RT enters no rolling window. Rolling-window edges and
    # other missing values are never filled. Entropy keeps the row-level filter only.
    "rt_threshold_na_handling": "bfill",   # "partial" | "drop_stimulus" | "bfill" | "ffill"
    # also set at LVL_0, LVL_2
    "ses_wise_drift": True,       # bool: True = per-session drift | False = overall drift
    # also set at LVL_0
    "age_adjust_drift_residuals": True,  # bool: regress out linear age effect

    # --------------------------------------------------------------------------
    # Block-feature construction
    # --------------------------------------------------------------------------
    # Which RT representation feeds {scale}_rt (and, when use_residuals=True, which
    # drift-model space the residual side is computed in). LVL_0 fits a drift curve
    # and computes residuals for all three; this only selects which set LVL_1 reads.
    # use_residuals then picks the value (False) or the residual (True) WITHIN this
    # representation.
    "feature_rt_transform": "log",  # str: "raw" | "log" | "boxcox"
    "rt_aggregation": "mean",      # str: "mean" or "median" - how to aggregate RT across scales per stimulus
    # also set at LVL_2
    "use_residuals": True,         # bool: True = residual features | False = raw RT features
    # "rolling_center": Whether rolling windows are centered or trailing
    #   - True: Window centered on current row (uses past AND future values)
    #   - False: Window right-aligned/trailing (uses only past values)
    # Use True for post-hoc analysis; False for causal/real-time interpretation.
    # Selected on the LVL_1 sweep.
    "rolling_center": True,
    # temporal_var_method: how the "Adj RT Var (win)" changepoint feature builds its
    # sliding-window (Esterman Variance-Time-Course) RT variance from the 5 scales:
    #   "aggregate" = temporal variance of the cross-scale-collapsed RT (aggregated_rt)
    #                 -- collapse scales, THEN variance over the trailing window.
    #   "per_scale" = mean across scales of each scale's OWN rolling temporal variance
    #                 -- variance over time per scale, THEN average (5 parallel VTC
    #                 streams). Toggles what local_rt_var / local_residual_var hold;
    #                 does NOT change cross-scale "RT Variance" (rt_variance).
    # sliding-var construction knob
    "temporal_var_method": "per_scale",  # "aggregate" | "per_scale"
    # Smoothing helps prevent false positives from noise spikes.
    # Applied BEFORE changepoint detection on the signal.
    # "smoothing_method": Type of smoothing kernel
    #   - "none": No smoothing (raw signal)
    #   - "boxcar": Uniform/moving average (equal weights)
    #   - "gaussian": Gaussian-weighted smoothing (center-weighted)
    # "smoothing_window": Window size for smoothing (in trials)
    #   - Larger = more smoothing, fewer false positives, but may miss real changes
    #   - Typical values: 5-20 trials
    "smoothing_method": "gaussian",   # str: "none" | "boxcar" | "gaussian" - smoothing kernel type
    # Gaussian kernel (sigma = window/6), Esterman-style. Selected on the LVL_1 sweep.
    "smoothing_window": 7,
    # Rolling-window width (stimuli) for Adj RT Var (win) and Entropy (win). Has no
    # effect on the Adj RT Level (Stim) detectors, which use no rolling window.
    "derivative_filter_size": 9,   # int: centred rolling window; must be ODD -- the
    # constraint check refuses even centred windows.
    # Minimum NON-missing observations a rolling window needs to return a value. One
    # rule for Adj RT Var (win) and Entropy (win) (cell 10): missing values inside a
    # window are skipped, windows with fewer than min_periods valid values are NaN,
    # and those NaNs are NEVER filled -- at segmentation the stimulus inherits the
    # neighbouring block instead (cell 18). Also read by the MAD-CPD threshold window.
    #   rolling_center=True,  window 9: stimuli 1-3 and the last 3 are NaN; the first
    #                                   estimate is at stimulus 4 (8 of 9 observed).
    #   rolling_center=False, window 9: stimuli 1-7 are NaN; the first estimate is at
    #                                   stimulus 8 (8 observed); no NaN at the end.
    "min_periods": 8,              # int: minimum non-missing observations in the rolling window

    # --------------------------------------------------------------------------
    # Changepoint detection (CPD)
    # --------------------------------------------------------------------------
    # Selects which families of changepoint methods to register. Drop entries
    # from this list to skip them entirely.
    #   "MAD"  - threshold-on-derivative with median + k*MAD*1.4826
    #   "HMM"  - Gaussian HMM Viterbi decoding (requires hmm_n_states)
    #   "PELT" - ruptures.Pelt penalty-based segmentation (no manual k)
    "cpd_method_types": ["MAD", "HMM", "PELT"],
    # +Adj RT Var (win) sliding var, CPD detection
    "cp_features":  ["Adj RT Level (Stim)", "Adj RT Var (win)", "Entropy (win)"],   # list[str]: single-feature changepoint (Adj RT Var (win) = temporal/sliding-window)
    "cp_composite": ["Adj RT Level (Stim)", "Adj RT Var (win)", "Entropy (win)"],   # list[str]: Features for composite changepoint
    "cp_comp_method": "PCA",   # str: "average" | "PCA" - method to combine composite features
    # cp_k is a robust-z cutoff (median + k*MAD*1.4826). Affects the CPD family ONLY;
    # PELT uses its penalty and HMM has no scalar sensitivity knob.
    "cp_k": 3,
    # MAD-CPD threshold window. None = one global median/MAD over all session
    # diffs. int = sliding-window MAD: at each candidate gap, the threshold uses
    # the median/MAD of diffs within a centered window of N stimuli around it.
    # 9 stimuli ~ 2 min of task time (median 1.92 min, IQR 1.36-2.64), the window length
    # Esterman et al. (2013) used for RT-variability measures.
    "mad_window_size": 9,        # None | int (e.g. 9); must be odd (centred window)

    # --------------------------------------------------------------------------
    # PELT detection
    # --------------------------------------------------------------------------
    # +Adj RT Var (win) sliding var, PELT detection
    "pelt_features": ["Adj RT Level (Stim)", "Adj RT Var (win)", "Entropy (win)"],  # list[str] - single-feature PELT (Adj RT Var (win) = temporal/sliding-window)
    "pelt_composite": ["Adj RT Level (Stim)", "Adj RT Var (win)", "Entropy (win)"],  # list[str] - composite PELT
    # PELT settings (requires `ruptures` package)
    "pelt_model": "l1",            # "l1" | "l2" | "rbf" — cost function
    # PELT penalty, univariate: b*log(n) form. The constant b is calibrated empirically:
    # the l1 cost gives no elbow to read it from.
    "pelt_pen_uni":   5.298,
    # PELT penalty for MULTIVARIATE (composite) methods. Multivariate PELT cost sums
    # 3*d*log(n). three paramteres
    "pelt_pen_multi": 15.895,
    # Selected on the LVL_1 sweep (must divide min_block_size).
    "pelt_jump": 1,

    # --------------------------------------------------------------------------
    # HMM detection
    # --------------------------------------------------------------------------
    # HMM Feature options:
    "hmm_features":  ["Adj RT Level (Stim)", "Adj RT Var (win)", "Entropy (win)"],  # list[str]: Features for basic HMM
    "hmm_composite": ["Adj RT Level (Stim)", "Adj RT Var (win)", "Entropy (win)"],  # list[str]: Features for composite HMM
    "hmm_n_states": [2, 3],        # list[int]: Number of hidden states to try for HMM
    "hmm_covariance_type": "tied", # str: "full" | "diag" | "spherical" | "tied" - covariance structure
    # With hmm_tol=1e-8 the slowest of 4,420 fits needed 1,616 iterations (median 29);
    # 2,500 converges every fit with ~1.5x headroom.
    "hmm_n_iter": 2500,            # int: Maximum EM iterations for HMM fitting
    # EM stops when |delta log-likelihood| < hmm_tol (AbsDeltaConvergenceMonitor, cell
    # 15). hmmlearn's own rule tests the SIGNED delta, so any decrease would stop EM --
    # and under the transition prior below the log-likelihood can dip while the MAP
    # objective is still rising. Set here explicitly; hmmlearn's default is 1e-2.
    "hmm_tol": 1e-8,               # float: |delta log-likelihood| that counts as converged
    "hmm_random_state": 26,        # int: Random seed for HMM initialization
    # State-persistence prior on the transition matrix. Value = the number of
    # pseudo self-transitions ADDED to each state's diagonal; off-diagonals get
    # exactly ZERO (no bias toward or against switching -- only the data decides
    # switches). Internally we pass (value+1) on the diagonal and 1 off-diagonal
    # so hmmlearn's `prior-1` MAP step cancels to +value diagonal, +0 off-diagonal.
    # It is a pseudo-COUNT, not a percentage: strength is relative to the ~T-1
    # observed transitions in a session, so the same value bites harder on short
    # sessions. Higher = stickier states = fewer, longer blocks.
    #   0   -> no prior (pure ML EM, hmmlearn default)
    #   200 -> +200 pseudo self-transitions per state (current setting)
    # HMM state-persistence prior 
    "hmm_transmat_self_prior": 200,  # float: pseudo self-transition count on diagonal (0 = off)

    # --------------------------------------------------------------------------
    # Block geometry and fixed-block controls
    # --------------------------------------------------------------------------
    # Selected on the LVL_1 sweep.
    "min_block_size": 20,
    "fixed_block_sizes": None,     # list|None: [25, 50, 100] for fixed-size blocks, None = skip

    # --------------------------------------------------------------------------
    # Block merging
    # --------------------------------------------------------------------------
    # After block detection, optionally merge adjacent blocks that are NOT
    # significantly different across specified validation features.
    # "merge_blocks": Enable/disable block merging
    #   - True: Merge adjacent non-significant blocks
    #   - False: Keep original block assignments
    # "merge_alpha": Significance threshold for merging decision
    #   - Blocks are merged if FDR-corrected p > merge_alpha
    "merge_blocks": True,          # bool: MAD-CPD only -- merge non-significant adjacent blocks
    # "merge_on": Threshold controlling how many features trigger/permit merging.
    #   ttest      : merge when n_significant < merge_on
    #     Higher = more permissive (more differences tolerated before blocking a merge)
    #     Lower  = stricter        (fewer differences needed to block a merge)
    #   robust_tost: merge when n_equivalent >= merge_on
    #     Higher = stricter        (more features must show equivalence to allow a merge)
    #     Lower  = more permissive (fewer equivalences needed to allow a merge)
    #   Example with 4 features:
    #     merge_on=4: ttest       → merge unless ALL 4 features differ         (most permissive)
    #                 robust_tost → merge only if ALL 4 features are equivalent (most strict)
    #     merge_on=3: ttest       → merge unless 3+ features differ
    #                 robust_tost → merge only if 3+ features are equivalent
    #     merge_on=2: ttest       → merge unless 2+ features differ
    #                 robust_tost → merge only if 2+ features are equivalent
    #     merge_on=1: ttest       → merge only if NO features differ            (most strict)
    #                 robust_tost → merge if ANY 1 feature is equivalent        (most permissive)
    #   Clamped down to the number of features the merged method was detected on,
    #   so a univariate detector effectively runs at merge_on=1.
    "merge_on": 2,                 # int: Number of significant features needed to prevent/cause merge
    # Merge method: How to determine if blocks should be merged
    #   "ttest"      : Welch t-test — merge if NOT significantly different (parametric)
    #   "wilcoxon"   : Mann-Whitney U — same logic as ttest but non-parametric (no
    #                  normality assumption). Recommended when merged features include
    #                  bounded/skewed features like Entropy (win) or Rating Variance.
    #   "robust_tost": merge when blocks ARE equivalent (Wilcoxon TOST via TOSTER R pkg)
    "merge_method": "wilcoxon",        # str: "ttest" | "wilcoxon" | "robust_tost"
    "merge_alpha": 0.05,           # float: p-value threshold — ttest/wilcoxon: merge if p > alpha; robust_tost: merge if p_tost < alpha
    # "merge_fdr": FDR correction applied to the per-feature p-values within each pair.
    #   -  None      : no correction — recommended for robust_tost (TOST is already
    #                  conservative; adding FDR is non-standard and overly strict)
    #   - "per_pair" : Benjamini-Hochberg across the M features tested per pair
    #                  (mild correction, sensible for ttest with several features)
    #   - "global"   : BH across ALL pairs × features in the session
    #                  (most conservative; requires a non-greedy pass — not yet
    #                   implemented separately, currently treated as "per_pair")
    "merge_fdr": "per_pair",          # None | "per_pair" | "global"
    # "merge_equivalence_bound": TOST equivalence margin in Cohen's d (SMD) units.
    #   Bounds are scale-invariant — same value applies across all merged features
    #   regardless of their measurement units.
    #   Typical values: 0.2 (small), 0.35 (small-medium), 0.5 (medium)
    "merge_equivalence_bound": 0.2,   # float: Cohen's d — used only when merge_method="robust_tost"
    # --------------------------------------------------------------------------
    # Validation
    # --------------------------------------------------------------------------
    # Statistical test for external block validation (drives Sig%/Corr% and the
    # effect size). "kruskal" = Kruskal-Wallis (nonparametric, rank-based; robust
    # to the skewed / bounded / heavy-tailed validation features -- recommended)
    # with the eta^2_H effect size. "anova" = parametric one-way ANOVA + omega-
    # squared. NOTE: changing this changes the reported numbers.
    # Kruskal-Wallis vs ANOVA validation
    "validate_test": "kruskal",    # str: "kruskal" (nonparametric) | "anova" (parametric)
    # True = validate on this feature, False = ignore. avg_effsize is an unweighted mean.
    # Actual columns depend on "use_residuals" (rt_col vs residual_col in
    # FEATURE_REGISTRY).
    # The validation set is exactly the three features the detectors segment on.
    # Effect size on those features IS the test of whether segmentation worked, so
    # every method is validated on all three regardless of what it segmented on.
    # Coherence (RT) / Coherence (Var) and the cross-scale RT Variance are not part
    # of the publication set. A feature absent from this dict is NOT validated.
    "validation_features": {           # dict: {feature_name: bool}
        "Adj RT Level (Stim)": True,   # aggregated_rt OR aggregated_residual
        "Adj RT Var (win)":    True,   # local_rt_var OR local_residual_var
        "Entropy (win)":       True,   # rating_entropy_norm (always response-based)
    },
    # also set at LVL_2
    "control_block_count": "per_session",  # str: "median" | "mean" | "per_session"

    # --------------------------------------------------------------------------
    # Parallelism
    # --------------------------------------------------------------------------
    "n_jobs": 31,                              # int: joblib workers for parallel block detection (the 31-CPU limit)
    "parallel_verbose": 10,                    # int: joblib verbosity for the parallel detection loop
}

SAVE_FIGURES = LVL_1_CONFIG["save_figures"]

In [ ]:
# [notebook only] LVL_1 CALIBRATION CHECK
# =============================================================================
# LVL_1_CONFIG must hold the all-data calibration: the knob values of the selected detector's best configuration, written by
# LVL1_sweep_report.ipynb as <sweep>_selection.json in LVL_OUTS, from a sweep run on this notebook's LVL_0 handoff.
# Stops on any difference. LVL1_SELECTION = that file (the H1 cells take the selected detector from it).
_sel_path = LVL_OUTS / f"{LVL1_SWEEP}_selection.json"
if LVL1_SWEEP is None or not _sel_path.is_file():
    raise SystemExit(f"{_sel_path.name} is missing: run LVL1_sweep_report.ipynb on the sweep {LVL1_SWEEP!r} first")
with open(_sel_path) as _f:
    LVL1_SELECTION = json.load(_f)
_stamp0 = lambda name: _re.match(r"lv0_out_(\d{8}_\d{6})", Path(str(name)).name).group(1)
if _stamp0(LVL1_SELECTION["lvl0_input"]) != _stamp0(LVL0_HANDOFF):
    raise SystemExit(f"the sweep {LVL1_SWEEP} ran on {LVL1_SELECTION['lvl0_input']}, not on {Path(LVL0_HANDOFF).name}")
_diff = {k: (LVL_1_CONFIG.get(k), v) for k, v in LVL1_SELECTION["LVL_1_CONFIG"].items() if LVL_1_CONFIG.get(k) != v}
if _diff:
    raise SystemExit(f"LVL_1_CONFIG differs from {_sel_path.name} ({{knob: (LVL_1_CONFIG, selection)}}): {_diff}")
print(f"calibration check: LVL_1_CONFIG = {_sel_path.name} {LVL1_SELECTION['LVL_1_CONFIG']} "
      f"(selected detector {LVL1_SELECTION['detector']}, {LVL1_SELECTION['config']})")

In [ ]:
# [cell 10]
# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def fill_rejected(values, rejected, direction, groups=None):
    """Replace ONLY the `rejected` cells of `values` by the next ("bfill") or previous
    ("ffill") non-missing value, within `groups` (e.g. id and session) when given.

    The single implementation of the rt_threshold_na_handling fills (cell 19). Every
    cell that is not rejected is returned unchanged, whatever it holds -- a rolling-
    window edge, a naturally missing value or a valid value. Rejected cells are emptied
    before the fill, so they are never a source for each other; a rejected cell with no
    valid value in `direction` stays NaN (there is no switch to the other direction)."""
    if direction not in ("bfill", "ffill"):
        raise ValueError(f"fill_rejected: direction must be 'bfill' or 'ffill', got {direction!r}")
    out = pd.Series(values, dtype=float, copy=True)
    rej = np.asarray(rejected, dtype=bool)
    if not rej.any():
        return out
    out[rej] = np.nan
    src = out.groupby(groups, sort=False) if groups is not None else out
    src = src.bfill() if direction == "bfill" else src.ffill()
    out[rej] = src[rej]
    return out


def compute_stim_resp_features(stim_lvl_df, config, rt_rejected=None, rt_rejected_fill=None):
    """
    Extract comprehensive features for each stimulus (one row per stim_order).

    Parameters
    ----------
    stim_lvl_df : pd.DataFrame
        Stim-level data with {scale}_rt, {scale}_response columns (and optional id/session/stim_order)
    config : dict
        Configuration dictionary (required)
    rt_rejected : pd.DataFrame of bool, optional
        One column per scale, same index as stim_lvl_df: True where that scale's rating
        was rejected by min_rt_thresh / max_rt_thresh (cell 19, level1_features). The
        caller has already removed or filled those {scale}_rt / {scale}_residual values.
        Here the rejected slot is left out of that scale's rolling-variance windows.
        None = no rejected cells.
    rt_rejected_fill : None | "bfill" | "ffill"
        None ("partial"): that scale's variance AT a rejected stimulus is NaN.
        "bfill"/"ffill": that scale's variance at a rejected stimulus is its own window's
        variance of the remaining valid values. Only when that window has fewer than
        min_periods of them is it filled from the next / previous valid variance of the
        same scale in the same session (fill_rejected) -- and not in a rolling-window
        edge, which stays NaN like every edge.

    Returns
    -------
    pd.DataFrame
        Stim-level data with computed features.
        
    Feature Notes
    -------------
    Detection / validation features (FEATURE_REGISTRY):
        - aggregated_residual (aggregated_rt when use_residuals=False): Adj RT Level (Stim),
          the rt_aggregation (mean) of the per-scale values
        - local_residual_var (local_rt_var): Adj RT Var (win), rolling per-scale variance
          averaged across scales (temporal_var_method="per_scale")
        - rating_entropy_norm: Entropy (win), rolling per-scale Shannon entropy / ln(K),
          averaged across scales. Low = straightlining, high = random responding.
    Rolling windows: derivative_filter_size wide, rolling_center, min_periods non-missing
    values; windows below min_periods are NaN and are not filled.
    Also computed but not used for detection or validation: cross-scale variances,
    coherence features, local means, per-scale entropies, index columns.
    """
    df = stim_lvl_df.copy()

    # Rejected-RT flags ride through the sort as temporary columns (removed below).
    _REJ = "__rt_rejected__"
    if rt_rejected is not None:
        if rt_rejected_fill not in (None, "bfill", "ffill"):
            raise ValueError(f"rt_rejected_fill must be None, 'bfill' or 'ffill', got {rt_rejected_fill!r}")
        for _s in config["scale_ranges"]:
            df[f"{_REJ}{_s}"] = rt_rejected[_s].reindex(df.index, fill_value=False).to_numpy(dtype=bool)

    # Robust ordering: always sort before rolling/gradient features
    sort_cols = [c for c in ["id", "session", "stim_order"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)
    rej_by_scale = ({_s: df.pop(f"{_REJ}{_s}").to_numpy(dtype=bool) for _s in config["scale_ranges"]}
                    if rt_rejected is not None else {})
    _sess_keys = [c for c in ["id", "session"] if c in df.columns]

    def scale_rolling_var(series, scale):
        """Rolling variance of one scale's series. A rejected slot never enters a window.
        At a rejected stimulus: "partial" -> NaN; "bfill"/"ffill" -> the window's variance
        of the remaining valid values, filled from the neighbouring valid variance only
        where that window is below min_periods but would reach it with the rejected
        slots counted as observed. A rejected stimulus in a rolling-window edge (or an
        otherwise under-filled window) keeps that NaN: the knob never fills an edge."""
        rej = rej_by_scale.get(scale)
        if rej is None or not rej.any():
            return series.rolling(window, center=center, min_periods=min_periods).var().values
        v = series.mask(rej).rolling(window, center=center, min_periods=min_periods).var()
        if rt_rejected_fill is None:
            v[rej] = np.nan
        else:
            n_obs = (series.notna().to_numpy() | rej).astype(float)
            n_obs = pd.Series(n_obs).rolling(window, center=center, min_periods=1).sum().to_numpy()
            short = rej & v.isna().to_numpy() & (n_obs >= min_periods)
            v = fill_rejected(v, short, rt_rejected_fill,
                              groups=[df[c].to_numpy() for c in _sess_keys] or None)
        return v.values

    window = config["derivative_filter_size"]
    min_periods = config["min_periods"]
    scales = list(config["scale_ranges"].keys())
    center = config["rolling_center"]

    n = len(df)

    # Identify available columns
    rt_cols = [f"{s}_rt" for s in scales if f"{s}_rt" in df.columns]
    resp_cols = [f"{s}_response" for s in scales if f"{s}_response" in df.columns]
    resid_cols = [f"{s}_residual" for s in scales if f"{s}_residual" in df.columns]

    if not rt_cols:
        raise ValueError("No RT columns found!")

    rt_matrix = df[rt_cols].to_numpy(dtype=float)

    # === HELPER: Pairwise Distance Sum ===
    def pairwise_distance_mean(row):
        valid = row[~np.isnan(row)]
        if len(valid) < 2:
            return np.nan
        total = 0.0
        n_pairs = 0
        for i in range(len(valid)):
            for j in range(i + 1, len(valid)):
                total += abs(valid[i] - valid[j])
                n_pairs += 1
        return total / n_pairs  # normalize by number of pairs

    # === HELPER: Shannon Entropy ===
    def shannon_entropy(values):
        """
        Compute Shannon entropy of a discrete distribution.
        H = -Σ p(x) * log(p(x))
        
        Returns entropy in nats (natural log). 
        Returns NaN if not enough valid values.
        """
        valid = values[~np.isnan(values)]
        if len(valid) < 2:
            return np.nan
        
        # round-then-validate instead of silent astype(int) truncation.
        # Truncating 1.5 → 1 distorts entropy on data-entry-error rows that
        # passed the in-range check but contain non-integer responses.
        rounded = np.round(valid).astype(int)
        if not np.allclose(rounded, valid, atol=1e-6):
            import warnings as _w_se
            _w_se.warn(
                "shannon_entropy: non-integer responses detected; rounded for entropy calc",
                stacklevel=2,
            )
        # Count frequency of each unique value
        unique, counts = np.unique(rounded, return_counts=True)
        probs = counts / counts.sum()
        
        # Shannon entropy: -Σ p * log(p)
        # Only include non-zero probabilities
        entropy = -np.sum(probs * np.log(probs))
        return entropy

    # =========================================================================
    # RT-BASED FEATURES (for use_residuals=False)
    # =========================================================================
    # Aggregation method from config (median when no config is passed)
    rt_agg = config["rt_aggregation"] if config else "median"
    
    df["mean_rt"] = np.nanmean(rt_matrix, axis=1)
    df["median_rt"] = np.nanmedian(rt_matrix, axis=1)
    
    # Set aggregated_rt based on rt_aggregation config
    if rt_agg == "mean":
        df["aggregated_rt"] = df["mean_rt"]
    else:
        df["aggregated_rt"] = df["median_rt"]
    df["rt_variance"] = np.nanvar(rt_matrix, axis=1)
    df["rt_std"] = np.nanstd(rt_matrix, axis=1)
    
    # Rating variance: variance of responses across scales for each stimulus.
    # Measures how much a participant differentiates between scales for each
    # stimulus. Restrict to ordinal/continuous scales -- nominal codes (e.g.,
    # emotion 1-7) are not numerically meaningful, so including them in
    # arithmetic variance contaminates the feature.
    if resp_cols:
        _scale_types_cfg = config["scale_types"]
        _ordinal_resp_cols = [
            c for c in resp_cols
            if _scale_types_cfg[c.replace("_response", "")] != "nominal"
        ]
        if _ordinal_resp_cols:
            resp_matrix = df[_ordinal_resp_cols].to_numpy(dtype=float)
            df["rating_variance"] = np.nanvar(resp_matrix, axis=1)
        else:
            # All response scales are nominal -- no meaningful rating_variance.
            df["rating_variance"] = np.nan

    # Local statistics (use aggregated_rt)
    rt_series = pd.Series(df["aggregated_rt"].values)
    df["local_rt_mean"] = rt_series.rolling(window, center=center, min_periods=min_periods).mean()
    # Sliding temporal RT var. Windows with < min_periods valid values stay NaN (never
    # filled); detectors run on observed rows only and the NaN stimuli inherit the
    # neighbouring block (cell 18). Same rule as Entropy (win) below.
    df["local_rt_var"] = rt_series.rolling(window, center=center, min_periods=min_periods).var()
    # Whole-task (GLOBAL) RT variance: variance of aggregated_rt over the ENTIRE session,
    # broadcast to every stimulus (constant within id x session). A person-level RT-
    # variability trait exposed for the BLOCK/CLUSTER stage ONLY -- constant over time,
    # so it CANNOT produce changepoints (never put it in cp_/pelt_/hmm_features).
    _gvcols = [c for c in ["id", "session"] if c in df.columns]
    # whole-task global RT var
    df["global_rt_var"] = (df.groupby(_gvcols)["aggregated_rt"].transform("var")
                           if _gvcols else df["aggregated_rt"].var())

    # Coherence (RT): pairwise distance of instantaneous RTs
    df["coherence_rt"] = np.apply_along_axis(pairwise_distance_mean, 1, rt_matrix)
    df["local_coherence_rt"] = pd.Series(df["coherence_rt"]).rolling(
        window, center=center, min_periods=min_periods
    ).mean()

    # Coherence (Var): pairwise distance of rolling variances per scale
    per_scale_rt_vars = {}
    for col in rt_cols:
        scale_name = col.replace("_rt", "")
        per_scale_rt_vars[scale_name] = scale_rolling_var(pd.Series(df[col]), scale_name)

    rt_var_matrix = np.column_stack([per_scale_rt_vars[s] for s in scales if f"{s}_rt" in df.columns])
    df["coherence_vars"] = np.apply_along_axis(pairwise_distance_mean, 1, rt_var_matrix)

    # "Adj RT Var (win)" temporal-variance flavour (config temporal_var_method). Default
    # "aggregate" (local_rt_var set above from aggregated_rt). "per_scale" instead
    # averages each scale's own rolling temporal variance (rt_var_matrix, already
    # computed for coherence_vars) -> mean-per-scale VTC. Edge / under-filled windows
    # stay NaN, exactly as for Entropy (win).
    if config["temporal_var_method"] == "per_scale":
        # temporal_var_method=per_scale override
        df["local_rt_var"] = pd.Series(np.nanmean(rt_var_matrix, axis=1))
    df["local_coherence_vars"] = pd.Series(df["coherence_vars"]).rolling(
        window, center=center, min_periods=min_periods
    ).mean()

    # =========================================================================
    # RESIDUAL-BASED FEATURES (for use_residuals=True)
    # =========================================================================
    if resid_cols:
        resid_matrix = df[resid_cols].to_numpy(dtype=float)
        
        # Level: aggregated residual across scales (mean or median based on config)
        df["mean_residual"] = np.nanmean(resid_matrix, axis=1)
        df["median_residual_raw"] = np.nanmedian(resid_matrix, axis=1)

        # NOTE -- naming quirk: `median_residual` holds whichever aggregation
        # rt_aggregation selected (mean OR median), NOT necessarily a median.
        # The name is kept because downstream consumers (get_validation_map /
        # get_feature_column) read it.
        # If rt_aggregation=="mean", median_residual == mean_residual.
        if rt_agg == "mean":
            df["median_residual"] = df["mean_residual"]
        else:
            df["median_residual"] = df["median_residual_raw"]
        
        # aggregated_residual mirrors aggregated_rt: uses rt_aggregation to match naming expected
        # by get_validation_map and get_feature_column
        df["aggregated_residual"] = df["median_residual"]
        
        # Variance: variance of residuals across scales
        df["residual_variance"] = np.nanvar(resid_matrix, axis=1)
        
        # Local statistics (use median_residual)
        resid_series = pd.Series(df["median_residual"].values)
        df["local_residual_mean"] = resid_series.rolling(window, center=center, min_periods=min_periods).mean()
        df["local_residual_var"] = resid_series.rolling(window, center=center, min_periods=min_periods).var()
        # residual counterpart of global_rt_var (whole-task variance; block/cluster only).
        _gvcols_r = [c for c in ["id", "session"] if c in df.columns]
        df["global_residual_var"] = (df.groupby(_gvcols_r)["median_residual"].transform("var")
                                     if _gvcols_r else df["median_residual"].var())
        
        # Coherence (Residual): pairwise distance of instantaneous residuals
        df["coherence_residual"] = np.apply_along_axis(pairwise_distance_mean, 1, resid_matrix)
        df["local_coherence_residual"] = pd.Series(df["coherence_residual"]).rolling(
            window, center=center, min_periods=min_periods
        ).mean()
        
        # Coherence (Residual Vars): pairwise distance of rolling residual variances per scale
        per_scale_resid_vars = {}
        for col in resid_cols:
            scale_name = col.replace("_residual", "")
            per_scale_resid_vars[scale_name] = scale_rolling_var(pd.Series(df[col]), scale_name)
        
        resid_var_matrix = np.column_stack([per_scale_resid_vars[s] for s in scales if f"{s}_residual" in df.columns])
        df["coherence_residual_vars"] = np.apply_along_axis(pairwise_distance_mean, 1, resid_var_matrix)

        # residual counterpart of the temporal_var_method toggle (see RT side).
        if config["temporal_var_method"] == "per_scale":
            df["local_residual_var"] = pd.Series(np.nanmean(resid_var_matrix, axis=1))
        df["local_coherence_residual_vars"] = pd.Series(df["coherence_residual_vars"]).rolling(
            window, center=center, min_periods=min_periods
        ).mean()

    # =========================================================================
    # RESPONSE PATTERN FEATURES (always computed)
    # =========================================================================
    if resp_cols:
        scale_ranges = config["scale_ranges"]
        resp_scales = [c.replace("_response", "") for c in resp_cols]

        missing = [s for s in resp_scales if s not in scale_ranges]
        if missing:
            raise ValueError(f"Missing scale_ranges for: {missing}. Found: {list(scale_ranges.keys())}")

        # Mark illegal values as NaN
        for col in resp_cols:
            s = col.replace("_response", "")
            lo, hi = scale_ranges[s]
            bad = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
            if bad.any():
                df.loc[bad, col] = np.nan

        # =====================================================================
        # Shannon Entropy over Rolling Window
        # =====================================================================
        # For each scale, compute Shannon entropy of response distribution
        # within a rolling window. Then average across scales.
        # Low entropy = responses concentrated (straightlining)
        # High entropy = responses spread uniformly (random responding)
        # A SHIFT in entropy (either direction) between blocks indicates
        # a change in response pattern - useful for detecting attention switches.
        # =====================================================================
        
        per_scale_entropy = []
        
        for col in resp_cols:
            scale_name = col.replace("_response", "")
            resp_series = df[col].values
            
            # Compute rolling Shannon entropy for this scale.
            # use pandas rolling so the entropy window matches the
            # RT/coherence rolling features (same window/center/min_periods
            # semantics), so entropy blocks and RT blocks align at boundaries.
            # Same NA rule as Adj RT Var (win): min_periods counts NON-missing
            # values, missing values inside the window are dropped, and windows
            # below min_periods (incl. the session edges) stay NaN -- not filled.
            _entropy_series = (
                pd.Series(resp_series)
                .rolling(window=window, center=center, min_periods=min_periods)
                .apply(shannon_entropy, raw=True)
            )
            entropy_vals = _entropy_series.to_numpy()
            
            per_scale_entropy.append(entropy_vals)
        
        # Average entropy across scales
        entropy_matrix = np.column_stack(per_scale_entropy)
        df["rating_entropy"] = np.nanmean(entropy_matrix, axis=1)
        
        # Also compute normalized entropy (0-1 scale based on max possible)
        # Max entropy for a scale with k possible values = log(k)
        max_entropies = []
        for col in resp_cols:
            s = col.replace("_response", "")
            lo, hi = scale_ranges[s]
            # validate scale range; otherwise log(k) for k<=0 silently
            # produces -inf/NaN and contaminates the normalized entropy column.
            if hi < lo:
                raise ValueError(
                    f"scale_ranges['{s}'] has hi ({hi}) < lo ({lo}); fix LVL_1_CONFIG['scale_ranges']"
                )
            k = max(1, hi - lo + 1)  # Number of possible values
            max_entropies.append(np.log(k))
        
        # Normalize each scale's entropy by its max, then average
        normalized_entropy = []
        for i, col in enumerate(resp_cols):
            normalized_entropy.append(per_scale_entropy[i] / max_entropies[i])
        
        norm_entropy_matrix = np.column_stack(normalized_entropy)
        df["rating_entropy_norm"] = np.nanmean(norm_entropy_matrix, axis=1)

        # Keep the PER-SCALE normalised entropy alongside the cross-scale average
        # (diagnostic columns; LVL_2 computes its own block-level entropy).
        for _i, _col in enumerate(resp_cols):
            df[f"rating_entropy_norm_{_col.replace('_response', '')}"] = normalized_entropy[_i]

    # === INDEX FEATURES ===
    df["trial_global"] = np.arange(n)
    df["trial_in_session"] = df.groupby("session").cumcount() if "session" in df.columns else np.arange(n)

    if "session" in df.columns:
        sessions = sorted(df["session"].dropna().unique())
        session_map = {s: i for i, s in enumerate(sessions)}
        df["session_indicator"] = df["session"].map(session_map).fillna(0)
    else:
        df["session_indicator"] = 0

    return df

In [ ]:
# [cell 11]
import math

class KnobConstraintError(ValueError):
    """Illegal knob value or combination. Never caught internally."""

ODD_ONLY_KNOBS = ("smoothing_window", "derivative_filter_size")
JUMP_CANDIDATES = (1, 2, 5)
POSITIVE_INT_KNOBS = ("smoothing_window", "derivative_filter_size", "min_block_size",
                      "pelt_jump", "hmm_states", "min_periods")

# rt_threshold_na_handling (cell 9): the four legal values. There is NO default.
RT_THRESHOLD_NA_MODES = ("partial", "drop_stimulus", "bfill", "ffill")


def rt_threshold_na_mode(cfg: dict) -> str:
    """The configured rt_threshold_na_handling, or KnobConstraintError. None, booleans
    and any other value are refused, never coerced."""
    if "rt_threshold_na_handling" not in cfg:
        raise KnobConstraintError("rt_threshold_na_handling is missing from the config; it has "
                                  f"no default -- set one of {list(RT_THRESHOLD_NA_MODES)}.")
    mode = cfg["rt_threshold_na_handling"]
    if not (isinstance(mode, str) and mode in RT_THRESHOLD_NA_MODES):
        raise KnobConstraintError(
            f"rt_threshold_na_handling={mode!r} is not one of {list(RT_THRESHOLD_NA_MODES)}. It has "
            "no default: set it in LVL_1_CONFIG, or pass --rt_threshold_na_handling to lvl1.py.")
    return mode


def jumps_for(min_block_size: int) -> list[int]:
    """Jumps dividing min_block_size exactly, so nominal == effective min block.

    For levels that are multiples of 5 this reproduces the intended rule exactly:
    odd (25,35,45) -> {1,5}; even (20,30) -> {1,2,5}. Divisibility rather than parity,
    because parity breaks if levels change (21 is odd but 21 % 5 != 0).
    """
    if min_block_size <= 0:
        raise KnobConstraintError(f"min_block_size must be positive, got {min_block_size}")
    return [j for j in JUMP_CANDIDATES if min_block_size % j == 0]

def effective_min_block(min_block_size: int, pelt_jump: int) -> int:
    return int(math.ceil(min_block_size / pelt_jump) * pelt_jump)

def validate_config(cfg: dict, *, context: str = "config", strict: bool = True) -> None:
    problems: list[str] = []
    def fail(msg: str) -> None:
        if strict:
            raise KnobConstraintError(f"[{context}] {msg}")
        problems.append(msg)

    for knob in POSITIVE_INT_KNOBS:
        val = cfg.get(knob)
        if val is None: continue
        if isinstance(val, bool) or not isinstance(val, int) or val < 1:
            fail(f"{knob} must be a positive integer, got {val!r}")

    for knob in ODD_ONLY_KNOBS + ("mad_window_size",):
        val = cfg.get(knob)
        if isinstance(val, int) and val % 2 == 0:
            fail(f"{knob}={val} is EVEN. {knob} is a centred rolling window; an even width "
                 f"cannot be centred symmetrically, so the window actually applied would not "
                 f"be the one requested. Use {val-1} or {val+1}.")

    mbs, jump = cfg.get("min_block_size"), cfg.get("pelt_jump")
    if isinstance(mbs, int) and isinstance(jump, int):
        if jump not in JUMP_CANDIDATES:
            fail(f"pelt_jump={jump} not in {list(JUMP_CANDIDATES)}")
        elif mbs % jump != 0:
            fail(f"pelt_jump={jump} does not divide min_block_size={mbs}: ruptures would "
                 f"enforce an effective min block of {effective_min_block(mbs,jump)}, not "
                 f"{mbs}. Legal jumps: {jumps_for(mbs)}.")

    if isinstance(mbs, int):
        for knob in ODD_ONLY_KNOBS:
            val = cfg.get(knob)
            if isinstance(val, int) and val >= mbs:
                fail(f"{knob}={val} >= min_block_size={mbs}: filter support spans the whole "
                     f"block, so no block of that size can be resolved.")

    if "rt_threshold_na_handling" in cfg:
        try:
            rt_threshold_na_mode(cfg)
        except KnobConstraintError as e:
            fail(str(e))

    nr = cfg.get("irr_null_random_reps")
    if isinstance(nr, int) and nr < 10:
        fail(f"irr_null_random_reps={nr} is below the minimum of 10; the null "
             f"diagnostic discards any comparison with fewer usable replicates, "
             f"leaving an empty results table.")

    mp = cfg.get("min_periods")
    for knob in ("derivative_filter_size", "mad_window_size"):
        dfs = cfg.get(knob)
        if isinstance(dfs, int) and isinstance(mp, int) and mp >= dfs:
            fail(f"min_periods={mp} must be strictly less than {knob}={dfs}; "
                 f"at min_periods == window width the rolling statistic is undefined at "
                 f"every edge and the smoother silently returns all-NaN.")

    if problems and not strict:
        raise KnobConstraintError(f"[{context}] {len(problems)} violation(s):\n  - "
                                  + "\n  - ".join(problems))


def describe_constraints() -> str:
    out = ["KNOB CONSTRAINTS (violations raise; never warned past)",
           f"  odd-only (centred windows) : {', '.join(ODD_ONLY_KNOBS)}",
           f"  jump candidates            : {list(JUMP_CANDIDATES)}",
           "  pelt_jump must divide min_block_size exactly:"]
    for m in (20, 25, 30, 35, 45):
        out.append(f"    min_block_size={m:3d} ({'even' if m%2==0 else 'odd '})"
                   f" -> legal jumps {jumps_for(m)}")
    return "\n".join(out)

# --- validate this notebook's FIXED configuration --------------------------------------
print(describe_constraints())
print()
_knobs = {k: LVL_1_CONFIG[k] for k in (
    "smoothing_window", "derivative_filter_size", "min_block_size",
    "pelt_jump", "min_periods",
) if k in LVL_1_CONFIG}
# Required knob with no default: checked whether or not the key is present.
_knobs["rt_threshold_na_handling"] = rt_threshold_na_mode(LVL_1_CONFIG)
if "hmm_n_states" in LVL_1_CONFIG:
    # hmm_n_states is legitimately a LIST of candidate state counts (cell 9:
    # "Number of hidden states to try") -- the HMM detector fits each candidate
    # and selects internally. Validate every element through the same scalar
    # rule instead of passing the whole list into the positive-int check.
    _hs = LVL_1_CONFIG["hmm_n_states"]
    _hs_list = list(_hs) if isinstance(_hs, (list, tuple)) else [_hs]
    if not _hs_list:
        raise KnobConstraintError("hmm_n_states must not be empty")
    for _s in _hs_list:
        validate_config({"hmm_states": _s},
                        context=f"LVL_1_CONFIG hmm_n_states candidate {_s!r}",
                        strict=True)

# strict=False reports ALL violations at once rather than only the first. It still
# raises: there is no mode in which an illegal knob is tolerated.
validate_config(_knobs, context="LVL_1_CONFIG", strict=False)
print(f"LVL_1_CONFIG passed knob validation: {_knobs}")
if "min_block_size" in _knobs and "pelt_jump" in _knobs:
    _eff = effective_min_block(_knobs["min_block_size"], _knobs["pelt_jump"])
    _nom = _knobs["min_block_size"]
    print(f"  effective_min_block = {_eff} (nominal {_nom})")

In [ ]:
# [cell 12]
# =============================================================================
# RT AND PRACTICE-STIMULUS FILTERING HELPER
# =============================================================================

def apply_rt_filters(df, max_rt_thresh, min_rt_thresh, pr_stim_drop,
                     rt_col="rt_sec", response_col="response", stim_col="stim_order", verbose=True):
    """
    Apply RT threshold and practice-stimulus filters by setting values to NaN.
    Preserves row structure (does not drop rows).

    Each threshold is its own gate: set it to 0 to disable that filter.

    Parameters
    ----------
    df : pd.DataFrame
        Long-format data
    max_rt_thresh : float
        Maximum acceptable RT; 0 disables the slow-RT filter
    min_rt_thresh : float
        Minimum acceptable RT; 0 disables the fast-RT filter
    pr_stim_drop : int
        Practice stimuli to NaN at session start; 0 disables the practice filter
    rt_col : str
        Name of RT column
    response_col : str
        Name of response column
    stim_col : str
        Name of stimulus-order column (default: "stim_order")
    verbose : bool
        Print filtering summary

    Returns
    -------
    pd.DataFrame
        Data with filtered values set to NaN
    """
    df = df.copy()

    n_total = len(df)
    n_max_filtered = 0
    n_min_filtered = 0
    n_ps_filtered = 0

    # Every RT-derived column must be NaN'd wherever the RT itself is: a trial the
    # level excluded must not survive in a derived column. `rt_input` is named
    # explicitly because it does not carry either prefix, and it is the ONLY RT
    # metric LVL_2 sees -- if it escaped filtering, LVL_2 would analyse trials LVL_1
    # had already thrown out.
    _resid_cols = [c for c in df.columns
                   if c.startswith(("predicted_rt", "drift_residual"))
                   or c == "rt_input"]

    # Filter RT > max_rt_thresh
    if max_rt_thresh > 0:
        mask_too_slow = df[rt_col] > max_rt_thresh
        n_max_filtered = mask_too_slow.sum()

        df.loc[mask_too_slow, rt_col] = np.nan
        df.loc[mask_too_slow, response_col] = np.nan
        df.loc[mask_too_slow, _resid_cols] = np.nan

    # Filter RT < min_rt_thresh
    if min_rt_thresh > 0:
        mask_too_fast = df[rt_col] < min_rt_thresh
        n_min_filtered = mask_too_fast.sum()

        df.loc[mask_too_fast, rt_col] = np.nan
        df.loc[mask_too_fast, response_col] = np.nan
        df.loc[mask_too_fast, _resid_cols] = np.nan

    # Filter responses to practice stimuli
    if pr_stim_drop > 0:
        mask_pract_stim = df[stim_col] <= pr_stim_drop
        n_ps_filtered = mask_pract_stim.sum()

        df.loc[mask_pract_stim, rt_col] = np.nan
        df.loc[mask_pract_stim, response_col] = np.nan
        df.loc[mask_pract_stim, _resid_cols] = np.nan

    if verbose:
        print("=" * 60)
        print("FILTERING SUMMARY")
        print("=" * 60)
        print(f"  max_rt_thresh  (>{max_rt_thresh}s):          {'ON' if max_rt_thresh > 0 else 'OFF'}")
        print(f"  min_rt_thresh  (<{min_rt_thresh}s):          {'ON' if min_rt_thresh > 0 else 'OFF'}")
        print(f"  pr_stim_drop   ({stim_col}<={pr_stim_drop}): {'ON' if pr_stim_drop > 0 else 'OFF'}")
        print("-" * 60)

        # counts can overlap (e.g. a too-slow practice trial counts twice),
        # so this is an upper bound on the number of distinct cells NaN'd.
        total_filtered = n_max_filtered + n_min_filtered + n_ps_filtered
        if total_filtered > 0 or max_rt_thresh > 0 or min_rt_thresh > 0 or pr_stim_drop > 0:
            print(f"  Slow RT (>{max_rt_thresh}s) set to NaN:      {n_max_filtered} ({100*n_max_filtered/n_total:.2f}%)")
            print(f"  Fast RT (<{min_rt_thresh}s) set to NaN:      {n_min_filtered} ({100*n_min_filtered/n_total:.2f}%)")
            print(f"  Practice stimuli set to NaN:                 {n_ps_filtered} ({100*n_ps_filtered/n_total:.2f}%)")
            print(f"  Total NaN'd (upper bound, overlaps possible):{total_filtered} ({100*total_filtered/n_total:.2f}%)")
        else:
            print("  No filtering applied (all thresholds 0)")
        print("=" * 60)

    return df


# =============================================================================
# SHARED ROBUST SCALING (MAD) -- the single implementation used by every level
# =============================================================================
# Robust z = (x - median) / (MAD_TO_SD * MAD [+ eps]). Every MAD-based centring /
# scaling in LVL_1 and LVL_2 calls these functions. What a zero (or near-zero) MAD
# turns into differs by site; that choice is an explicit `zero_mad` argument at each
# call site rather than a hand-written variant. A zero MAD occurs in sessions whose
# Entropy (win) is 0 in most windows (long straight-lined stretches).

MAD_TO_SD = 1.4826   # 1.4826 x MAD estimates the SD of a normal distribution


def mad(x, axis=0, skipna=True):
    """Raw (unscaled) median absolute deviation about the median."""
    med_fn = np.nanmedian if skipna else np.median
    x = np.asarray(x, dtype=float)
    return med_fn(np.abs(x - np.nanmedian(x, axis=axis)), axis=axis)


def robust_z(x, center, raw_mad, eps=1e-8):
    """(x - center) / (MAD_TO_SD * raw_mad + eps) -- the one robust-z formula."""
    return (x - center) / (raw_mad * MAD_TO_SD + eps)


def robust_center_scale(x, zero_mad="floor", eps=1e-8, sd_below=1e-6, skipna_mad=True):
    """Median and robust scale of x (column-wise for a DataFrame / 2-D array).

    zero_mad -- how a (near-)zero MAD is handled:
      "floor": scale = MAD_TO_SD * MAD + eps   (MAD = 0 gives scale = eps)
      "sd":    a MAD <= sd_below is replaced by the SD (ddof=1) first, then as "floor"
      "zeros": scale = MAD_TO_SD * MAD + eps with eps normally 0; robust_scale returns
               0 where that scale is not > 0 (also when it is NaN)
    The center is the NaN-skipping median. skipna_mad=False takes the MAD with
    np.median, so any NaN in x makes it NaN.
    Returns (center, scale): floats for 1-D input, per-column Series for a DataFrame,
    per-column arrays for a 2-D array."""
    if zero_mad not in ("floor", "sd", "zeros"):
        raise ValueError(f"robust_center_scale: unknown zero_mad={zero_mad!r}")
    if isinstance(x, pd.DataFrame):
        pairs = [robust_center_scale(x[c], zero_mad, eps, sd_below, skipna_mad) for c in x.columns]
        return (pd.Series([p[0] for p in pairs], index=x.columns, dtype=float),
                pd.Series([p[1] for p in pairs], index=x.columns, dtype=float))
    arr = np.asarray(x, dtype=float)
    if arr.ndim == 2:
        pairs = [robust_center_scale(arr[:, j], zero_mad, eps, sd_below, skipna_mad) for j in range(arr.shape[1])]
        return np.array([p[0] for p in pairs]), np.array([p[1] for p in pairs])
    center = float(np.nanmedian(arr))
    raw = float((np.nanmedian if skipna_mad else np.median)(np.abs(arr - center)))
    if zero_mad == "sd" and not raw > sd_below:
        raw = float(pd.Series(arr).std())
    return center, raw * MAD_TO_SD + eps


def robust_scale(x, zero_mad="floor", eps=1e-8, sd_below=1e-6, skipna_mad=True):
    """Robust z of x (column-wise for a DataFrame / 2-D array); see robust_center_scale."""
    center, scale = robust_center_scale(x, zero_mad, eps, sd_below, skipna_mad)
    if zero_mad != "zeros":
        return (x - center) / scale
    if isinstance(x, pd.DataFrame):
        ok = scale > 0
        return pd.DataFrame({c: ((x[c] - center[c]) / scale[c]) if ok[c] else 0.0 for c in x.columns},
                            index=x.index)
    if isinstance(x, pd.Series):
        return (x - center) / scale if scale > 0 else pd.Series(0.0, index=x.index)
    arr = np.asarray(x, dtype=float)
    return (arr - center) / scale if scale > 0 else np.zeros_like(arr)


In [ ]:
# [cell 13]
# =============================================================================
# DATA RESHAPING: LONG - TRIAL-LEVEL
# =============================================================================

def long_to_stim_level(df, config, participant_col='id', session_col='session', 
                        stim_col='stim_order', scale_col='scale'):
    """
    Convert long-format data to trial-level features.
    
    Parameters
    ----------
    df : pd.DataFrame
        Long-format data with one row per scale rating
    config : dict
        Configuration dictionary (required)
    
    Returns
    -------
    pd.DataFrame
        Trial-level data with one row per trial
        Preserves stim_label if present in input data.
    """
    # validate config['scale_ranges'] presence with a clear error.
    if "scale_ranges" not in config or not config["scale_ranges"]:
        raise ValueError(
            "long_to_stim_level: config['scale_ranges'] is required and must be non-empty"
        )
    scales = list(config["scale_ranges"].keys())

    # Resolve which RT representation feeds {scale}_rt / {scale}_residual*.
    # feature_rt_transform picks raw/log/boxcox; ses_wise_drift and
    # age_adjust_drift_residuals pick the base/age variant within it. LVL_0
    # computes all three representations (cell 7); this only selects which one
    # LVL_1 reads. use_residuals (read downstream by get_feature_column) then
    # picks the value vs. the residual WITHIN this same representation.
    _RT_VALUE_COL = {"raw": "rt_sec", "log": "log_rt", "boxcox": "bc_rt"}
    _RT_TAG = {"raw": "raw", "log": "log", "boxcox": "bc"}
    _transform = config["feature_rt_transform"]
    _rtval_col = _RT_VALUE_COL[_transform]
    _tag = _RT_TAG[_transform]
    _base = "per_ses" if config["ses_wise_drift"] else "overall"
    # Explicit flag, validated against what LVL_0 actually produced (also checked in
    # the LVL_1 run cell): a missing _age column must fail, not silently drop the pivot.
    _age_sfx = "_age" if config["age_adjust_drift_residuals"] else ""
    if _age_sfx and f"drift_residual_{_tag}_{_base}_age" not in df.columns:
        raise SystemExit(
            f"long_to_stim_level: age_adjust_drift_residuals=True but "
            f"drift_residual_{_tag}_{_base}_age is not in the LVL_0 handoff.")
    _pred_col = f"predicted_rt_{_tag}_{_base}"
    _resid_col = f"drift_residual_{_tag}_{_base}{_age_sfx}"

    df = df.copy()

    # Ensure numeric types
    df['rt_sec'] = pd.to_numeric(df['rt_sec'], errors='coerce')
    df['response'] = pd.to_numeric(df['response'], errors='coerce')
    if _rtval_col in df.columns:
        df[_rtval_col] = pd.to_numeric(df[_rtval_col], errors='coerce')
    if _pred_col in df.columns:
        df[_pred_col] = pd.to_numeric(df[_pred_col], errors='coerce')
    if _resid_col in df.columns:
        df[_resid_col] = pd.to_numeric(df[_resid_col], errors='coerce')
        df['_feature_residual_abs'] = df[_resid_col].abs()
    # rt_input: the single canonical RT metric resolved at cell 20. Pivoted to
    # {scale}_rt_input so LVL_2 can build all of its block features from ONE
    # column family instead of re-deriving the raw-vs-residual choice.
    if 'rt_input' in df.columns:
        df['rt_input'] = pd.to_numeric(df['rt_input'], errors='coerce')
    
    # Preserve stim_label: get unique stim_label per (participant, session, stim)
    # stim_label is the same for all scales within a stimulus
    stim_label_df = None
    if 'stim_label' in df.columns:
        # warn if any (pid, session, stim) cell has > 1 distinct stim_label;
        # .first() would silently swallow the inconsistency.
        _nunique_lab = (
            df.groupby([participant_col, session_col, stim_col])['stim_label']
              .nunique(dropna=True)
        )
        if (_nunique_lab > 1).any():
            import warnings as _w_ls4
            _w_ls4.warn(
                'long_to_stim_level: multiple stim_label values within the same '
                '(participant, session, stim_order); .first() used.',
                stacklevel=2,
            )
        stim_label_df = df.groupby([participant_col, session_col, stim_col])['stim_label'].first().reset_index()
    
    # Preserve condition: get unique condition per (participant, session, stim)
    # condition is the same for all rows of a participant (between-subjects)
    condition_df = None
    if 'condition' in df.columns:
        _nunique_cond = (
            df.groupby([participant_col, session_col, stim_col])['condition']
              .nunique(dropna=True)
        )
        if (_nunique_cond > 1).any():
            import warnings as _w_ls4b
            _w_ls4b.warn(
                'long_to_stim_level: multiple condition values within the same '
                '(participant, session, stim_order); .first() used.',
                stacklevel=2,
            )
        condition_df = df.groupby([participant_col, session_col, stim_col])['condition'].first().reset_index()
    
    # Pivot RT values (from whichever representation feature_rt_transform selects)
    rt_pivot = df.pivot_table(
        index=[participant_col, session_col, stim_col],
        columns=scale_col,
        values=_rtval_col,
        aggfunc='first'
    ).reset_index()
    rt_pivot.columns = [f"{c}_rt" if c in scales else c for c in rt_pivot.columns]
    
    # Pivot response values
    resp_pivot = df.pivot_table(
        index=[participant_col, session_col, stim_col],
        columns=scale_col,
        values='response',
        aggfunc='first'
    ).reset_index()
    resp_pivot.columns = [f"{c}_response" if c in scales else c for c in resp_pivot.columns]
    
    # Pivot predicted RT (same representation/base as the residual below)
    if _pred_col in df.columns:
        pred_pivot = df.pivot_table(
            index=[participant_col, session_col, stim_col],
            columns=scale_col,
            values=_pred_col,
            aggfunc='first'
        ).reset_index()
        pred_pivot.columns = [f"{c}_predicted" if c in scales else c for c in pred_pivot.columns]
    else:
        pred_pivot = None
    
    # Pivot drift residuals
    if _resid_col in df.columns:
        resid_pivot = df.pivot_table(
            index=[participant_col, session_col, stim_col],
            columns=scale_col,
            values=_resid_col,
            aggfunc='first'
        ).reset_index()
        resid_pivot.columns = [f"{c}_residual" if c in scales else c for c in resid_pivot.columns]
    else:
        resid_pivot = None
    
    # Pivot rt_input (canonical LVL_2 hand-off metric)
    if 'rt_input' in df.columns:
        rtin_pivot = df.pivot_table(
            index=[participant_col, session_col, stim_col],
            columns=scale_col,
            values='rt_input',
            aggfunc='first'
        ).reset_index()
        rtin_pivot.columns = [f"{c}_rt_input" if c in scales else c for c in rtin_pivot.columns]
    else:
        rtin_pivot = None

    # Pivot absolute residuals
    if '_feature_residual_abs' in df.columns:
        resid_abs_pivot = df.pivot_table(
            index=[participant_col, session_col, stim_col],
            columns=scale_col,
            values='_feature_residual_abs',
            aggfunc='first'
        ).reset_index()
        resid_abs_pivot.columns = [f"{c}_residual_abs" if c in scales else c for c in resid_abs_pivot.columns]
    else:
        resid_abs_pivot = None
    
    # Merge all pivots
    stim_lvl_df = rt_pivot.copy()
    
    for pivot_df in [resp_pivot, pred_pivot, resid_pivot, resid_abs_pivot, rtin_pivot]:
        if pivot_df is None:
            continue
        merge_cols = [c for c in pivot_df.columns if c not in [participant_col, session_col, stim_col]]
        stim_lvl_df = stim_lvl_df.merge(
            pivot_df[[participant_col, session_col, stim_col] + merge_cols],
            on=[participant_col, session_col, stim_col],
            how='left'
        )
    
    # Merge stim_label if it was present in input
    if stim_label_df is not None:
        stim_lvl_df = stim_lvl_df.merge(
            stim_label_df,
            on=[participant_col, session_col, stim_col],
            how='left'
        )
    
    # Merge condition if it was present in input
    if condition_df is not None:
        stim_lvl_df = stim_lvl_df.merge(
            condition_df,
            on=[participant_col, session_col, stim_col],
            how='left'
        )
    
    # Rename for consistency
    stim_lvl_df = stim_lvl_df.rename(columns={
        participant_col: 'id',
        session_col: 'session', 
        stim_col: 'stim_order'
    })
    
    return stim_lvl_df

In [ ]:
# [cell 14]
# =============================================================================
# ENFORCE MINIMUM BLOCK LENGTH
# =============================================================================

def enforce_min_block_length(blocks, config):
    """
    Merge small blocks into neighbors until all blocks >= min_block_size.
    
    Parameters
    ----------
    blocks : array-like
        Block assignments
    config : dict
        Configuration dictionary (required)
    
    Returns
    -------
    np.ndarray
        Adjusted block assignments
    """
    min_len = config["min_block_size"]
    
    blocks = np.asarray(blocks)
    n = len(blocks)
    
    if n == 0:
        return blocks
    if n < min_len:
        return np.zeros(n, dtype=int)
    
    # Build contiguous segments
    segments = []
    start = 0
    for i in range(1, n):
        if blocks[i] != blocks[i - 1]:
            segments.append([start, i])
            start = i
    segments.append([start, n])
    
    # Iteratively merge until all segments >= min_len
    # hard cap at 2*n iterations — each iteration removes one
    # segment, and pathological inputs (e.g. min_len > n/2 with alternating
    # singletons) shouldn't loop indefinitely.
    _max_iter = 2 * n + 10
    _iter = 0
    while True:
        _iter += 1
        if _iter > _max_iter:
            import warnings as _w_m5
            _w_m5.warn(
                f"enforce_min_block_length: hit iteration cap {_max_iter}; returning current segments",
                stacklevel=2,
            )
            break
        lengths = np.array([end - beg for (beg, end) in segments], dtype=int)
        if len(lengths) == 0:
            break
        
        min_idx = int(np.argmin(lengths))
        smallest = int(lengths[min_idx])
        
        if smallest >= min_len:
            break
        if len(segments) == 1:
            break
        
        if min_idx == 0:
            merge_idx = 1
        elif min_idx == len(segments) - 1:
            merge_idx = len(segments) - 2
        else:
            left_len = segments[min_idx - 1][1] - segments[min_idx - 1][0]
            right_len = segments[min_idx + 1][1] - segments[min_idx + 1][0]
            merge_idx = min_idx - 1 if left_len <= right_len else min_idx + 1
        
        i1, i2 = sorted([min_idx, merge_idx])
        s1, e1 = segments[i1]
        s2, e2 = segments[i2]
        new_start = min(s1, s2)
        new_end = max(e1, e2)
        
        segments.pop(i2)
        segments.pop(i1)
        
        insert_pos = 0
        while insert_pos < len(segments) and segments[insert_pos][0] < new_start:
            insert_pos += 1
        segments.insert(insert_pos, [new_start, new_end])
    
    new_blocks = np.zeros(n, dtype=int)
    for bid, (beg, end) in enumerate(segments):
        new_blocks[beg:end] = bid
    
    return new_blocks

In [ ]:
# [cell 15]
# =============================================================================
# BLOCK DETECTION METHODS
# =============================================================================
# Block ordering: order_blocks_by_inattention() (block 0 = most inattentive);
# PCA composite sign: PC1 oriented by the direction-weighted feature mean.

# Single source of truth for the display-name <-> column-name <-> direction
# contract. Every consumer of feature-display names (get_feature_column,
# detect_blocks_hmm's state ordering, create_block_methods.get_defined_on_single,
# get_validation_map, mad_center_columns, summarize_block_methods) should derive its
# mapping from this dict instead of hard-coding parallel tables. Adding a new feature
# becomes ONE entry here.
# `direction` semantics:
#   +1 = high value indicates MORE inattentive (variable, random)
#   -1 = high value indicates MORE attentive   (slow/deliberate, coherent across scales)
# NOTE on Adj RT Level (Stim): the project's operational convention is that LOW RT
# indicates inattention (rushing) -- consistent with inattent_behav's fast-RT
# threshold and the LVL_2 component selection. So Adj RT Level (Stim) has
# direction=-1 (high RT = attentive/deliberate).
# `defined_on` mirrors create_block_methods.get_defined_on_single (cell 18):
# the validation step skips features whose `defined_on` matches the method's
# own defined_on tag, so a method cannot "validate" itself.
# NAMING NOTE, so this is not "corrected" later. The three LEVEL 1 detection features
# (manuscript notation in brackets) are:
#   "Adj RT Level (Stim)" [$AdjRT_{stim}$]  stimulus-level drift-adjusted RT
#   "Adj RT Var (win)"    [$AdjVar_{win}$]  sliding-window (Esterman VTC) variance of it
#   "Entropy (win)"       [$Ent_{win}$]     sliding-window normalised response entropy
# They are NOT the Level 2 block features ("Adj RT Level (Block)", "Exp RT Var (Block)",
# "Exp Entropy (Block)"), which are rebuilt per block in the LVL_2 cells.
# The registry entry literally named "RT Variance" is the cross-scale within-stimulus
# variance; it is not used for detection or validation. All of cp_features /
# pelt_features / hmm_features use the three names above. Method IDs are derived from
# these names (spaces -> "_", parens dropped): CPD_Adj_RT_Level_Stim, HMM_2_Entropy_win.
# validation_prefix names the per-row validation columns (<prefix>_F / _p / _effsize /
# _p_holm_within): adj_rt_level_stim, adj_rt_var_win, entropy_win.
FEATURE_REGISTRY = {
    "Adj RT Level (Stim)": {"rt_col": "aggregated_rt",       "residual_col": "aggregated_residual",      "direction": -1, "defined_on": "adj_rt_level_stim",
                        "validation_prefix": "adj_rt_level_stim"},
    "RT Variance":     {"rt_col": "rt_variance",         "residual_col": "residual_variance",        "direction": +1, "defined_on": "variance",
                        "validation_prefix": "rt_var"},
    # Temporal (sliding-window) RT variance: each scale's variance over a
    # derivative_filter_size window (centred when rolling_center=True), averaged across
    # scales (temporal_var_method="per_scale") -- the Esterman variance-time-course idea.
    # Distinct from "RT Variance" (cross-scale, within-stimulus).
    "Adj RT Var (win)":    {"rt_col": "local_rt_var",        "residual_col": "local_residual_var",       "direction": +1, "defined_on": "variance",
                        "validation_prefix": "adj_rt_var_win"},
    # Coherence (RT): mean pairwise distance of RTs across scales for a given
    # stimulus (instantaneous, cross-scale). High distance = low cross-scale
    # agreement = MORE inattentive.
    "Coherence (RT)":  {"rt_col": "coherence_rt",  "residual_col": "coherence_residual",      "direction": +1, "defined_on": "coherence",
                        "validation_prefix": "coherence_rt"},
    # Coherence (Var): pairwise distance of each scale's rolling RT variance
    # (cross-scale dispersion of the local variance signal, not a single scale's
    # own variance). High distance = MORE inattentive.
    "Coherence (Var)": {"rt_col": "coherence_vars", "residual_col": "coherence_residual_vars", "direction": +1, "defined_on": "coherence",
                        "validation_prefix": "coherence_var"},
    "Entropy (win)":       {"rt_col": "rating_entropy_norm", "residual_col": "rating_entropy_norm",      "direction": +1, "defined_on": "entropy",
                        "validation_prefix": "entropy_win"},
    "Rating Variance": {"rt_col": "rating_variance",     "residual_col": "rating_variance",          "direction": +1, "defined_on": "variance",
                        "validation_prefix": "rate_var"},
}


def get_feature_column(display_name, features, config):
    """
    Map feature display name to actual column name based on use_residuals setting.

    Reads from FEATURE_REGISTRY (single source of truth, defined above).

    Parameters
    ----------
    display_name : str
        Must be a key of FEATURE_REGISTRY (e.g. "Adj RT Level (Stim)", "RT Variance",
        "Coherence (RT)", "Coherence (Var)", "Entropy (win)", "Rating Variance").
    features : pd.DataFrame
        Feature dataframe to check column availability.
    config : dict
        Config with use_residuals setting.

    Returns
    -------
    str or None
        Column name if available, None otherwise. When use_residuals=True and
        the residual column is missing, falls back to the RT column.
    """
    spec = FEATURE_REGISTRY.get(display_name)
    if spec is None:
        return None
    use_residuals = config["use_residuals"]
    primary = spec["residual_col"] if use_residuals else spec["rt_col"]
    if primary in features.columns:
        return primary
    # When the residual column is missing (e.g. the drift fit failed for this group), the RT column is used. With
    # use_residuals=False a missing RT column means something is wrong upstream, so None is returned.
    if use_residuals:
        rt_col = spec["rt_col"]
        if rt_col in features.columns:
            return rt_col
    return None


def apply_smoothing(signal, method, window):
    """
    Apply smoothing to a signal before changepoint detection.
    
    Parameters
    ----------
    signal : np.ndarray
        Input signal
    method : str
        "none", "boxcar", or "gaussian"
    window : int
        Smoothing window size
    
    Returns
    -------
    np.ndarray
        Smoothed signal
    """
    # explicit guard against invalid window sizes.
    if window <= 0:
        raise ValueError(f"smoothing_window must be > 0, got {window}")
    if method == "none" or window <= 1:
        return signal
    
    signal = np.asarray(signal, dtype=float)
    
    # NaN guard for other callers (e.g. plotting). The detectors never pass NaN: they
    # receive observed stimuli only (_on_observed_rows). np.convolve would spread a NaN
    # across the whole window, so any NaN is nearest-filled here before smoothing.
    if np.any(np.isnan(signal)):
        signal = pd.Series(signal).ffill().bfill().values
        if np.any(np.isnan(signal)):
            # All-NaN input: return as-is (cannot smooth nothing).
            return signal
    
    if method == "boxcar":
        kernel = np.ones(window) / window
    elif method == "gaussian":
        sigma = window / 6
        x = np.arange(window) - window // 2
        kernel = np.exp(-0.5 * (x / sigma) ** 2)
        kernel = kernel / kernel.sum()
    else:
        # Unknown smoothing method (typo? new value?). Warn rather than
        # silently return the unsmoothed signal -- otherwise a misspelt
        # LVL_1_CONFIG["smoothing_method"] disables smoothing without any signal.
        import warnings as _w_sm
        _w_sm.warn(
            f"apply_smoothing: unknown method '{method}'; expected one of "
            f"'none', 'boxcar', 'gaussian'. Returning signal unchanged.",
            stacklevel=2,
        )
        return signal
    
    # symmetric padding: avoids a one-element phase bias on even windows.
    pad_left = window // 2
    pad_right = (window - 1) // 2
    padded = np.pad(signal, (pad_left, pad_right), mode='reflect')
    smoothed = np.convolve(padded, kernel, mode='valid')
    
    if len(smoothed) > len(signal):
        smoothed = smoothed[:len(signal)]
    elif len(smoothed) < len(signal):
        smoothed = np.pad(smoothed, (0, len(signal) - len(smoothed)), mode='edge')
    
    return smoothed


def detect_blocks_fixed(n_trials, block_size):
    """Fixed-size blocks (ceiling division avoids orphan label)."""
    if block_size <= 0:
        raise ValueError(f"block_size must be > 0, got {block_size}")
    if n_trials <= 0:
        return np.zeros(0, dtype=int)
    n_blocks = (n_trials + block_size - 1) // block_size  # ceiling division
    return np.repeat(np.arange(n_blocks), block_size)[:n_trials]


# within-session MAD-z helper
def mad_center_columns(features):
    """Within-session robust (MAD) z-scoring of the segmentation feature columns,
    applied BEFORE MAD-CPD / HMM / PELT so every feature is on a common scale.
    Makes pelt_pen and cp_k comparable across features on different units.
    Idempotent w.r.t. the HMM / multi-feature-PELT internal MAD-z (MAD-z of MAD-z
    == MAD-z) whenever MAD > 0 (MAD = 0 only in heavily straight-lined sessions); single-feature
    PELT and the CPD threshold become scale-invariant. Returns a COPY with only the mapped feature columns
    rescaled -- the original `features` (validation / quality / block matrices) is
    untouched."""
    cols = set()
    for spec in FEATURE_REGISTRY.values():
        for key in ("rt_col", "residual_col"):
            col = spec.get(key)
            if col and col in features.columns:
                cols.add(col)
    if not cols:
        return features
    out = features.copy()
    for col in cols:
        out[col] = robust_scale(out[col].to_numpy(dtype=float), zero_mad="floor")
    return out


# EM stopping rule for detect_blocks_hmm. hmmlearn's stock ConvergenceMonitor stops
# when the SIGNED change in log-likelihood is < tol, so ANY decrease ends EM. With the
# transition prior the M-step maximises log-likelihood + log-prior; the plain
# log-likelihood can dip while that objective is still rising, and the stock rule
# stopped those fits at the dip. This monitor stops on the ABSOLUTE change instead.
# Installed the way hmmlearn documents it: point model.monitor_ at a subclass
# before fit().
from hmmlearn.base import ConvergenceMonitor as _HMMConvergenceMonitor


class AbsDeltaConvergenceMonitor(_HMMConvergenceMonitor):
    """Converged when |loglik_t - loglik_(t-1)| < tol, or when n_iter is reached."""
    @property
    def converged(self):
        return (self.iter == self.n_iter or
                (len(self.history) >= 2 and
                 abs(self.history[-1] - self.history[-2]) < self.tol))


# HMM transmat prior + per-feature smoothing
def detect_blocks_hmm(features, n_states, feature_list, config, participant_id=None, session_id=None):
    """
    Generic HMM block detection using configurable feature list.
    
    Parameters
    ----------
    features : pd.DataFrame
        Feature dataframe
    n_states : int
        Number of HMM states
    feature_list : list
        List of feature display names (e.g., ["Adj RT Level (Stim)", "Adj RT Var (win)"])
    config : dict
        Configuration with HMM settings
        
    Returns
    -------
    np.ndarray
        Block assignments
    """
    n_iter = config["hmm_n_iter"]
    tol = float(config["hmm_tol"])
    random_state = config["hmm_random_state"]
    cov_type = config["hmm_covariance_type"]
    
    # Map display names to column names. keep parallel
    # display-name list so the post-fit state-ordering can apply
    # direction-aware MAD-z scoring instead of summing raw means.
    cols = []
    display_for_col = []
    for name in feature_list:
        col = get_feature_column(name, features, config)
        if col:
            cols.append(col)
            display_for_col.append(name)
    
    if len(cols) < 1:
        return np.zeros(len(features), dtype=int)
    
    # Prepare feature matrix. No fill: every detector receives only stimuli whose
    # features are all observed (_on_observed_rows, cell 18); rolling-window edges and
    # NaN stimuli (e.g. rt_threshold_na_handling="drop_stimulus") are removed before this
    # call and inherit a block after.
    X = features[cols].to_numpy(dtype=float)

    # Robust z per column (shared helper; MAD = 0 -> eps floor).
    X = robust_scale(X, zero_mad="floor")

    # Smooth each feature column before fitting, with the same smoothing_method /
    # smoothing_window as detect_blocks_cpd and detect_blocks_pelt ('none' = no-op).
    _sm_method = config["smoothing_method"]
    _sm_window = config["smoothing_window"]
    X = X.astype(float)
    for _j in range(X.shape[1]):
        X[:, _j] = apply_smoothing(X[:, _j], _sm_method, _sm_window)

    # state-persistence prior on the transition matrix. We want the
    # EFFECTIVE pseudo-counts hmmlearn adds to be exactly `hmm_transmat_self_prior`
    # on the diagonal and ZERO off-diagonal. hmmlearn's MAP step computes
    # (transmat_prior - 1 + observed_counts), so we pass (self + 1) on the diagonal
    # and 1 off-diagonal: the -1 cancels everywhere -> off-diagonals add nothing
    # (pure observed counts, no suppression of lone switches) and each diagonal
    # adds exactly `self` pseudo self-transitions. It is a pseudo-COUNT, not a
    # percentage; its strength is relative to the ~T-1 observed transitions per
    # session. Higher self = stickier states = fewer, longer blocks. 0 -> no
    # prior (matches hmmlearn's transmat_prior=1.0 default = pure ML EM).
    _self_p = float(config["hmm_transmat_self_prior"])
    _transmat_prior = np.ones((n_states, n_states), dtype=float)   # off-diag 1 -> effective 0
    np.fill_diagonal(_transmat_prior, _self_p + 1.0)               # diag -> effective +_self_p

    # Set inside the function on purpose: block detection runs under joblib/loky, and
    # child processes do not inherit the parent's logging configuration.
    logging.getLogger("hmmlearn").setLevel(logging.CRITICAL)

    try:
        model = hmm.GaussianHMM(n_components=n_states, covariance_type=cov_type,
                                n_iter=n_iter, tol=tol, random_state=random_state,
                                transmat_prior=_transmat_prior, verbose=False)
        model.monitor_ = AbsDeltaConvergenceMonitor(tol, n_iter, False)
        model.fit(X)
        # hmmlearn's monitor_.converged is True whenever iter == n_iter, so it reports
        # success for a fit that merely exhausted its iteration budget. Decide from the
        # log-likelihood history instead: hitting the cap, or a final decrease larger
        # than tol. Changes within +/-tol are what the stopping rule itself treats as
        # converged, so they are not flagged.
        _mon = getattr(model, "monitor_", None)
        _hist = list(getattr(_mon, "history", []) or [])
        _delta = (_hist[-1] - _hist[-2]) if len(_hist) >= 2 else None
        _hit_cap = bool(_mon is not None and _mon.iter >= n_iter)
        _decreased = bool(_delta is not None and _delta < -tol)
        if _hit_cap or _decreased:
            _why = "hit_iter_cap" if _hit_cap else "loglik_decreased"
            print(f"[HMM non-convergence] id={participant_id} session={session_id} "
                  f"n_states={n_states} n_iter={n_iter} reason={_why} "
                  f"delta={_delta if _delta is not None else float('nan'):+.4g}")
        states = model.predict(X)
        
        # Order states by composite score (higher = more inattentive).
        # Direction-aware MAD-z scoring:
        # Raw summed means are dominated by RT-Level scale; MAD-normalized
        # contributions with explicit signs respect the theoretical direction
        # of each feature (high coherence = MORE attentive, etc.).
        # NOTE: state ordering is QC/diagnostic only — block boundaries
        # drive downstream analysis, ordering does not.
        # Direction sign per feature -- single source of truth lives in
        # FEATURE_REGISTRY. The feature is spelled "Coherence (Var)".
        _state_directions = {name: spec["direction"] for name, spec in FEATURE_REGISTRY.items()}
        _col_meds, _col_mads = robust_center_scale(features[cols], zero_mad="floor")
        state_scores = []
        for s in range(n_states):
            mask = states == s
            if mask.sum() > 0:
                score = 0.0
                for _col, _disp in zip(cols, display_for_col):
                    _dir = _state_directions.get(_disp, +1)  # unknown features default to +1
                    _z = (features.loc[mask, _col].mean() - _col_meds[_col]) / _col_mads[_col]
                    score += _dir * float(_z)
            else:
                score = 0.0
            state_scores.append(score)
        
        # Higher score = more inattentive (State 0)
        order = np.argsort(state_scores)[::-1]
        state_map = {old: new for new, old in enumerate(order)}
        ordered_states = np.array([state_map[s] for s in states])
        # Log HMM diagnostic (block count = number of runs in state sequence).
        n_runs = 1 + int(np.sum(np.diff(ordered_states) != 0)) if len(ordered_states) > 0 else 0
        _CPD_DIAGNOSTICS.append({
            "participant": participant_id,
            "session": session_id,
            "method": f"HMM_{n_states}",
            "n_trials": len(features),
            "mad": np.nan,
            "median_diff": np.nan,
            "mad_window_size": None,
            "k_used": np.nan,
            "n_changepoints": max(0, n_runs - 1),
            "n_blocks": n_runs,
            "features_used": ",".join(feature_list),
            "hmm_converged": not (_hit_cap or _decreased),
        })
        return ordered_states
    except Exception as _e_hmm:
        # A failed fit returns one block (all zeros); it is recorded as a failure in the diagnostics and, through
        # process_session, in the results (detector_failed), so it is not read as a genuine one-block segmentation.
        import warnings as _w_c3
        _w_c3.warn(
            f"detect_blocks_hmm: HMM failed for id={participant_id} session={session_id} "
            f"(n_states={n_states}, features={feature_list}) ({type(_e_hmm).__name__}: {_e_hmm}); "
            f"returning one block, recorded as detector_failed",
            stacklevel=2,
        )
        _CPD_DIAGNOSTICS.append({
            "participant": participant_id, "session": session_id, "method": f"HMM_{n_states}",
            "n_trials": len(features), "n_blocks": 1, "features_used": ",".join(feature_list),
            "detector_failed": True,
        })
        return np.zeros(len(features), dtype=int)

_CPD_DIAGNOSTICS = []


def get_cpd_diagnostics_df():
    """Return accumulated CPD diagnostics as a DataFrame (one row per call)."""
    return pd.DataFrame(_CPD_DIAGNOSTICS) if _CPD_DIAGNOSTICS else pd.DataFrame()



def _compute_diff_thresholds(diffs, k, mad_window_size, min_periods=None, center=True):
    """
    Compute per-gap (threshold, median_diff, mad) for changepoint detection.

    Parameters
    ----------
    diffs : np.ndarray
        Absolute first-differences of the signal, length n-1.
    k : float
        Sensitivity multiplier.
    mad_window_size : int or None
        None -> global scalar MAD over all diffs.
        int  -> centered sliding window of that many stimuli around each gap.

    Returns
    -------
    threshold : np.ndarray (length n-1)
    median_diff_summary : float (scalar; for windowed path returns median of local medians)
    mad_summary : float (scalar; for windowed path returns median of local MADs)
    """
    if mad_window_size is None or int(mad_window_size) >= len(diffs):
        med = float(np.median(diffs))
        m = float(mad(diffs))
        local_med = np.full_like(diffs, med, dtype=float)
        local_mad = np.full_like(diffs, m, dtype=float)
        thr = local_med + k * local_mad * MAD_TO_SD
        return thr, local_med, local_mad
    # pandas rolling with center=rolling_center and min_periods, like the rolling
    # features. These are THRESHOLDS, not feature values: undefined edge thresholds
    # take the nearest defined threshold.
    w = int(mad_window_size)
    s = pd.Series(diffs, dtype=float)
    roll = s.rolling(w, center=center, min_periods=min_periods)
    local_med = roll.median()
    local_mad = roll.apply(mad, raw=True)
    local_med = local_med.bfill().ffill().to_numpy()
    local_mad = local_mad.bfill().ffill().to_numpy()
    thr = local_med + k * local_mad * MAD_TO_SD
    return thr, local_med, local_mad


# ======================================================================================
# Direction-aware block ordering, shared by detect_blocks_cpd and detect_blocks_pelt.
# Convention: BLOCK 0 = MOST INATTENTIVE, matching the state ordering of detect_blocks_hmm
# (a direction-weighted MAD-z composite). Ordering by the raw signal mean instead would
# reverse this on a direction = -1 feature (e.g. "Adj RT Level (Stim)", where SLOW =
# attentive), so block 0 would mean different things for different detectors.
# Sign comes from FEATURE_REGISTRY[...]["direction"].
# ======================================================================================
def order_blocks_by_inattention(blocks, features, cols, display_names):
    """Renumber blocks so that block 0 is the most inattentive.

    Parameters
    ----------
    blocks : np.ndarray
        Integer block assignment per trial.
    features : pd.DataFrame
        Feature frame the blocks were detected on.
    cols : list[str]
        Concrete column names used for detection.
    display_names : list[str]
        FEATURE_REGISTRY display names, parallel to `cols`.

    Returns
    -------
    np.ndarray
        Renumbered block assignments.
    """
    unique_blocks = np.unique(blocks)
    if len(unique_blocks) <= 1:
        return blocks

    directions = {name: spec["direction"] for name, spec in FEATURE_REGISTRY.items()}
    col_meds, col_mads = robust_center_scale(features[cols], zero_mad="floor")

    scores = []
    for b in unique_blocks:
        mask = blocks == b
        if mask.sum() == 0:
            scores.append(0.0)
            continue
        score = 0.0
        for col, disp in zip(cols, display_names):
            direction = directions.get(disp, +1)   # unknown features default to +1
            z = (features.loc[mask, col].mean() - col_meds[col]) / col_mads[col]
            score += direction * float(z)
        scores.append(score)

    # Higher composite score = more inattentive = block 0.
    order = np.argsort(scores)[::-1]
    block_map = {int(unique_blocks[old]): new for new, old in enumerate(order)}
    return np.array([block_map[int(b)] for b in blocks])


def detect_blocks_cpd(features, feature_list, config, composite_method="average",
                      participant_id=None, session_id=None):
    """
    Unified changepoint detection using configurable feature list.
    
    Stores MAD diagnostics in _CPD_DIAGNOSTICS for later reporting.
    
    Parameters
    ----------
    features : pd.DataFrame
        Feature dataframe
    feature_list : list
        List of feature display names
    config : dict
        Configuration with CP settings
    composite_method : str
        "average" or "PCA" for combining multiple features
    participant_id : str or int, optional
        Participant identifier for diagnostic logging
    session_id : str or int, optional
        Session identifier for diagnostic logging
        
    Returns
    -------
    np.ndarray
        Block assignments
    """
    global _CPD_DIAGNOSTICS 

    min_block = config["min_block_size"]
    smoothing_method = config["smoothing_method"]
    smoothing_window = config["smoothing_window"]
    
    n = len(features)
    
    # Map display names to column names
    cols = []
    display_for_col_cpd = []   # parallel display names, needed for direction lookup
    for name in feature_list:
        col = get_feature_column(name, features, config)
        if col:
            cols.append(col)
            display_for_col_cpd.append(name)

    if len(cols) == 0:
        return np.zeros(n, dtype=int)
    
    # Combine features into signal
    # No fill anywhere below: detectors receive observed stimuli only (cell 18).
    _pca_stats = None
    if len(cols) == 1:
        signal = features[cols[0]].to_numpy(dtype=float)
    else:
        if composite_method == "PCA":
            from sklearn.decomposition import PCA
            X = features[cols].copy()
            # Robust z per column; a MAD <= 1e-6 is replaced by the column SD.
            X = robust_scale(X, zero_mad="sd", sd_below=1e-6)
            # Fit PCA with all components so we get eigenvalues + variance ratios.
            n_comp_full = min(len(cols), max(1, X.shape[0] - 1))
            pca_full = PCA(n_components=n_comp_full).fit(X.values)
            _pca_eigs = pca_full.explained_variance_
            _pca_evar = pca_full.explained_variance_ratio_
            signal = pca_full.transform(X.values)[:, 0]
            # scikit-learn does not fix PC1's sign across fits,
            # so the composite signal (and therefore block ordering) could flip between
            # participants. Orient PC1 against the direction-weighted mean of the
            # standardised features so the sign is deterministic and comparable.
            _dirs = {name: spec["direction"] for name, spec in FEATURE_REGISTRY.items()}
            _oriented = np.nanmean(
                np.vstack([
                    _dirs.get(_disp, +1) * X.values[:, _j]
                    for _j, _disp in enumerate(display_for_col_cpd)
                ]),
                axis=0,
            )
            if np.dot(signal, _oriented) < 0:
                signal = -signal
            # Bartlett sphericity (manual): chi2 = -[n-1-(2p+5)/6] * ln(det(R))
            try:
                _corr = np.corrcoef(X.values, rowvar=False)
                _det = float(np.linalg.det(_corr))
                _n_obs, _p_vars = X.shape
                if _det > 0 and _n_obs > 5:
                    from scipy.stats import chi2 as _chi2
                    _bart_chi2 = -((_n_obs - 1.0 - (2.0 * _p_vars + 5.0) / 6.0)) * np.log(_det)
                    _bart_df = _p_vars * (_p_vars - 1) // 2
                    _bart_p = float(1.0 - _chi2.cdf(_bart_chi2, df=_bart_df))
                else:
                    _bart_chi2 = float("inf")
                    _bart_p = 0.0
            except Exception:
                _bart_chi2 = float("nan")
                _bart_p = float("nan")
            _pca_stats = {
                "pca_explained_var_ratio_pc1": float(_pca_evar[0]) if len(_pca_evar) else float("nan"),
                "pca_eigenvalue_pc1": float(_pca_eigs[0]) if len(_pca_eigs) else float("nan"),
                "pca_eigenvalues": tuple(float(e) for e in _pca_eigs),
                "pca_bartlett_chi2": float(_bart_chi2),
                "pca_bartlett_p": float(_bart_p),
                "pca_n_components": int(_n_comp_full := n_comp_full),
            }
        else:
            components = []
            for col in cols:
                comp = features[col].values.copy()
                if np.all(np.isnan(comp)):
                    import warnings as _w_m6
                    _w_m6.warn(
                        f"detect_blocks_cpd: feature '{col}' is all-NaN; excluded",
                        stacklevel=2,
                    )
                    continue
                comp = (comp - np.nanmean(comp)) / (np.nanstd(comp) + 1e-8)
                components.append(comp)
            if not components:
                return np.zeros(n, dtype=int)
            signal = np.nanmean(np.vstack(components), axis=0)
    
    signal = apply_smoothing(signal, smoothing_method, smoothing_window)
    
    k = float(config["cp_k"])
    mad_window_size = config["mad_window_size"]
    diffs = np.abs(np.diff(signal))
    threshold, local_med_arr, local_mad_arr = _compute_diff_thresholds(
        diffs, k, mad_window_size,
        min_periods=config["min_periods"], center=config["rolling_center"])
    diff_scores = robust_z(diffs, local_med_arr, local_mad_arr)
    median_diff = float(np.median(local_med_arr))
    mad = float(np.median(local_mad_arr))
    
    changepoints = np.where(diffs > threshold)[0] + 1
    
    if len(changepoints) > 0:
        filtered = [changepoints[0]] if changepoints[0] >= min_block else []
        last_cp = filtered[-1] if filtered else 0
        for cp in changepoints[1:]:
            if cp - last_cp >= min_block:
                filtered.append(cp)
                last_cp = cp
        if filtered and (n - filtered[-1] < min_block):
            filtered.pop()
        changepoints = np.array(filtered) if filtered else np.array([])
    else:
        changepoints = np.array([])
    
    # === STORE DIAGNOSTIC ===
    _CPD_DIAGNOSTICS.append({
        "participant": participant_id,
        "session": session_id,
        "method": "CPD",
        "n_trials": n,
        "mad": mad,
        "median_diff": median_diff,
        "mad_window_size": mad_window_size,
        "k_used": k,
        "n_changepoints": len(changepoints),
        "n_blocks": len(changepoints) + 1,
        "features_used": ",".join(feature_list),
        "diffs": diffs.copy(),
        "local_med": local_med_arr.copy(),
        "local_mad": local_mad_arr.copy(),
        "diff_scores": diff_scores.copy(),
    })
    if _pca_stats is not None:
        _CPD_DIAGNOSTICS[-1].update(_pca_stats)

    # Create block labels + reorder
    blocks = np.zeros(n, dtype=int)
    for i, cp in enumerate(changepoints):
        blocks[cp:] = i + 1
    
    # direction-aware ordering; block 0 = most inattentive.
    blocks = order_blocks_by_inattention(blocks, features, cols, display_for_col_cpd)

    return blocks


# dimension-aware PELT penalty (uni/multi)
def detect_blocks_pelt(features, feature_list, config, composite_method="average",
                       participant_id=None, session_id=None):
    """
    PELT changepoint detection via the `ruptures` library.

    Penalty-based segmentation: no manual k. Tune `pelt_pen_uni` / `pelt_pen_multi` (higher =
    fewer breakpoints). Builds the same signal as `detect_blocks_cpd`, then
    delegates segmentation to `ruptures.Pelt`.

    Diagnostic record uses the same _CPD_DIAGNOSTICS list as MAD-CPD so the
    report can show PELT alongside the other families.
    """
    global _CPD_DIAGNOSTICS

    min_block = config["min_block_size"]
    pelt_model = config["pelt_model"]
    _pen_uni = float(config["pelt_pen_uni"])
    _pen_multi = float(config["pelt_pen_multi"])
    pelt_jump = int(config["pelt_jump"])
    smoothing_method = config["smoothing_method"]
    smoothing_window = config["smoothing_window"]

    n = len(features)

    cols = []
    display_for_col = []   # parallel display names, needed for direction lookup
    for name in feature_list:
        col = get_feature_column(name, features, config)
        if col:
            cols.append(col)
            display_for_col.append(name)
    if len(cols) == 0:
        return np.zeros(n, dtype=int)

    # Build signal. Single feature -> 1D (mirrors detect_blocks_cpd). Multi-feature
    # -> 2D (n_trials, n_features) so PELT operates in the joint space rather than
    # on a 1D projection. MAD-z normalize per column so cost-function scale matches.
    # No fill: detectors receive observed stimuli only (cell 18).
    if len(cols) == 1:
        signal = features[cols[0]].to_numpy(dtype=float)
        signal = apply_smoothing(signal, smoothing_method, smoothing_window)
        fit_data = signal.reshape(-1, 1)
        block_ordering_signal = signal
    else:
        X = features[cols].copy()
        X = robust_scale(X, zero_mad="floor")
        # Smooth each column independently before fitting.
        Xv = X.values.astype(float)
        for j in range(Xv.shape[1]):
            Xv[:, j] = apply_smoothing(Xv[:, j], smoothing_method, smoothing_window)
        fit_data = Xv
        # For final block-label ordering, use the average across z-scored columns.
        block_ordering_signal = np.nanmean(Xv, axis=1)

    # Dimension-aware penalty: multivariate PELT cost sums over features, so it
    # needs a larger penalty than univariate (BIC ~ dim*log(n)). Separate knobs.
    pelt_pen = _pen_uni if fit_data.shape[1] == 1 else _pen_multi

    try:
        algo = rpt.Pelt(model=pelt_model, jump=pelt_jump, min_size=int(min_block)).fit(fit_data)
        bkps = algo.predict(pen=pelt_pen)
        changepoints = np.array([b for b in bkps[:-1] if 0 < b < n], dtype=int)
    except Exception as e:
        import warnings as _w
        _w.warn(f"detect_blocks_pelt failed: {type(e).__name__}: {e}", stacklevel=2)
        return np.zeros(n, dtype=int)

    # k_used is a CROSS-METHOD COLUMN. Different detectors store different
    # "tuning parameter" values in this slot so a single diagnostics DataFrame
    # can be sorted/filtered across families:
    #   * MAD-CPD              -> stores k  (the MAD multiplier)
    #   * HMM                  -> NaN (HMM has no scalar sensitivity knob)
    #   * PELT                 -> stores pelt_pen (penalty; higher = fewer cps)
    # Readers comparing rows across detectors should remember the unit of
    # k_used depends on the row's `method` column, not on a shared scale.
    _CPD_DIAGNOSTICS.append({
        "participant": participant_id,
        "session": session_id,
        "method": "PELT",
        "n_trials": n,
        "mad": np.nan,
        "median_diff": np.nan,
        "mad_window_size": None,
        "k_used": pelt_pen,        # see note above re: cross-method overload
        "n_changepoints": len(changepoints),
        "n_blocks": len(changepoints) + 1,
        "features_used": ",".join(feature_list),
        "pelt_model": pelt_model,
        "pelt_n_features": int(fit_data.shape[1]),
    })

    blocks = np.zeros(n, dtype=int)
    for i, cp in enumerate(changepoints):
        blocks[cp:] = i + 1

    unique_blocks = np.unique(blocks)
    if len(unique_blocks) > 1:
        # direction-aware ordering; block 0 = most inattentive.
        blocks = order_blocks_by_inattention(blocks, features, cols, display_for_col)

    return blocks

In [ ]:
# [cell 16]
# =============================================================================
# BLOCK MERGING (POST-DETECTION)
# =============================================================================

# =============================================================================
# TOSTER HELPER  (robust_tost merge method)
# =============================================================================
# Lazy rpy2 import so the notebook works even without rpy2/R installed as
# long as merge_method != "robust_tost".

def _run_wilcox_tost(x, y, equiv_bound, alpha):
    """
    Call TOSTER::wilcox_TOST via rpy2.

    Equivalence bounds are in SMD (Cohen's d) units so they are
    scale-invariant across different merged features.

    Returns
    -------
    float
        TOST p-value = max(p_lower, p_upper).  The *max* is used because
        equivalence requires BOTH one-sided tests to be significant; the
        larger p-value is the binding constraint.
        p < alpha  →  equivalence established.
    """
    from rpy2.robjects import FloatVector
    import rpy2.robjects.packages as rpackages

    TOSTER = rpackages.importr('TOSTER')

    x_r = FloatVector(x.tolist())
    y_r = FloatVector(y.tolist())

    r_result = TOSTER.wilcox_TOST(
        x_r,
        y_r,
        low_eqbound  = -equiv_bound,
        high_eqbound =  equiv_bound,
        eqbound_type = "SMD",
        paired       = False,
        alpha        = alpha,
    )

    # TOSTER::wilcox_TOST returns an R list whose $TOST element is a data
    # frame with rows = (lower-bound test, upper-bound test) and a p.value
    # column. rpy2 here uses R-style 1-indexed access on the R DataFrame:
    #   rx2('TOST')[1]   -> the p.value column
    #   col[1], col[2]   -> rows 1 and 2 of that column
    # Indices depend on TOSTER's $TOST column order; revalidate if you
    # upgrade the R package.
    tost_col  = r_result.rx2('TOST')[1]   # p.value column
    p_lower   = float(tost_col[1])        # lower one-sided p
    p_upper   = float(tost_col[2])        # upper one-sided p
    return max(p_lower, p_upper)          # binding constraint (TOST conclusion)


def compute_pair_merge_score(features, mask_a, mask_b, feature_cols, config, merge_on):
    """
    Compute merge score for a pair of adjacent blocks.

    Higher score = more appropriate to merge.

    merge_method logic
    ------------------
    "ttest"      : merge when blocks are NOT significantly different (Welch t-test).
                   Qualifies when n_significant < merge_on  (fewer diffs → merge).
    "wilcoxon"   : same logic as ttest but uses Mann-Whitney U (rank-based, no
                   normality assumption). Better for bounded/skewed features like
                   Entropy. Qualifies when n_significant < merge_on.
    "robust_tost": merge when blocks ARE statistically equivalent via Wilcoxon
                   TOST (TOSTER R package, bounds in SMD units).
                   Qualifies when n_equivalent >= merge_on  (more equiv → merge).

    Returns
    -------
    score : float
        Merge score (higher = better candidate for merging)
    qualifies : bool
        Whether this pair meets the config merge criteria
    details : dict
        Per-feature statistics
    """
    merge_method  = config["merge_method"]
    merge_alpha   = config["merge_alpha"]
    equiv_bound   = config["merge_equivalence_bound"]

    p_values        = []
    equiv_scores    = []
    tested_features = []

    for col in feature_cols:
        values_a = features.loc[mask_a, col].dropna().values
        values_b = features.loc[mask_b, col].dropna().values

        if len(values_a) < 2 or len(values_b) < 2:
            continue

        tested_features.append(col)

        if merge_method == "ttest":
            # Welch t-test — higher p = less different = better for merging
            try:
                _, p_val = stats.ttest_ind(values_a, values_b, equal_var=False)
                p_values.append(float(p_val))
                equiv_scores.append(float(p_val))
            except Exception as _e_h6a:
                # surface the failure rather than swallowing silently.
                import warnings as _w_h6a
                _w_h6a.warn(
                    f"compute_pair_merge_score (ttest): {type(_e_h6a).__name__}: {_e_h6a}; using p=1.0",
                    stacklevel=2,
                )
                p_values.append(1.0)
                equiv_scores.append(1.0)

        elif merge_method == "wilcoxon":
            # Mann-Whitney U (Wilcoxon rank-sum) — non-parametric alternative to ttest.
            # Same merge logic: higher p = less different = better for merging.
            # Recommended for bounded/skewed features (e.g. Entropy, Rating Variance).
            try:
                _, p_val = stats.mannwhitneyu(values_a, values_b, alternative="two-sided")
                p_values.append(float(p_val))
                equiv_scores.append(float(p_val))
            except Exception as _e_h6b:
                # surface the failure rather than swallowing silently.
                import warnings as _w_h6b
                _w_h6b.warn(
                    f"compute_pair_merge_score (wilcoxon): {type(_e_h6b).__name__}: {_e_h6b}; using p=1.0",
                    stacklevel=2,
                )
                p_values.append(1.0)
                equiv_scores.append(1.0)

        elif merge_method == "robust_tost":
            # Wilcoxon TOST via TOSTER (R).  Bounds in SMD (Cohen's d) units.
            # p_tost = max(p_lower, p_upper) — binding constraint.
            # p_tost < alpha  →  equivalence established.
            try:
                p_tost = _run_wilcox_tost(values_a, values_b, equiv_bound, merge_alpha)
                p_values.append(float(p_tost))
                equiv_scores.append(1.0 - float(p_tost))   # higher = more equivalent
            except Exception as e:
                # Conservative: treat as NOT equivalent
                p_values.append(1.0)
                equiv_scores.append(0.0)

    if len(p_values) == 0:
        return -np.inf, False, {}

    merge_fdr = config['merge_fdr']

    # Apply FDR correction (or not) to the per-feature p-values
    if merge_fdr is None:
        # No correction — recommended for robust_tost
        p_fdr = p_values
    else:
        # 'per_pair' or 'global' (global treated as per_pair in greedy context)
        _, p_fdr = fdrcorrection(p_values, alpha=merge_alpha)
        p_fdr = list(p_fdr)

    if merge_method in ("ttest", "wilcoxon"):
        # Merge when few features are significantly DIFFERENT
        n_significant = int(sum(p < merge_alpha for p in p_fdr))
        n_equivalent  = None
        qualifies     = n_significant < merge_on
        score         = float(np.mean(p_fdr))           # higher p = more similar

    else:  # robust_tost
        # Merge when enough features show equivalence (p_tost < alpha)
        n_equivalent  = int(sum(p < merge_alpha for p in p_fdr))
        n_significant = None
        qualifies     = n_equivalent >= merge_on
        score         = float(np.mean(equiv_scores))    # higher score = more equivalent

    details = {
        "p_values":        p_values,
        "p_fdr":           list(p_fdr),
        "equiv_scores":    equiv_scores,
        "features_tested": tested_features,
        "n_significant":   n_significant,
        "n_equivalent":    n_equivalent,
    }

    return score, qualifies, details


def merge_nonsignificant_blocks(features, blocks, config, merge_features):
    """
    Merge adjacent blocks using greedy best-pair-first strategy.
    
    At each iteration, finds the most equivalent (or least different) adjacent
    pair that qualifies for merging, and merges that pair. Repeats until no
    more pairs qualify.
    
    Parameters
    ----------
    features : pd.DataFrame
        Trial-level features
    blocks : np.ndarray
        Initial block assignments (0-indexed)
    config : dict
        Configuration with:
        - merge_blocks: Enable/disable merging
        - merge_method: "ttest" | "wilcoxon" | "robust_tost"
        - merge_alpha: Significance threshold
        - merge_on: ttest       — max n_significant to ALLOW merge (fewer diffs → merge)
                    robust_tost — min n_equivalent to ALLOW merge (more equiv → merge)
        - merge_equivalence_bound: TOST bound in Cohen's d — used only with robust_tost
        - merge_fdr: FDR correction of the per-feature p-values within a pair
                     (None | "per_pair"; "global" is treated as "per_pair")
    
    Returns
    -------
    np.ndarray
        Merged block assignments
    """
    if not config["merge_blocks"]:
        return blocks

    merge_features = list(merge_features)
    merge_method = config["merge_method"]

    if not merge_features:
        return blocks
    
    # merge_on applies to MULTIVARIATE methods only. A univariate detector merges
    # when its single feature is non-significant, i.e. merge_on == 1 by construction.
    if len(merge_features) == 1:
        merge_on = 1
    else:
        merge_on = int(config["merge_on"])
        if merge_on < 1:
            raise ValueError(f"merge_on must be >= 1, got {merge_on}")
        if merge_on > len(merge_features):
            raise ValueError(
                f"merge_on ({merge_on}) cannot exceed the number of detection features "
                f"for this multivariate method ({len(merge_features)}: {merge_features})."
            )
    
    # Get validation map to translate feature names to columns
    validation_map = get_validation_map(config)
    
    # Get actual column names for merge features
    feature_cols = []
    for feat_name in merge_features:
        if feat_name in validation_map:
            col_name = validation_map[feat_name][0]
            if col_name in features.columns:
                feature_cols.append(col_name)
    
    if not feature_cols:
        print(f"[Merge] WARNING: No merge feature columns found in features df. "
              f"Requested: {merge_features}. Skipping merge.")
        return blocks

    print(f"[Merge] method={merge_method}, merge_on={merge_on}, alpha={config['merge_alpha']}, "
          f"features={feature_cols}")

    blocks = np.array(blocks).copy()

    # pre-merge contiguity normalization. CPD upstream may have
    # reordered block labels by signal magnitude, leaving the same label in
    # multiple separated temporal regions. Relabel each contiguous run with a
    # fresh ID so that any subsequent label-based merge fuses exactly the
    # two adjacent regions found in spatial_pairs (no aliasing across gaps).
    blocks = np.asarray(blocks).copy()
    if len(blocks) > 0:
        _new_blocks = np.zeros(len(blocks), dtype=int)
        _curr = 0
        _new_blocks[0] = 0
        for _i in range(1, len(blocks)):
            if blocks[_i] != blocks[_i - 1]:
                _curr += 1
            _new_blocks[_i] = _curr
        if not np.array_equal(_new_blocks, blocks):
            print(
                f"  [Merge pre-check] Relabelled {int(np.unique(blocks).size)} "
                f"original blocks → {_curr + 1} contiguous regions before merging."
            )
        blocks = _new_blocks
    
    # Greedy best-pair-first merging
    max_iterations = 100
    
    for iteration in range(max_iterations):
        unique_blocks = sorted(np.unique(blocks))
        
        if len(unique_blocks) < 2:
            break
        
        # Build spatially-adjacent pairs from actual block sequence
        # (block IDs may be reordered by signal level, so consecutive IDs
        #  are NOT necessarily spatially adjacent)
        spatial_pairs = []
        for i in range(len(blocks) - 1):
            if blocks[i] != blocks[i + 1]:
                pair = (blocks[i], blocks[i + 1])
                if pair not in spatial_pairs:
                    spatial_pairs.append(pair)
        
        # Evaluate ALL spatially adjacent pairs
        pair_scores = []
        
        for block_a, block_b in spatial_pairs:
            
            mask_a = blocks == block_a
            mask_b = blocks == block_b
            
            if mask_a.sum() < 3 or mask_b.sum() < 3:
                continue
            
            score, qualifies, details = compute_pair_merge_score(
                features, mask_a, mask_b, feature_cols, config, merge_on
            )
            
            if qualifies:
                pair_scores.append({
                    "block_a": block_a,
                    "block_b": block_b,
                    "score": score,
                    "details": details,
                })
        
        if not pair_scores:
            # No pairs qualify for merging
            break
        
        # Find the best pair (highest score = most equivalent / least different)
        best_pair = max(pair_scores, key=lambda x: x["score"])
        
        # Merge best pair (right into left)
        n_qual = len(pair_scores)
        print(f"  [Merge iter {iteration+1}] block {best_pair['block_a']}+{best_pair['block_b']} "
              f"(score={best_pair['score']:.3f}, {n_qual} qualifying pairs)")
        blocks[blocks == best_pair["block_b"]] = best_pair["block_a"]
    
    # Renumber blocks sequentially
    unique_blocks = sorted(np.unique(blocks))
    block_map = {old: new for new, old in enumerate(unique_blocks)}
    blocks = np.array([block_map[b] for b in blocks])
    
    return blocks

In [ ]:
# [cell 17]
# =============================================================================
# VALIDATION FUNCTIONS
# =============================================================================

# Kruskal/ANOVA + eta2_H/omega2 branch (validate_test)
def validate_anova(values, blocks, config=None):
    """One-way test for block differences on a validation feature.
    config["validate_test"]:
      "kruskal" (default) -> Kruskal-Wallis (nonparametric, rank-based; robust to
                             the skewed / bounded / heavy-tailed validation
                             features) with the eta^2_H effect size
                             eta2_H = max(0, (H - k + 1) / (n - k)).
      "anova"             -> parametric one-way ANOVA + omega-squared.
    Returns (stat, p_value, effect_size)."""
    test = config["validate_test"]
    valid = ~np.isnan(values)
    unique_blocks = np.unique(blocks[valid])
    if len(unique_blocks) < 2:
        return np.nan, 1.0, 0.0
    groups = [values[valid & (blocks == b)] for b in unique_blocks]
    groups = [g for g in groups if len(g) >= 3]
    if len(groups) < 2:
        return np.nan, 1.0, 0.0
    n_total = sum(len(g) for g in groups)
    k = len(groups)
    try:
        if test == "kruskal":
            # KW is undefined if every value is identical across all groups.
            if np.unique(np.concatenate(groups)).size < 2:
                return np.nan, 1.0, 0.0
            h_stat, p_val = stats.kruskal(*groups)
            if n_total <= k:
                return h_stat, p_val, np.nan
            # eta^2_H (Cohen, 2008): the KW analog of omega^2, with the same
            # (k-1)/(n-k)-style bias correction, so spurious extra blocks are still
            # penalized. Range [0, 1]; clipped at 0.
            eta2_h = max(0.0, (h_stat - k + 1) / (n_total - k))
            return h_stat, p_val, eta2_h
        else:  # parametric ANOVA
            f_stat, p_val = stats.f_oneway(*groups)
            if n_total <= k:
                return f_stat, p_val, np.nan
            grand_mean = np.concatenate(groups).mean()
            ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
            ss_within = sum(((g - g.mean())**2).sum() for g in groups)
            ms_within = ss_within / (n_total - k)
            omega_sq = max(0, (ss_between - (k-1) * ms_within) / (ss_between + ss_within + ms_within))
            return f_stat, p_val, omega_sq
    except Exception:
        return np.nan, 1.0, 0.0


def validate_levene(values, blocks):
    """Levene test for variance differences."""
    valid = ~np.isnan(values)
    unique_blocks = np.unique(blocks[valid])
    
    if len(unique_blocks) < 2:
        return np.nan, 1.0
    
    groups = [values[valid & (blocks == b)] for b in unique_blocks]
    groups = [g for g in groups if len(g) >= 3]
    
    if len(groups) < 2:
        return np.nan, 1.0
    
    try:
        stat, p_val = stats.levene(*groups)
        return stat, p_val
    except Exception:
        return np.nan, 1.0


def get_validation_map(config):
    """
    Get validation feature map based on use_residuals setting.

    Returns dict mapping display_name -> (column_name, validation_prefix).
    Consumers (merge_nonsignificant_blocks, compute_block_quality_metrics)
    index the column at position 0.

    All fields derive from FEATURE_REGISTRY -- this function
    is a thin view over the registry, parameterised by use_residuals.
    Consumers that need the column actually present (residual, else RT) should call
    get_feature_column directly instead of this map's column field.
    """
    use_residuals = config["use_residuals"]
    col_field = "residual_col" if use_residuals else "rt_col"
    return {
        name: (spec[col_field], spec["validation_prefix"])
        for name, spec in FEATURE_REGISTRY.items()
    }


def compute_block_quality_metrics(features, blocks, method_info, config):
    """
    Compute validation metrics for block assignments.
    
    Uses use_residuals config to select appropriate features:
    - use_residuals=True: residual columns (aggregated_residual, local_residual_var)
    - use_residuals=False: raw-RT columns (aggregated_rt, local_rt_var)
    """
    defined_on = method_info["defined_on"]
    blocks = np.asarray(blocks)
    unique_blocks = np.unique(blocks)
    n_blocks = len(unique_blocks)
    
    metrics = {"n_blocks": n_blocks}
    
    # Store block sizes for CV calculation
    block_sizes = [int(np.sum(blocks == b)) for b in unique_blocks]
    metrics["block_sizes"] = block_sizes
    
    # Compute CV
    if len(block_sizes) > 1:
        sizes = np.array(block_sizes)
        metrics["block_cv"] = sizes.std() / sizes.mean() if sizes.mean() > 0 else 0.0
    else:
        metrics["block_cv"] = 0.0
    
    if n_blocks < 2:
        metrics["block_size_mean"] = len(blocks)
        metrics["block_size_std"] = 0.0
        metrics["n_transitions"] = 0
        metrics["fragmentation_index"] = 1.0
        metrics["n_external_features_used"] = 0
        return metrics
    
    # Get validation settings from config
    val_settings = config["validation_features"]
    use_residuals = config["use_residuals"]
    
    # Get the appropriate validation map based on use_residuals
    validation_map = get_validation_map(config)
    
    n_external_features_used = 0
    
    for display_name, (_, prefix) in validation_map.items():
        # Only features flagged True in config["validation_features"] are validated.
        if not val_settings.get(display_name, False):
            continue  # Skip this feature entirely

        # Resolve the actual column (the RT column when the residual column is missing). Single source of truth:
        # cell 15 get_feature_column + FEATURE_REGISTRY; validation_map's column field is not used here, so the
        # resolution lives in one place.
        col = get_feature_column(display_name, features, config)
        if col is None:
            continue  # Feature not available in data

        # Perform validation
        f, p, omega = validate_anova(features[col].values, blocks, config)
        metrics[f"{prefix}_F"] = f
        metrics[f"{prefix}_p"] = p
        metrics[f"{prefix}_effsize"] = omega
        n_external_features_used += 1
    
    # Diagnostic-only: Holm-Bonferroni correction across the QC features
    # tested *within this row* (one block partition). Stored as
    # `{prefix}_p_holm_within` (Holm controls FWER, NOT FDR). NOT consumed by
    # method ranking, which reads the raw `{prefix}_p`. Use only when printing
    # per-row diagnostics where the family-wise error rate across the
    # simultaneous feature tests matters.
    _within_prefixes = []
    _within_pvals = []
    for _disp, (_col, _pref) in validation_map.items():
        _key = f"{_pref}_p"
        if _key in metrics and pd.notna(metrics[_key]):
            _within_prefixes.append(_pref)
            _within_pvals.append(float(metrics[_key]))
    if len(_within_pvals) >= 2:
        try:
            from statsmodels.stats.multitest import multipletests
            _, _p_holm, _, _ = multipletests(_within_pvals, alpha=0.05, method="holm")
            for _pref, _ph in zip(_within_prefixes, _p_holm):
                metrics[f"{_pref}_p_holm_within"] = float(_ph)
        except Exception:
            for _pref, _p in zip(_within_prefixes, _within_pvals):
                metrics[f"{_pref}_p_holm_within"] = float(_p)
    elif len(_within_pvals) == 1:
        metrics[f"{_within_prefixes[0]}_p_holm_within"] = float(_within_pvals[0])
    
    # Store count of features actually used for validation
    metrics["n_external_features_used"] = n_external_features_used
    
    # Levene test - use appropriate level feature
    level_col = "aggregated_residual" if use_residuals else "aggregated_rt"
    if level_col not in features.columns:
        level_col = "aggregated_rt"  # no residual level column: the RT level
    
    if defined_on not in ["variance"] and level_col in features.columns:
        stat, p = validate_levene(features[level_col].values, blocks)
        metrics["levene_stat"] = stat
        metrics["levene_p"] = p
    
    # Block structure
    metrics["block_size_mean"] = np.mean(block_sizes)
    metrics["block_size_std"] = np.std(block_sizes)
    
    transitions = int(np.sum(np.diff(blocks) != 0))
    metrics["n_transitions"] = transitions
    metrics["fragmentation_index"] = (transitions + 1) / n_blocks if n_blocks > 0 else np.nan
    
    return metrics

In [ ]:
# [cell 18]
# =============================================================================
# METHOD REGISTRY
# =============================================================================

def _labels_to_all_rows(labels_obs, observed):
    """Map labels fitted on the observed rows back onto every row.

    An unobserved row (rolling-window edge, or a stimulus dropped by an RT threshold
    under rt_threshold_na_handling="drop_stimulus") takes the
    label of the nearest PRECEDING observed row; rows before the first observed row
    take the first observed label. So an unobserved run inside a block stays in that
    block, a run sitting exactly at a boundary joins the block before it, leading /
    trailing runs join the first / last block, and an unobserved row can never start
    a block of its own -- missing values neither create nor move a changepoint."""
    observed = np.asarray(observed, dtype=bool)
    labels_obs = np.asarray(labels_obs)
    idx = np.flatnonzero(observed)
    prev = np.searchsorted(idx, np.arange(len(observed)), side="right") - 1
    return labels_obs[np.clip(prev, 0, len(idx) - 1)]


def _on_observed_rows(detect, feature_names, config):
    """Run `detect` only on stimuli where every feature it segments on is observed.

    Nothing is imputed: NaN rows are removed before detection and get their block from
    _labels_to_all_rows afterwards. Methods with no features (Fixed_*) are unchanged."""
    def run(f, pid=None, sid=None):
        cols = [c for c in (get_feature_column(nm, f, config) for nm in feature_names) if c]
        if not cols:
            return detect(f, pid=pid, sid=sid)
        observed = f[cols].notna().all(axis=1).to_numpy()
        if observed.all():
            return detect(f, pid=pid, sid=sid)
        if not observed.any():
            return np.zeros(len(f), dtype=int)
        labels = detect(f.loc[observed].reset_index(drop=True), pid=pid, sid=sid)
        return _labels_to_all_rows(labels, observed)
    return run


def create_block_methods(config):
    """
    Create block detection method registry from config.
    
    Creates individual methods for EACH feature in the feature lists,
    plus composite methods that combine all features.
    
    Method Naming Convention
    ------------------------
    Individual feature methods:
    - HMM_[N]_[Feature]: HMM on single feature (N states)
    - CPD_[Feature]: MAD changepoint on single feature
    - PELT_[Feature]: PELT on single feature

    Composite methods:
    - HMM_Attention_comp_[N]: HMM on all hmm_composite features
    - CPD_Attention_comp: CPD on all cp_composite features
    - PELT_Attention_comp: PELT on all pelt_composite features

    Fixed_[N] (fixed_block_sizes) are registered here; the matched controls
    Fixed_[Method]_based are built after detection (control_block_count).
    """
    methods = {}
    # use_residuals isn't read in this function; downstream detection
    # functions resolve column choice themselves via get_feature_column.
    cp_comp_method = config["cp_comp_method"]
    
    # Get feature lists from config
    hmm_features = config["hmm_features"]
    hmm_composite = config["hmm_composite"]
    cp_features = config["cp_features"]
    cp_composite = config["cp_composite"]
    
    # Helper to create clean method name from feature display name
    def feature_to_name(feature):
        """Convert display name to method-safe name."""
        return feature.replace(" ", "_").replace("(", "").replace(")", "")
    
    # Helper to determine defined_on. Single source of truth: cell 15's
    # FEATURE_REGISTRY[feature]["defined_on"]. Unknown features fall through
    # to "mixed" so a misnamed composite still registers without crashing.
    def get_defined_on_single(feature):
        return FEATURE_REGISTRY.get(feature, {}).get("defined_on", "mixed")
    
    # Fixed blocks (skip if None)
    if config["fixed_block_sizes"] is not None:
        for size in config["fixed_block_sizes"]:
            methods[f"Fixed_{size}"] = {
                "func": lambda f, pid=None, sid=None, s=size: detect_blocks_fixed(len(f), s),
                "defined_on": "time",
                "features": [],
                "description": f"Fixed {size}-trial blocks",
            }
    
    # Method-family filter: only register families listed in cpd_method_types
    method_types = config["cpd_method_types"]
    method_types_set = {t.upper() for t in method_types}

    # ==========================================================================
    # HMM METHODS - One per feature in hmm_features
    # ==========================================================================
    if "HMM" in method_types_set and config["hmm_n_states"] is not None:
        for feature in hmm_features:
            feat_name = feature_to_name(feature)
            defined_on = get_defined_on_single(feature)
            
            for n in config["hmm_n_states"]:
                method_name = f"HMM_{n}_{feat_name}"
                methods[method_name] = {
                    "func": _on_observed_rows(
                        lambda f, pid=None, sid=None, n=n, fl=[feature], c=config:
                            detect_blocks_hmm(f, n, fl, c, participant_id=pid, session_id=sid),
                        [feature], config),
                    "defined_on": defined_on,
                    "features": [feature],
                    "description": f"HMM on {feature} ({n} states)",
                }
        
        # HMM Composite method (all hmm_composite features combined)
        if hmm_composite:
            for n in config["hmm_n_states"]:
                methods[f"HMM_Attention_comp_{n}"] = {
                    "func": _on_observed_rows(
                        lambda f, pid=None, sid=None, n=n, fl=hmm_composite, c=config:
                            detect_blocks_hmm(f, n, fl, c, participant_id=pid, session_id=sid),
                        hmm_composite, config),
                    "defined_on": "attention_composite",
                    "features": list(hmm_composite),
                    "description": f"HMM composite ({n} states)",
                }
    
    # ==========================================================================
    # MAD-CPD METHODS - One per feature in cp_features
    # ==========================================================================
    if "MAD" in method_types_set and config["cp_k"] is not None:
        for feature in cp_features:
            feat_name = feature_to_name(feature)
            defined_on = get_defined_on_single(feature)
            
            method_name = f"CPD_{feat_name}"
            methods[method_name] = {
                "func": _on_observed_rows(
                    lambda f, pid=None, sid=None, fl=[feature], c=config:
                        detect_blocks_cpd(f, fl, c, composite_method="average",
                                           participant_id=pid, session_id=sid),
                    [feature], config),
                "defined_on": defined_on,
                "features": [feature],
                "description": f"CPD on {feature}",
            }
        
        # CPD Composite method (all cp_composite features combined)
        if cp_composite:
            methods["CPD_Attention_comp"] = {
                "func": _on_observed_rows(
                    lambda f, pid=None, sid=None, fl=cp_composite, c=config:
                        detect_blocks_cpd(f, fl, c, composite_method=cp_comp_method,
                                           participant_id=pid, session_id=sid),
                    cp_composite, config),
                "defined_on": "attention_composite",
                "features": list(cp_composite),
                "description": f"CPD composite via {cp_comp_method}",
            }
    
    # ==========================================================================
    # PELT METHODS - One per feature in pelt_features
    # ==========================================================================
    if "PELT" in method_types_set:
        pelt_features = config["pelt_features"]
        pelt_composite = config["pelt_composite"]
        _pen_uni_str = config["pelt_pen_uni"]
        _pen_multi_str = config["pelt_pen_multi"]
        pelt_cost_str = config["pelt_model"]

        for feature in pelt_features:
            feat_name = feature_to_name(feature)
            defined_on = get_defined_on_single(feature)
            method_name = f"PELT_{feat_name}"
            methods[method_name] = {
                "func": _on_observed_rows(
                    lambda f, pid=None, sid=None, fl=[feature], c=config:
                        detect_blocks_pelt(f, fl, c, composite_method="average",
                                            participant_id=pid, session_id=sid),
                    [feature], config),
                "defined_on": defined_on,
                "features": [feature],
                "description": f"PELT ({pelt_cost_str}, pen={_pen_uni_str}) on {feature}",
            }

        if pelt_composite:
            methods["PELT_Attention_comp"] = {
                "func": _on_observed_rows(
                    lambda f, pid=None, sid=None, fl=pelt_composite, c=config:
                        detect_blocks_pelt(f, fl, c, composite_method=cp_comp_method,
                                            participant_id=pid, session_id=sid),
                    pelt_composite, config),
                "defined_on": "attention_composite",
                "features": list(pelt_composite),
                "description": f"PELT ({pelt_cost_str}, pen={_pen_multi_str}) composite via {cp_comp_method}",
            }

    return methods

In [ ]:
# [cell 19]
# =============================================================================
# HELPER: MERGE BLOCK LABELS BACK TO LONG FORMAT
# =============================================================================
# This avoids the need to melt features_df back to long in Level 2

def merge_block_labels_to_long(long_df, features_df, block_cols=None):
    """
    Merge block labels from features_df (wide) back to long_df.
    
    This is more efficient than melting features_df back to long format
    because we only transfer the block assignment columns.
    
    Parameters
    ----------
    long_df : pd.DataFrame
        Original long-format data with id, session, stim_order, scale columns
    features_df : pd.DataFrame
        Wide-format features with block_{method} columns
    block_cols : list, optional
        Specific block columns to merge. If None, merges all block_* columns.
    
    Returns
    -------
    pd.DataFrame
        long_df with block labels added
    """
    if features_df is None or len(features_df) == 0:
        return long_df
    
    long_df = long_df.copy()
    
    # Identify block columns
    if block_cols is None:
        block_cols = [c for c in features_df.columns if c.startswith('block_')]
    
    if not block_cols:
        print("  No block columns found in features_df")
        return long_df
    
    # Select merge keys and block columns from features_df
    merge_keys = ['id', 'session', 'stim_order']
    cols_to_merge = merge_keys + block_cols
    cols_to_merge = [c for c in cols_to_merge if c in features_df.columns]
    
    block_lookup = features_df[cols_to_merge].copy()
    
    # Ensure consistent dtypes for merge
    for key in merge_keys:
        if key in long_df.columns and key in block_lookup.columns:
            long_df[key] = long_df[key].astype(str)
            block_lookup[key] = block_lookup[key].astype(str)
    
    # Drop existing block columns to avoid duplicates
    existing_block_cols = [c for c in block_cols if c in long_df.columns]
    if existing_block_cols:
        long_df = long_df.drop(columns=existing_block_cols)
    
    # Merge block labels to long format
    long_with_blocks = long_df.merge(
        block_lookup,
        on=merge_keys,
        how='left'
    )
    
    n_matched = long_with_blocks[block_cols[0]].notna().sum() if block_cols else 0
    print(f"  Merged {len(block_cols)} block columns to long_df ({n_matched}/{len(long_with_blocks)} rows matched)")
    
    return long_with_blocks


# =============================================================================
# RT-THRESHOLD MISSINGNESS (Level-1 only): rt_threshold_na_handling
# =============================================================================
# A scale-level rating is REJECTED when its RT actually violates min_rt_thresh or
# max_rt_thresh (a threshold of 0 is off, as in apply_rt_filters). The knob decides how
# that rejection reaches the Level-1 RT features; see LVL_1_CONFIG (cell 9):
#   partial        the rejected scale's RT values are removed; the stimulus averages the
#                  remaining scales, and that scale's rolling variance there is NaN.
#   drop_stimulus  the whole stimulus is invalid: all of its per-scale inputs are removed
#                  before the features are built (so it enters no neighbour's window),
#                  and every derived feature at that stimulus is NaN.
#   bfill / ffill  the rejected scale's RT values are filled from the next / previous
#                  valid value of the same scale in the same session (fill_rejected).
#                  That scale's rolling variance at the stimulus is its window's variance
#                  of the remaining valid values; only when fewer than min_periods
#                  remain is it filled the same way (never in a rolling-window edge).
# In every mode the rejected RT enters no rolling window. Only rejected cells are ever
# written: window edges, practice stimuli and naturally missing values are untouched.
# Entropy keeps the row-level filter (apply_rt_filters already NaNs the rejected
# rating's response) except under drop_stimulus, which removes the whole stimulus.
# The per-scale pass-through columns ({scale}_response, {scale}_rt_input, ...) keep
# the row-level filter only in every mode, because LVL_2 reads them from the handoff.
_STIM_INPUT_SUFFIXES = ("_rt", "_response", "_predicted", "_residual", "_residual_abs", "_rt_input")
# The per-scale RT observations the Level-1 RT features are built from. {scale}_predicted
# is the drift model's prediction (not an observation, and unused by the features): it
# keeps the row-level filter only.
_RT_OBS_SUFFIXES = ("_rt", "_residual", "_residual_abs")
# Derived columns that are NOT per-stimulus measurements and stay as computed.
_KEEP_DERIVED = {"trial_global", "trial_in_session", "session_indicator",
                 "global_rt_var", "global_residual_var"}
_STIM_KEYS = ["id", "session", "stim_order"]


def rt_threshold_rejections(long_rows, config):
    """Scale-level ratings rejected by min_rt_thresh / max_rt_thresh: a boolean frame
    indexed by (id, session, stim_order), one column per scale. Read from
    rt_sec_prefilter, the untouched copy cell 20 keeps before apply_rt_filters NaNs
    rt_sec. A missing RT is not a rejection; nor is a practice stimulus (pr_stim_drop
    removes it, not an RT threshold)."""
    if "rt_sec_prefilter" not in long_rows.columns:
        raise SystemExit("rt_threshold_rejections: rt_sec_prefilter missing; "
                         "cell 20 must snapshot rt_sec before apply_rt_filters.")
    rt = pd.to_numeric(long_rows["rt_sec_prefilter"], errors="coerce")
    rej = pd.Series(False, index=long_rows.index)
    if config["max_rt_thresh"] > 0:
        rej |= rt > config["max_rt_thresh"]
    if config["min_rt_thresh"] > 0:
        rej |= rt < config["min_rt_thresh"]
    if config["pr_stim_drop"] > 0:
        rej &= long_rows["stim_order"] > config["pr_stim_drop"]
    scales = list(config["scale_ranges"].keys())
    return (long_rows[_STIM_KEYS + ["scale"]].assign(_rej=rej.to_numpy())
            .groupby(_STIM_KEYS + ["scale"])["_rej"].any()
            .unstack("scale", fill_value=False)
            .reindex(columns=scales, fill_value=False).astype(bool))


def level1_features(stim_lvl_df, rejected, config):
    """compute_stim_resp_features with the rt_threshold_na_handling rule applied.
    `rejected` is rt_threshold_rejections() for the same rows."""
    mode = rt_threshold_na_mode(config)
    scales = list(config["scale_ranges"].keys())
    rej = pd.DataFrame(
        rejected.reindex(pd.MultiIndex.from_frame(stim_lvl_df[_STIM_KEYS]), fill_value=False)
                .reindex(columns=scales, fill_value=False).to_numpy(dtype=bool),
        index=stim_lvl_df.index, columns=scales)
    n_rej = pd.Series(rej.sum(axis=1).to_numpy(), index=pd.MultiIndex.from_frame(stim_lvl_df[_STIM_KEYS]))

    if mode == "drop_stimulus":
        contaminated = set(stim_lvl_df.loc[rej.any(axis=1), "stim_order"].tolist())
        inputs = [f"{s}{x}" for s in scales for x in _STIM_INPUT_SUFFIXES
                  if f"{s}{x}" in stim_lvl_df.columns]
        hit = stim_lvl_df["stim_order"].isin(contaminated).to_numpy()
        masked = stim_lvl_df.copy()
        masked.loc[hit, inputs] = np.nan
        features = compute_stim_resp_features(masked, config)
        contam = features["stim_order"].isin(contaminated).to_numpy()
        if contam.any():
            # restore the row-filtered per-scale values handed to LVL_2
            raw = stim_lvl_df.set_index("stim_order")[inputs]
            features.loc[contam, inputs] = raw.loc[features.loc[contam, "stim_order"]].to_numpy()
            derived = [c for c in features.columns
                       if c not in stim_lvl_df.columns and c not in _KEEP_DERIVED]
            features.loc[contam, derived] = np.nan
    else:
        order = stim_lvl_df.sort_values(_STIM_KEYS, kind="stable").index
        base, rej = stim_lvl_df.loc[order].copy(), rej.loc[order]
        groups = [base[k].to_numpy() for k in ("id", "session")]
        obs = [(s, f"{s}{x}") for s in scales for x in _RT_OBS_SUFFIXES if f"{s}{x}" in base.columns]
        for s, col in obs:
            base.loc[rej[s], col] = np.nan
            if mode != "partial":
                base[col] = fill_rejected(base[col], rej[s], mode, groups=groups).to_numpy()
        features = compute_stim_resp_features(base, config, rt_rejected=rej,
                                              rt_rejected_fill=None if mode == "partial" else mode)
        # restore the row-filtered per-scale values handed to LVL_2 (as drop_stimulus does)
        cols = [c for _, c in obs]
        features[cols] = (stim_lvl_df.set_index(_STIM_KEYS)[cols]
                          .reindex(pd.MultiIndex.from_frame(features[_STIM_KEYS])).to_numpy())
        contam = np.zeros(len(features), dtype=bool)
    features["stim_rt_contaminated"] = contam      # True only where drop_stimulus removed the stimulus
    features["n_rt_rejected_scales"] = n_rej.reindex(
        pd.MultiIndex.from_frame(features[_STIM_KEYS])).to_numpy()
    return features


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def run_block_detection(long_df, config, methods=None):
    """
    Run comprehensive block detection on long-format data.
    """
    import warnings

    min_block_size = config["min_block_size"]

    # Create method registry
    block_methods = create_block_methods(config)

    if methods is None:
        methods = list(block_methods.keys())

    # Block-merging applies to the MAD-CPD family only; every other detector keeps
    # its raw detected blocks. Fixed_* baselines are never merged.
    MERGE_PREFIX = "CPD_"
    if config["merge_blocks"]:
        print(f"Merging applies to the MAD family only (prefix: {MERGE_PREFIX})")

    print("=" * 100)
    print("BLOCK DETECTION PIPELINE")
    print("=" * 100)
    print(f"min_block_size: {min_block_size}")
    print(f"Methods: {len(methods)}")

    participants = long_df['id'].unique()
    sessions = sorted(long_df['session'].unique())

    print(f"Participants: {len(participants)}")
    print(f"Sessions: {sessions}")
    print("-" * 100)

    def process_session(pid, sess):
        """Process one participant-session combination. When run under joblib's
        loky backend, each worker has its own _CPD_DIAGNOSTICS module-global;
        we reset it at the start and return the accumulated entries so the
        driver can merge them back into the parent process's _CPD_DIAGNOSTICS.

        HMM fit FAILURES are surfaced via warnings.warn from inside
        detect_blocks_hmm (routed to suppressed_warnings.log by
        logging.captureWarnings in the imports cell); non-convergence at the
        iteration cap is printed as "[HMM non-convergence]". No driver-side
        aggregation here."""
        global _CPD_DIAGNOSTICS
        _CPD_DIAGNOSTICS = []

        mask = (long_df['id'] == pid) & (long_df['session'] == sess)
        participant_data = long_df[mask].copy()

        trial_col = "resp_order" if "resp_order" in participant_data.columns else "stim_order"
        sort_cols = ["id", "session", trial_col]
        if "scale" in participant_data.columns:
            sort_cols.append("scale")

        participant_data = participant_data.sort_values(sort_cols).reset_index(drop=True)

        if len(participant_data) < min_block_size * 2:
            return None, None, []

        # Suppress warnings during feature computation
        rejected = rt_threshold_rejections(participant_data, config)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            stim_lvl_df = long_to_stim_level(participant_data, config)
            features = level1_features(stim_lvl_df, rejected, config)

        session_results = []

        # Within-participant MAD centering of segmentation features. Detection runs
        # on the centered copy; validation, quality metrics, merging and block
        # matrices all use the ORIGINAL `features`.
        features_seg = mad_center_columns(features)

        for method_name in methods:
            if method_name not in block_methods:
                continue

            method_info = block_methods[method_name]

            try:
                _n_diag = len(_CPD_DIAGNOSTICS)
                raw_blocks = method_info['func'](features_seg, pid=pid, sid=sess)
                _failed = any(r.get("detector_failed", False) for r in _CPD_DIAGNOSTICS[_n_diag:])
                blocks = enforce_min_block_length(raw_blocks, config)
                # Attach post-min-size count to the most recent diagnostic row
                # for this (pid, sess, method) so HMM is comparable to CPD/PELT.
                try:
                    if _CPD_DIAGNOSTICS:
                        n_post_min = 1 + int(np.sum(np.diff(blocks) != 0)) if len(blocks) > 0 else 0
                        for _row in reversed(_CPD_DIAGNOSTICS):
                            if (_row.get("participant") == pid and _row.get("session") == sess
                                    and "n_blocks_post_minsize" not in _row
                                    and _row.get("features_used") is not None):
                                _row["n_blocks_post_minsize"] = n_post_min
                                _row["method_name"] = method_name
                                break
                except Exception:
                    pass
                
                # Merge only the MAD family, on the features that method was
                # detected with (all of them when multivariate).
                if config["merge_blocks"] and method_name.startswith(MERGE_PREFIX):
                    blocks = merge_nonsignificant_blocks(
                        features, blocks, config, method_info["features"]
                    )
                
                features[f'block_{method_name}'] = blocks
                
                metrics = compute_block_quality_metrics(features, blocks, method_info, config)
                metrics['id'] = pid
                metrics['session'] = sess
                metrics['method'] = method_name
                metrics['defined_on'] = method_info['defined_on']
                metrics['detector_failed'] = _failed
                
                session_results.append(metrics)
            except Exception as e:
                print(f"[WARN] block detection failed ({method_name}/{pid}/{sess}): {type(e).__name__}: {e}")
        
        features['id'] = pid
        features['session'] = sess

        return session_results, features, list(_CPD_DIAGNOSTICS)
    
    all_results = []
    all_features = []

    # Parallel processing across (participant, session). n_jobs from config:
    #   -1 (default) -> all cores; 1 -> serial; other int -> exact worker count.
    n_jobs = int(config["n_jobs"])
    verbose = int(config["parallel_verbose"])
    tasks = [(pid, sess) for pid in participants for sess in sessions]
    print(f"  Parallel: n_jobs={n_jobs} over {len(tasks)} (participant, session) tasks")

    results = Parallel(n_jobs=n_jobs, verbose=verbose, backend="loky")(
        delayed(process_session)(pid, sess) for pid, sess in tasks
    )

    # Reset driver-side diagnostics and merge workers' entries.
    global _CPD_DIAGNOSTICS
    _CPD_DIAGNOSTICS = []
    for session_results, features, worker_diags in results:
        if session_results is not None:
            all_results.extend(session_results)
        if features is not None:
            all_features.append(features)
        _CPD_DIAGNOSTICS.extend(worker_diags)

    results_df = pd.DataFrame(all_results)
    features_df = pd.concat(all_features, ignore_index=True) if all_features else pd.DataFrame()
    
    print("=" * 100)
    print("COMPLETE")
    print(f"Results: {len(results_df)} rows")
    print(f"Features: {len(features_df)} rows")
    return results_df, features_df

In [ ]:
# [cell 20]
# =============================================================================
# RUN BLOCK DETECTION
# =============================================================================

# Load data
full_path = Path(LVL0_HANDOFF)
long_df = pd.read_csv(full_path)

# Resolve the untagged residual/prediction columns from feature_rt_transform.
# LVL_0 writes one tagged column set per RT representation and picks no primary;
# this is where LVL_1's choice is materialised so it flows into the LVL_1 handoff.
_RT_TAG = {"raw": "raw", "log": "log", "boxcox": "bc"}
_tag = _RT_TAG[LVL_1_CONFIG["feature_rt_transform"]]
_base = "per_ses" if LVL_1_CONFIG["ses_wise_drift"] else "overall"
# EXPLICIT + VALIDATED. The LVL_1 flag states the intent; the check below catches
# the case where LVL_0 did not actually produce what was asked for -- including
# LVL_0'''s own silent auto-flip to False when no age column is present. Without it a
# mismatch either KeyErrors here, or (worse) silently drops the residual pivot in
# long_to_stim_level so every {scale}_residual disappears with no warning.
_want_age = bool(LVL_1_CONFIG["age_adjust_drift_residuals"])
_age_col  = f"drift_residual_{_tag}_{_base}_age"
_have_age = _age_col in long_df.columns
if _want_age and not _have_age:
    raise SystemExit(
        f"LVL_1 age_adjust_drift_residuals=True but LVL_0 did not hand over {_age_col}. "
        "Set LVL_0_CONFIG['age_adjust_drift_residuals']=True and re-run LVL_0, or set "
        "the LVL_1 flag to False.")
if _have_age and not _want_age:
    print(f"  [age] LVL_0 provided {_age_col} but LVL_1 flag is False "
          f"-> deliberately using the UNADJUSTED residual")
_age_sfx = "_age" if _want_age else ""
long_df["drift_residual"] = long_df[f"drift_residual_{_tag}_{_base}{_age_sfx}"]
long_df["drift_residual_abs"] = long_df["drift_residual"].abs()
long_df["predicted_rt"] = long_df[f"predicted_rt_{_tag}_{_base}"]
print(f"  LVL_1 residual column: drift_residual_{_tag}_{_base}{_age_sfx}"
      f"  (age_adjust flag={_want_age}, LVL_0 supplied _age={_have_age})")

# ---------------------------------------------------------------------------
# rt_input -- THE canonical RT metric, and the LVL_1 -> LVL_2 hand-off column.
# ---------------------------------------------------------------------------
# LVL_0 fits and residualises EVERY combination of RT representation x drift base
# (drift_residual_{raw,log,bc}_{overall,per_ses}[_age]). Exactly one of those is the
# metric this pipeline actually analyses, and the four LVL_1 knobs above
# (feature_rt_transform, ses_wise_drift, age_adjust_drift_residuals, use_residuals)
# are what select it. That selection happens HERE, ONCE, and the result is named
# rt_input. LVL_2 reads rt_input and nothing else, so it needs no copies of those
# knobs and there is no second place where the choice can silently drift out of
# sync with LVL_1. The unused combinations are dropped at the cell-25 export.
_RT_VALUE_COL = {"raw": "rt_sec", "log": "log_rt", "boxcox": "bc_rt"}
if LVL_1_CONFIG["use_residuals"]:
    long_df["rt_input"] = long_df["drift_residual"]
    _rt_input_desc = f"drift_residual_{_tag}_{_base}{_age_sfx}"
else:
    long_df["rt_input"] = long_df[_RT_VALUE_COL[LVL_1_CONFIG["feature_rt_transform"]]]
    _rt_input_desc = _RT_VALUE_COL[LVL_1_CONFIG["feature_rt_transform"]]
print(f"  rt_input (LVL_1 -> LVL_2 hand-off metric) = {_rt_input_desc}")

# Pre-filter snapshot. Filtering is a WITHIN-LEVEL working state: LVL_1 NaNs values so
# detection ignores them, but LVL_2 must be able to run standalone from this file and
# apply its OWN filter knobs, so the untouched values ride along and are dropped again
# at the Step 12 export.
_PREFILTER_COLS = ["rt_sec", "response", "rt_input"]
for _c in _PREFILTER_COLS:
    long_df[f"{_c}_prefilter"] = long_df[_c]

# Apply RT / practice-stimulus filtering (each threshold is its own gate; 0 = off)
long_df = apply_rt_filters(
    long_df,
    max_rt_thresh=LVL_1_CONFIG["max_rt_thresh"],
    min_rt_thresh=LVL_1_CONFIG["min_rt_thresh"],
    pr_stim_drop=LVL_1_CONFIG["pr_stim_drop"],
    verbose=True,
)

# Set out-of-range responses to NaN per scale (preserve row structure)
_scale_ranges = LVL_1_CONFIG["scale_ranges"]
_oor_total = 0
for _sc, (_lo, _hi) in _scale_ranges.items():
    _mask = (long_df["scale"] == _sc) & long_df["response"].notna()
    _oor = _mask & ((long_df["response"] < _lo) | (long_df["response"] > _hi))
    _n_oor = _oor.sum()
    if _n_oor > 0:
        print(f"  [FILTER] {_sc}: setting {_n_oor} out-of-range responses to NaN (expected [{_lo}, {_hi}])")
        long_df.loc[_oor, "response"] = np.nan
        _oor_total += _n_oor
if _oor_total > 0:
    print(f"  [FILTER] Total out-of-range responses set to NaN: {_oor_total}")
else:
    print(f"  [FILTER] No out-of-range responses found")

# Run block detection
results_df, features_df = run_block_detection(
    long_df,
    config=LVL_1_CONFIG,
    methods=None,  # All methods
)

In [ ]:
# [cell 22]
# =============================================================================
# MERGE BLOCK LABELS TO LONG FORMAT 
# =============================================================================

print("\n" + "=" * 70)
print("MERGING BLOCK LABELS TO LONG FORMAT")
print("=" * 70)

# Merge block labels from features_df back to long_df
# This avoids the wasteful melt() operation in Level 2
long_df = merge_block_labels_to_long(long_df, features_df)

# Create trial_label for unique item identification (needed for IRT)
item_col = "stim_label" if "stim_label" in long_df.columns else "stim_order"
long_df["trial_label"] = long_df["scale"].astype(str) + "_" + long_df[item_col].astype(str)

print(f"  Merged block labels into long_df: {len(long_df)} rows")

long_df_with_blocks = long_df.copy()

print("=" * 70)

In [ ]:
# [cell 23]
# =============================================================================
# CREATE FIXED SISTER BLOCKS FOR ALL DYNAMIC METHODS
# =============================================================================
# For each dynamic block method, create a fixed-length control column with:
# - Same median number of blocks per session
# - Equal-sized blocks (longest block at end, given fatigue findings)
# This enables fair comparison: dynamic vs fixed blocking with same granularity

print("="*80)
print("CREATING FIXED SISTER BLOCKS FOR DYNAMIC METHODS")
print("="*80)

def create_fixed_blocks_for_session(n_rows, n_blocks):
    """
    Create fixed-length block assignments for a session.
    
    Parameters
    ----------
    n_rows : int
        Number of rows in the session
    n_blocks : int
        Target number of blocks
    
    Returns
    -------
    np.ndarray
        Block assignments (0 to n_blocks-1)
        Longest block is at the end (for fatigue)
    """
    if n_blocks <= 0:
        return np.zeros(n_rows, dtype=int)
    
    if n_blocks >= n_rows:
        return np.arange(n_rows, dtype=int)
    
    # Calculate base block size and remainder
    base_size = n_rows // n_blocks
    remainder = n_rows % n_blocks
    
    # Create block assignments
    # Put extra rows in LAST blocks (longest at end for fatigue)
    blocks = []
    row_idx = 0
    
    for block_num in range(n_blocks):
        # Last 'remainder' blocks get one extra row
        if block_num >= (n_blocks - remainder):
            block_size = base_size + 1
        else:
            block_size = base_size
        
        blocks.extend([block_num] * block_size)
        row_idx += block_size
    
    return np.array(blocks, dtype=int)


def make_blocks_unique_per_participant(df, block_col, id_col='id', session_col='session'):
    """
    Make block IDs unique per participant instead of per session.
    
    E.g., if session 1 has blocks [0, 1, 2] and session 2 has blocks [0, 1, 2, 3],
    this function renumbers them to [0, 1, 2] and [3, 4, 5, 6] respectively.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with block assignments
    block_col : str
        Name of the block column to renumber
    id_col : str
        Participant ID column
    session_col : str
        Session column
    
    Returns
    -------
    np.ndarray
        Renumbered block assignments unique per participant
    """
    new_blocks = []
    
    for pid in df[id_col].unique():
        pid_mask = df[id_col] == pid
        pid_data = df.loc[pid_mask].copy()
        sessions = sorted(pid_data[session_col].unique())
        
        block_offset = 0
        for sess in sessions:
            sess_mask = pid_data[session_col] == sess
            sess_blocks = pid_data.loc[sess_mask, block_col].values
            
            # Renumber blocks for this session
            renumbered = sess_blocks + block_offset
            new_blocks.extend(renumbered)
            
            # Update offset for next session
            block_offset = renumbered.max() + 1 if len(renumbered) > 0 else block_offset
    
    return np.array(new_blocks, dtype=int)


# Identify dynamic methods (all except Fixed_*)
# control_block_count=None disables sister/control block generation entirely.
# The sweep does not need controls: the LVL_1 method summary (effect size, sig%,
# corr%) is built from results_df, which this cell never touches. Controls are
# regenerated by running the winning config in the notebook with a real value.
_make_controls = LVL_1_CONFIG['control_block_count'] is not None
if not _make_controls:
    print('control_block_count=None -> skipping sister/control block generation')

if _make_controls:
    block_cols = [col for col in features_df.columns if col.startswith('block_')]
    dynamic_methods = [col.replace('block_', '') for col in block_cols 
                       if not col.startswith('block_Fixed_')]

    print(f"\nDynamic methods found: {len(dynamic_methods)}")
    for method in dynamic_methods:
        print(f"  - {method}")

    # Track control columns created
    control_columns = {}

    print("\n" + "-"*80)
    print("Computing median block counts and creating control columns...")
    print("-"*80)

    for method in dynamic_methods:
        block_col = f"block_{method}"
    
        if block_col not in features_df.columns:
            print(f"  {method}: Column not found, skipping")
            continue
    
        # Calculate number of blocks per session
        blocks_per_session = features_df.groupby(["id", "session"])[block_col].apply(
            lambda x: x.nunique()
        )
    
        # Get median (rounded to nearest integer)
        median_n_blocks = int(np.round(blocks_per_session.median()))
        mean_n_blocks = blocks_per_session.mean()
        min_n_blocks = blocks_per_session.min()
        max_n_blocks = blocks_per_session.max()

        # Granularity rule for the control, from
        # LVL_1_CONFIG["control_block_count"] ("median" | "mean" | "per_session").
        # Why it matters: at "median" the control gets
        # ~9% MORE blocks than the dynamic design it controls for, so it removes more data
        # and wins the raw Delta-alpha comparison on volume rather than boundary quality.
        _control_rule = str(LVL_1_CONFIG["control_block_count"]).lower()
        if _control_rule == "mean":
            _target_n_blocks = int(np.round(mean_n_blocks))
        else:                       # "median" and "per_session" both use the median here
            _target_n_blocks = median_n_blocks

        print(f"\n{method}:")
        print(f"  Blocks per session: median={median_n_blocks}, mean={mean_n_blocks:.1f}, range=[{min_n_blocks}, {max_n_blocks}]")
        print(f"  Control rule: {_control_rule}" +
              ("" if _control_rule == "per_session" else f" -> {_target_n_blocks} blocks/session"))

        if _target_n_blocks < 1 and _control_rule != "per_session":
            print(f"  Skipping: {_control_rule} blocks < 1")
            continue
    
        # Create control column name
        control_col = f"block_Fixed_{method}_based"
    
        # Create fixed blocks for each session
        control_blocks = []
    
        # Sort features_df to ensure control_blocks aligns correctly with rows
        features_df = features_df.sort_values(["id", "session", "stim_order"]).reset_index(drop=True)
    
        for (pid, sess), group in features_df.groupby(["id", "session"], sort=True):
            n_rows = len(group)
            # per_session matches THIS session's dynamic block
            # count, so the control has identical granularity and only the boundary
            # PLACEMENT differs. Falls back to the global target if the session is missing
            # from blocks_per_session, and clamps to >=1 so a session can never get 0 blocks.
            if _control_rule == "per_session":
                _n_this = int(blocks_per_session.get((pid, sess), _target_n_blocks))
                _n_this = max(1, _n_this)
            else:
                _n_this = _target_n_blocks
            fixed_assignments = create_fixed_blocks_for_session(n_rows, _n_this)
            control_blocks.extend(fixed_assignments)
    
        # Add to features_df
        features_df[control_col] = control_blocks
    
        # Verify block sizes
        control_blocks_per_session = features_df.groupby(["id", "session"])[control_col].nunique()
    
        print(f"  Created: {control_col}")
        print(f"  Fixed blocks per session: {control_blocks_per_session.unique()}")
    
        # Store mapping
        control_columns[method] = {
            "dynamic_col": block_col,
            "control_col": control_col,
            "median_n_blocks": median_n_blocks,
            "mean_n_blocks": mean_n_blocks
        }

    # =============================================================================
    # MAKE BLOCK IDs UNIQUE PER PARTICIPANT (NOT PER SESSION)
    # =============================================================================
    # Block detection currently numbers blocks 0, 1, 2... within each session.
    # For downstream analysis, it's cleaner to have unique block IDs per participant:
    # Session 1: blocks 0, 1, 2 -> Session 2: blocks 3, 4, 5, 6

    print("\n" + "="*80)
    print("MAKING BLOCK IDs UNIQUE PER PARTICIPANT")
    print("="*80)

    # Get all block columns (dynamic and fixed controls)
    all_block_cols = [col for col in features_df.columns if col.startswith('block_')]

    for block_col in all_block_cols:
        # Create unique version column name
        unique_col = f"{block_col}_unique"
    
        # Make blocks unique per participant
        features_df[unique_col] = make_blocks_unique_per_participant(
            features_df, block_col, id_col='id', session_col='session'
        )
    
    print(f"\nCreated {len(all_block_cols)} unique block ID columns:")
    print("  Original column -> Unique column")
    for col in all_block_cols[:5]:  # Show first 5
        print(f"    {col} -> {col}_unique")
    if len(all_block_cols) > 5:
        print(f"    ... and {len(all_block_cols) - 5} more")

    # =============================================================================
    # SUMMARY: DYNAMIC vs FIXED SISTER BLOCK COMPARISON
    # =============================================================================

    print("\n" + "="*80)
    print("SUMMARY: DYNAMIC METHODS AND THEIR FIXED SISTERS")
    print("="*80)

    print(f"\n{'Method':<30} {'Median Blocks':<15} {'Control Column':<40}")
    print("-"*85)

    for method, info in control_columns.items():
        print(f"{method:<30} {info['median_n_blocks']:<15} {info['control_col']:<40}")

    print(f"\nTotal control columns created: {len(control_columns)}")

    # =============================================================================
    # BLOCK QUALITY COMPARISON: DYNAMIC vs FIXED SISTERS
    # =============================================================================

    print("\n" + "="*80)
    print("BLOCK QUALITY METRICS: DYNAMIC vs FIXED CONTROL")
    print("="*80)

    def compute_block_size_metrics(df, block_col):
        """Compute quality metrics for blocks defined by block_col."""
        metrics = {}
    
        # Block size statistics
        block_sizes = df.groupby(["id", "session", block_col]).size()
        metrics["block_size_mean"] = block_sizes.mean()
        metrics["block_size_std"] = block_sizes.std()
        metrics["block_size_cv"] = block_sizes.std() / block_sizes.mean() if block_sizes.mean() > 0 else 0
    
        # Number of blocks
        n_blocks = df.groupby(["id", "session"])[block_col].nunique()
        metrics["n_blocks_mean"] = n_blocks.mean()
        metrics["n_blocks_std"] = n_blocks.std()
    
        # RT variance within blocks (lower = more homogeneous = better blocking)
        if "rt_variance" in df.columns:
            within_block_rt_var = df.groupby(["id", "session", block_col])["rt_variance"].mean()
            metrics["within_block_rt_var"] = within_block_rt_var.mean()
    
        return metrics

    print(f"\n{'Method':<25} {'Type':<10} {'Blocks':<10} {'Size Mean':<12} {'Size CV':<10} {'RT Var':<10}")
    print("-"*85)

    for method, info in control_columns.items():
        # Dynamic method metrics
        dyn_metrics = compute_block_size_metrics(features_df, info["dynamic_col"])
        print(f"{method:<25} {'Dynamic':<10} {dyn_metrics['n_blocks_mean']:<10.1f} "
              f"{dyn_metrics['block_size_mean']:<12.1f} {dyn_metrics['block_size_cv']:<10.2f} "
              f"{dyn_metrics.get('within_block_rt_var', 'N/A'):<10}")
    
        # Control (fixed) method metrics
        fix_metrics = compute_block_size_metrics(features_df, info["control_col"])
        print(f"{'  -> Fixed Control':<25} {'Fixed':<10} {fix_metrics['n_blocks_mean']:<10.1f} "
              f"{fix_metrics['block_size_mean']:<12.1f} {fix_metrics['block_size_cv']:<10.2f} "
              f"{fix_metrics.get('within_block_rt_var', 'N/A'):<10}")
        print()

    # Store control_columns mapping for use in Level 2
    CONTROL_COLUMNS = control_columns

    # =============================================================================
    # MERGE FIXED CONTROL BLOCK COLUMNS TO LONG FORMAT
    # =============================================================================
    # The long_df_with_blocks was created before fixed blocks existed.
    # We need to merge the new control block columns for IRT scoring.

    print("\n" + "="*80)
    print("MERGING FIXED CONTROL BLOCKS TO LONG FORMAT")
    print("="*80)

    # Get the new control columns that need to be merged
    fixed_block_cols = [info["control_col"] for info in control_columns.values()]
    fixed_unique_cols = [f"{col}_unique" for col in fixed_block_cols]
    cols_to_merge = fixed_block_cols + fixed_unique_cols

    # Only keep columns that exist in features_df
    cols_to_merge = [c for c in cols_to_merge if c in features_df.columns]

    if len(cols_to_merge) > 0:
        # Merge on id, session, stim_order (the key that links features_df rows to long_df rows)
        merge_keys = ["id", "session", "stim_order"]

        # Select only the merge keys and new columns from features_df
        features_subset = features_df[merge_keys + cols_to_merge].copy()
    
        # Ensure consistent dtypes for merge keys (handle int vs object mismatch)
        for key in merge_keys:
            if key in long_df_with_blocks.columns and key in features_subset.columns:
                # Convert both to string to ensure compatibility
                long_df_with_blocks[key] = long_df_with_blocks[key].astype(str)
                features_subset[key] = features_subset[key].astype(str)

        # Merge into long_df_with_blocks
        long_df_with_blocks = long_df_with_blocks.merge(
            features_subset,
            on=merge_keys,
            how="left"
        )

        print(f"  Merged {len(cols_to_merge)} fixed control columns to long_df_with_blocks")
        for col in cols_to_merge[:6]:  # Show first 6
            print(f"    - {col}")
        if len(cols_to_merge) > 6:
            print(f"    ... and {len(cols_to_merge) - 6} more")
    else:
        print("  No fixed control columns to merge")

    print("\n" + "="*80)
    print("FIXED SISTER BLOCKS COMPLETE")
    print("="*80)
    print("\nControl columns stored in: CONTROL_COLUMNS")
    print("These will be used for comparison in Level 2 analysis.")

In [ ]:
# [cell 24]
# =============================================================================
# SUMMARY: METHOD PERFORMANCE & BLOCK COUNTS 
# =============================================================================

def summarize_block_methods(results_df, config, min_n=10):
    """
    Compute comprehensive summary of block detection methods.
    Handles cases where no validation features are available.
    
    Corr_Sig%: significance rate corrected for sessions with only 1 block.
    - Sig% = significant / sessions_with_2+_blocks
    - Corr_Sig% = significant / total_sessions
    """
    val_settings = config["validation_features"]
    
    # Validation features derived from FEATURE_REGISTRY (cell 15) so the
    # `<prefix>_p` / `<prefix>_effsize` column names stay in lockstep with the
    # validation metrics that compute_block_quality_metrics writes. Restricted to
    # config["validation_features"] so the reported set matches what was tested.
    VALIDATION_METRICS = [
        (f"{spec['validation_prefix']}_p",
         f"{spec['validation_prefix']}_effsize",
         name)
        for name, spec in FEATURE_REGISTRY.items()
        if val_settings.get(name, False)
    ]

    rows = []
    methods_without_validation = []
    
    for method, md in results_df.groupby("method"):
        if len(md) < min_n:
            continue

        row = {
            "method": method,
            "defined_on": md["defined_on"].iloc[0] if "defined_on" in md.columns else None,
            "n_obs": len(md),
            "n_participants": md["id"].nunique() if "id" in md.columns else len(md),
        }

        # Block count statistics
        if "n_blocks" in md.columns:
            row["blocks_mean"] = md["n_blocks"].mean()
            row["blocks_median"] = md["n_blocks"].median()
            row["blocks_std"] = md["n_blocks"].std()
            row["blocks_min"] = int(md["n_blocks"].min())
            row["blocks_max"] = int(md["n_blocks"].max())
            
            # Count sessions with 2+ blocks (validatable)
            row["n_validatable"] = int((md["n_blocks"] >= 2).sum())
            row["n_single_block"] = int((md["n_blocks"] < 2).sum())
        else:
            row["n_validatable"] = 0
            row["n_single_block"] = len(md)
        
        # Block size balance (CV)
        if "block_cv" in md.columns:
            row["blocks_cv"] = md["block_cv"].mean()
        else:
            row["blocks_cv"] = np.nan

        # ANY session with external validation features counts (not just the first)
        if "n_external_features_used" in md.columns:
            n_features_used = md["n_external_features_used"].max()  # Use MAX, not iloc[0]
        else:
            n_features_used = 0
        
        if n_features_used == 0 or row["n_validatable"] == 0:
            # No external features available for this method
            methods_without_validation.append(method)
            row["avg_sig%"] = np.nan
            row["avg_corr_sig%"] = np.nan
            row["avg_effsize"] = np.nan
            rows.append(row)
            continue

        # Validation metrics
        total_sig = 0.0
        total_corr_sig = 0.0
        total_omega = 0.0
        n_metrics = 0
        total_sessions = len(md)

        for p_col, omega_col, display_name in VALIDATION_METRICS:
            if not val_settings.get(display_name, False):
                continue

            if p_col not in md.columns:
                continue

            # Get p-values (NaN for single-block sessions)
            p_vals_all = md[p_col].values
            p_vals = md[p_col].dropna().values
            
            if len(p_vals) == 0:
                continue

            # FDR correction on validatable sessions
            _, p_fdr = fdrcorrection(p_vals, alpha=0.05)
            n_significant = (p_fdr < 0.05).sum()
            
            # Sig%: significant / validatable sessions
            sig_rate = 100.0 * n_significant / len(p_vals) if len(p_vals) > 0 else 0.0
            
            # Corr_Sig%: significant / ALL sessions (treats single-block as non-significant)
            corr_sig_rate = 100.0 * n_significant / total_sessions if total_sessions > 0 else 0.0

            omega_vals = md[omega_col].dropna().values if omega_col in md.columns else np.array([])
            omega_mean = float(np.mean(omega_vals)) if len(omega_vals) else 0.0

            row[f"{display_name}_sig%"] = sig_rate
            row[f"{display_name}_corr_sig%"] = corr_sig_rate
            row[f"{display_name}_effsize"] = omega_mean

            total_sig += sig_rate
            total_corr_sig += corr_sig_rate
            total_omega += omega_mean
            n_metrics += 1

        if n_metrics > 0:
            row["avg_sig%"] = total_sig / n_metrics
            row["avg_corr_sig%"] = total_corr_sig / n_metrics
            row["avg_effsize"] = total_omega / n_metrics
            
        else:
            row["avg_sig%"] = np.nan
            row["avg_corr_sig%"] = np.nan
            row["avg_effsize"] = np.nan

        rows.append(row)
    
    # Print warning about methods without validation
    if methods_without_validation:
        print("\n" + "=" * 70)
        print("WARNING: Methods with no external validation features available:")
        print("=" * 70)
        for method in methods_without_validation:
            method_data = results_df[results_df["method"] == method]
            n_single = (method_data["n_blocks"] < 2).sum() if "n_blocks" in method_data.columns else len(method_data)
            print(f"  - {method}: {n_single}/{len(method_data)} sessions have only 1 block")
        print("=" * 70 + "\n")

    summary = pd.DataFrame(rows)


    return summary


def print_method_summary(summary_df):
    """Print the method ranking table, ordered by mean effect size."""

    if summary_df.empty:
        print("No results to summarize.")
        return summary_df

    # Rank on separation magnitude (mean effect size).
    _sort_col = "avg_effsize" if "avg_effsize" in summary_df.columns else "avg_corr_sig%"
    summary_df = summary_df.sort_values(_sort_col, ascending=False).reset_index(drop=True)

    W = 137
    print("=" * W)
    print("BLOCK DETECTION METHOD SUMMARY (ranked by mean effect size)")
    print("=" * W)
    print(f"{'Rank':<5} {'Method':<22} {'Sig%':>7} {'Corr%':>7} {'effect':>8} "
          f"{'Mean':>6} {'Mdn':>5} {'Std':>5} {'Min':>4} {'Max':>4} {'CV':>5} {'N':>5} {'Val':>4}")
    print("-" * W)

    for i, row in summary_df.iterrows():
        cv = row.get("blocks_cv", np.nan)
        cv_str = f"{cv:.2f}" if pd.notna(cv) else "?"

        sig_pct = row.get('avg_sig%', np.nan)
        corr_sig_pct = row.get('avg_corr_sig%', np.nan)
        omega = row.get('avg_effsize', np.nan)

        sig_str = f"{sig_pct:>6.1f}%" if pd.notna(sig_pct) else f"{'nan':>6}%"
        corr_str = f"{corr_sig_pct:>6.1f}%" if pd.notna(corr_sig_pct) else f"{'nan':>6}%"
        omega_str = f"{omega:>8.4f}" if pd.notna(omega) else f"{'nan':>8}"

        print(f"{i+1:<5} {row['method']:<22} "
              f"{sig_str} "
              f"{corr_str} "
              f"{omega_str} "
              f"{row.get('blocks_mean', np.nan):>6.1f} "
              f"{row.get('blocks_median', np.nan):>5.1f} "
              f"{row.get('blocks_std', np.nan):>5.2f} "
              f"{row.get('blocks_min', '?'):>4} "
              f"{row.get('blocks_max', '?'):>4} "
              f"{cv_str:>5} "
              f"{row.get('n_obs', 0):>5} "
              f"{row.get('n_validatable', 0):>4}")

    print("=" * W)
    print("effect = ω² if validate_test='anova', else η²_H (Kruskal-Wallis)")
    print("Sig%  = significant sessions / sessions with 2+ blocks")
    print("Corr% = significant sessions / ALL sessions (corrected for single-block sessions)")
    print("Val   = number of sessions with 2+ blocks (validatable)")
    print("CV    = coefficient of variation of block sizes (0 = perfectly equal)")
    print("        CV < 0.3: good balance | 0.3-0.6: moderate | > 0.6: high imbalance")

    return summary_df


def plot_block_histogram(results_df, method_name):
    """Plot histogram of block counts for a selected method."""
    
    method_data = results_df[results_df["method"] == method_name]
    
    if len(method_data) == 0:
        print(f"Method '{method_name}' not found.")
        print(f"Available: {sorted(results_df['method'].unique())}")
        return
    
    block_counts = method_data.groupby("id")["n_blocks"].mean()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(block_counts, bins=range(1, int(block_counts.max()) + 2),
            edgecolor="black", alpha=0.7, align="left", color="steelblue")
    ax.set_xlabel("Number of Blocks")
    ax.set_ylabel("Number of Participants")
    ax.set_title(f"Block Count Distribution: {method_name}")
    ax.set_xticks(range(1, int(block_counts.max()) + 1))
    
    # Get CV for stats box
    if "block_cv" in method_data.columns:
        mean_cv = method_data["block_cv"].mean()
    else:
        mean_cv = np.nan
    
    if pd.notna(mean_cv):
        stats_text = (f"Mean: {block_counts.mean():.2f}\n"
                      f"Median: {block_counts.median():.0f}\n"
                      f"Std: {block_counts.std():.2f}\n"
                      f"CV: {mean_cv:.2f}")
    else:
        stats_text = (f"Mean: {block_counts.mean():.2f}\n"
                      f"Median: {block_counts.median():.0f}\n"
                      f"Std: {block_counts.std():.2f}")
    
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
            va="top", ha="right", bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    plt.show()


# =============================================================================
# RUN SUMMARY
# =============================================================================

summary_df = summarize_block_methods(results_df, config=LVL_1_CONFIG, min_n=10)
summary_df = print_method_summary(summary_df)

_hist_method = "PELT_Attention_comp"
plot_block_histogram(results_df, method_name=_hist_method)

In [ ]:
# [notebook only] LVL_1 REPORTING: MAD-CPD DIAGNOSTICS
# =============================================================================
# The MAD-CPD detectors' per-gap statistics, pooled over every session they segmented in this run: one row per feature
# set, the local MAD at each gap (left) and the gap's MAD-z score (right), with the 90th / 95th percentiles and the
# threshold k applied. From get_cpd_diagnostics_df() ([cell 15]), recorded by the detection of [cell 20]. Draws only.
def plot_mad_cpd_diagnostics():
    df = get_cpd_diagnostics_df()
    mad_df = df[df["method"] == "CPD"].copy() if "method" in df.columns else pd.DataFrame()
    if mad_df.empty:
        print("[CPD Diagnostics] No MAD-CPD runs to plot.")
        return
    feature_groups = []
    for fg in mad_df["features_used"].unique():
        sub = mad_df[mad_df["features_used"] == fg]
        arrs = [a for a in sub.get("local_mad", []) if a is not None]
        if arrs:
            feature_groups.append(fg)
    if not feature_groups:
        return
    n_groups = len(feature_groups)
    fig, axes = plt.subplots(n_groups, 2, figsize=(13, 4 * n_groups), squeeze=False)

    for row_i, feats in enumerate(feature_groups):
        sub = mad_df[mad_df["features_used"] == feats]
        pooled_mad = np.concatenate([np.asarray(a, dtype=float) for a in sub["local_mad"].dropna()])
        pooled_z_raw = np.concatenate([np.asarray(a, dtype=float) for a in sub["diff_scores"].dropna()])
        pooled_mad = pooled_mad[~np.isnan(pooled_mad)]
        pooled_z = pooled_z_raw[~np.isnan(pooled_z_raw) & np.isfinite(pooled_z_raw)]

        if "k_used" in sub.columns:
            kvs = sub["k_used"].dropna().unique()
            k_label = f"k={kvs[0]:.2f}" if len(kvs) else "k=?"
            k_band = None
            k_lines = list(kvs)
        else:
            k_label, k_band, k_lines = "k=?", None, []

        ws_val = sub["mad_window_size"].iloc[0] if "mad_window_size" in sub.columns else None
        window_label = "global MAD" if (ws_val is None or (isinstance(ws_val, float) and np.isnan(ws_val))) else f"window={int(ws_val)}"

        axL = axes[row_i, 0]
        axL.hist(pooled_mad, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
        q95 = np.quantile(pooled_mad, 0.95)
        q90 = np.quantile(pooled_mad, 0.90)
        axL.axvline(np.median(pooled_mad), color='black', ls=':', lw=1.5,
                    label=f'median={np.median(pooled_mad):.3f}')
        axL.axvline(q90, color='orange', ls='--', lw=2, label=f'10% cutoff (90th)={q90:.3f}')
        axL.axvline(q95, color='red', ls='--', lw=2, label=f'5% cutoff (95th)={q95:.3f}')
        axL.set_xlabel('local MAD (raw, in signal units)')
        axL.set_ylabel('Count of gaps')
        axL.set_title(f'{feats} - local MAD per gap\n{window_label}, {k_label}, n_gaps={len(pooled_mad)}')
        axL.legend(loc='upper right', fontsize=8)

        axR = axes[row_i, 1]
        if len(pooled_z) > 0:
            z_plot = pooled_z[(pooled_z > -20) & (pooled_z < 50)]
            axR.hist(z_plot, bins=60, edgecolor='black', alpha=0.7, color='seagreen')
            z_q90 = np.quantile(pooled_z, 0.90)
            z_q95 = np.quantile(pooled_z, 0.95)
            axR.axvline(z_q90, color='orange', ls='--', lw=2, label=f'10% cutoff (90th)={z_q90:.2f}')
            axR.axvline(z_q95, color='red', ls='--', lw=2, label=f'5% cutoff (95th)={z_q95:.2f}')
            if k_band is not None:
                axR.axvspan(k_band[0], k_band[1], color='purple', alpha=0.15,
                            label=f'k applied range = [{k_band[0]:.2f}, {k_band[1]:.2f}]')
            for k_val in k_lines:
                axR.axvline(k_val, color='purple', ls='-', lw=2, alpha=0.7,
                            label=f'k applied = {k_val:.2f}')
        axR.set_xlabel('per-gap MAD-z score (k equivalent)')
        axR.set_ylabel('Count of gaps')
        axR.set_title(f'{feats} - z-scored diffs\n{window_label}, {k_label}, n_gaps={len(pooled_z)}')
        axR.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()


plot_mad_cpd_diagnostics()


In [ ]:
# [cell 25]
# LVL_1 output, written after the control-block augmentation so the files LVL_2 reads
# carry the control-block and per-participant-unique columns.

# Filtering is a within-level working state. Restore the untouched values before
# export so LVL_2 receives an unfiltered frame and applies its own thresholds.
for _c in _PREFILTER_COLS:
    long_df_with_blocks[_c] = long_df_with_blocks[f"{_c}_prefilter"]
long_df_with_blocks = long_df_with_blocks.drop(
    columns=[c for c in long_df_with_blocks.columns if c.endswith("_prefilter")])

# Collapse the LVL_0 residual grid before it crosses the LVL_1/LVL_2 boundary.
# LVL_0 wrote drift_residual_{raw,log,bc}_{overall,per_ses}[_age] and the matching
# predicted_rt_* columns so each level COULD pick independently. Independence is
# exactly the failure mode: LVL_2 re-deriving the same choice from its own knobs is
# how the two levels drift apart. Cell 20 already resolved the one combination this
# pipeline uses into `rt_input` (plus the untagged drift_residual / predicted_rt that
# the Step 12 export documents), so every other combination is dead weight and is
# dropped here. LVL_2 therefore has only one RT metric available to it: rt_input.
_handoff_drop = [
    c for c in long_df_with_blocks.columns
    if (c.startswith("drift_residual_") or c.startswith("predicted_rt_"))
    and c != "drift_residual_abs"
]
if _handoff_drop:
    long_df_with_blocks = long_df_with_blocks.drop(columns=_handoff_drop)
    print(f"LVL_1 handoff: dropped {len(_handoff_drop)} unused drift/prediction "
          f"combinations; LVL_2 receives rt_input only")

# TIMESTAMPED LVL_1 HANDOFF -- the boundary between LVL_1 and LVL_2. Everything LVL_2
# needs is written here as an immutable stamped set, so a downstream analysis can pin
# one generation of LVL_1 output instead of racing the next pipeline run:
#   *_long.csv     trial-level, with every block_* column (dynamic + control + unique)
#   *_features.csv stimulus-level features_df, same block columns
#   *_meta.json    provenance + the LVL_1 config that produced them
# written to LVL_OUTS ([cell 2]), with the run log *_parameters.json (below).
LVL1_EXPORT_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
_lvl1_long  = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}_long.csv"
_lvl1_feats = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}_features.csv"
_lvl1_meta  = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}_meta.json"
# results_df is in the handoff because get_block_feature_matrix() (LVL_2) reads it
# to rebuild the per-method block matrices.
_lvl1_results = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}_results.csv"

long_df_with_blocks.to_csv(_lvl1_long, index=False)
features_df.to_csv(_lvl1_feats, index=False)
results_df.to_csv(_lvl1_results, index=False)

_blk_cols = sorted(c for c in long_df_with_blocks.columns if c.startswith("block_"))
json.dump({
    "export_timestamp":   LVL1_EXPORT_TS,
    "stage":              "LVL_1 output (input to LVL_2)",
    "input_long_path":    str(INPUT_LONG_PATH),
    "long_rows":          int(len(long_df_with_blocks)),
    "long_participants":  int(long_df_with_blocks["id"].nunique()),
    "features_rows":      int(len(features_df)),
    "block_columns":      _blk_cols,
    "lvl1_config":        {k: str(v) for k, v in LVL_1_CONFIG.items()},
    "use_residuals":      bool(LVL_1_CONFIG["use_residuals"]),
}, open(_lvl1_meta, "w"), indent=1)

print(f"\nTimestamped LVL_1 handoff ({LVL1_EXPORT_TS}):")
print(f"  {_lvl1_long.name}      {len(long_df_with_blocks):,} rows, "
      f"{long_df_with_blocks['id'].nunique()} participants, {len(_blk_cols)} block columns")
print(f"  {_lvl1_feats.name}  {len(features_df):,} rows")
print(f"  {_lvl1_meta.name}")

# =============================================================================
# LVL_1 PARAMETER / STATISTICAL-RESULTS MANIFEST
# =============================================================================
_stat_cols = [c for c in results_df.columns
              if c in ("id", "session", "method", "defined_on")
              or any(k in c.lower() for k in ("sig", "pval", "p_", "effect", "quality",
                                              "n_blocks", "eps", "effsize", "stat"))]
_lvl1_stats = {
    "validation_tests": {
        "test": LVL_1_CONFIG["validate_test"],
        "features": list(LVL_1_CONFIG["validation_features"].keys()),
        "per_session_results": results_df[_stat_cols] if _stat_cols else results_df,
        "by_method_summary": (
            results_df.select_dtypes(include=[np.number])
                      .groupby(results_df["method"]).mean().reset_index()
            if "method" in results_df.columns and len(results_df) else pd.DataFrame()
        ),
    },
    "changepoint_diagnostics": get_cpd_diagnostics_df(),
    "block_counts_by_method": (
        features_df[_blk_cols].nunique().to_dict() if _blk_cols else {}
    ),
}
dump_level_json(
    "LVL_1",
    LVL_1_CONFIG,
    results=_lvl1_stats,
    extra={
        "annotated_long_csv": str(_lvl1_long),
        "features_csv": str(_lvl1_feats),
        "results_csv": str(_lvl1_results),
        "block_columns": _blk_cols,
        "n_rows": int(len(long_df_with_blocks)),
        "n_participants": int(long_df_with_blocks["id"].nunique()),
    },
    name=f"lv1_out_{LVL1_EXPORT_TS}",
)

In [ ]:
# [notebook only] LVL_1 HANDOFF LINEAGE
# =============================================================================
# Records in the stamped handoff's _meta.json ([cell 25]) which LVL_0 handoff this LVL_1 run
# read, the run log holding its LVL_0 configuration and the raw input's SHA-256 (the LVL2 handoff check verifies it),
# so a branch traces back:
#   lv1_out_<ts>_meta.json (lvl1_config) -> lvl0_input -> lvl0_record (lv0_out_<ts>_parameters.json: LVL_0 config)
# (A handoff written by lvl1.py --export has no lvl0_input here; its sweep rows carry one.)
with open(_lvl1_meta) as _f:
    _m1 = json.load(_f)
_m1["lvl0_input"] = Path(LVL0_HANDOFF).name
_m1["lvl0_input_path"] = str(Path(LVL0_HANDOFF).resolve())
_m1["lvl0_record"] = LVL0_RECORD_PATH.name
_m1["input_long_sha256"] = LVL0_RECORD["outputs"]["input_long_sha256"]
with open(_lvl1_meta, "w") as _f:
    json.dump(_m1, _f, indent=1)
print(f"lineage -> {_lvl1_meta.name}: lvl0_input={_m1['lvl0_input']}, lvl0_record={_m1['lvl0_record']}")

In [ ]:
# [notebook only] LVL_1 REPORTING: DETECTOR SUMMARY EXPORT
# =============================================================================
# The detector summary of [cell 24] (summary_df: every detector on this handoff, ranked best first by mean effect
# size), written as <handoff prefix>_report_method_summary.csv. LVL1_report.ipynb reads it to order the detectors of its
# participant figures.
_ms_out = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}_report_method_summary.csv"
summary_df.to_csv(_ms_out, index=False)
print(f"detector summary -> {_ms_out.name} ({len(summary_df)} detectors)")


In [ ]:
# [notebook only] H1: SESSION-LEVEL DATA OF THE DATASETS
# =============================================================================
# H1 (which detectors segment the sessions best?) on the datasets DS, exported for the final report as
# <handoff prefix>_h1_<table>.csv. Per participant-session and detector:
#   eta   eta^2_H = (H - k + 1)/(n - k) of the Kruskal-Wallis test of each validation feature across the session's
#         blocks, averaged over the three features; defined only for sessions split into >= 2 blocks
#   corr  share of the three features significant after BH across the detector's validatable sessions of the dataset
#         (an unsplit session scores 0)
# T2+T3 calibration = each detector at its own eta^2_H-optimal sweep configuration (the LVL1 sweep report's
# best_per_detector; the sweep's per-session results); WE = corr in T2+T3 calibration with each detector at its own
# Corr%-optimal sweep configuration instead (best_corr). T3 only = this handoff's T3 sessions (every detector at the
# selected configuration).
from statsmodels.stats.multitest import multipletests

H1_SWEEP_DIR = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/practical_application_emst/sweeps") / LVL1_SWEEP
H1_STEM = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}_h1"
HANDOFF = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}"
WINNER_LVL1 = LVL1_SELECTION["detector"]
ALPHA = 0.05
DS = ["T2+T3 calibration", "T3 only"]
# the pooled datasets
DS += ["Pooled, overall", "Pooled, T2", "Pooled, T3"]
H1_TABLES = {}


def h1_export(key, frame):
    """Write one H1 table as <handoff prefix>_h1_<key>.csv and show it."""
    H1_TABLES[key] = frame
    frame.to_csv(f"{H1_STEM}_{key}.csv", index=False)
    display(frame)


def h1_check(name, ok, detail=""):
    if not ok:
        raise SystemExit(f"H1 input check failed: {name} ({detail})")
    print(f"check: {name}" + (f" ({detail})" if detail else ""))


def smad(x):
    """Scaled MAD = 1.4826 x median absolute deviation."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return 1.4826 * np.median(np.abs(x - np.median(x))) if len(x) else np.nan


def summ(x):
    """n, median, Q1, Q3, scaled MAD of the finite values of x."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if not len(x):
        return dict(n=0, median=np.nan, q1=np.nan, q3=np.nan, smad=np.nan)
    return dict(n=len(x), median=np.median(x), q1=np.percentile(x, 25), q3=np.percentile(x, 75), smad=smad(x))


def summ_str(x, nd=3):
    """'median [Q1, Q3] (sMAD)' of x."""
    s = summ(x)
    return "—" if not s["n"] else f"{s['median']:.{nd}f} [{s['q1']:.{nd}f}, {s['q3']:.{nd}f}] ({s['smad']:.{nd}f})"


def bh(p):
    """Benjamini-Hochberg q-values; NaN stays NaN."""
    p = np.asarray(p, float); q = np.full(len(p), np.nan); ok = np.isfinite(p)
    if ok.any():
        q[ok] = multipletests(p[ok], method="fdr_bh")[1]
    return q


EFF1 = ["adj_rt_level_stim_effsize", "adj_rt_var_win_effsize", "entropy_win_effsize"]
P1 = [c.replace("_effsize", "_p") for c in EFF1]
res1 = pd.read_csv(f"{HANDOFF}_results.csv")
COND_OF = (pd.read_csv(f"{HANDOFF}_features.csv", usecols=["id", "condition"]).drop_duplicates()
             .set_index("id")["condition"])
res1["condition"] = res1["id"].map(COND_OF)
_best_eta = pd.read_csv(LVL_OUTS / f"{LVL1_SWEEP}_report_best_per_detector.csv").set_index("method")
METHODS1 = sorted(_best_eta.index)
h1_check("the sweep's detectors are the handoff's", set(METHODS1) == set(res1.method.unique()),
         f"{sorted(set(METHODS1) ^ set(res1.method.unique()))}")
cal = []
for m in METHODS1:
    _s = pd.read_csv(H1_SWEEP_DIR / f"sessions_{_best_eta.loc[m, 'cfg_file']}")
    cal.append(_s[_s.method == m])
cal = pd.concat(cal, ignore_index=True)


def corr_scores(d):
    """Per session x detector: share of the 3 features significant after BH across the detector's
    validatable sessions in d (an unsplit session = 0)."""
    out = []
    for m, g in d.groupby("method"):
        flags = []
        for pc in P1:
            s = pd.Series(0.0, index=g.index); ok = g[pc].notna()
            if ok.any():
                s[ok] = (bh(g.loc[ok, pc].to_numpy()) < ALPHA).astype(float)
            flags.append(s)
        out.append(pd.Series(pd.concat(flags, axis=1).mean(axis=1).values, index=g.index))
    return pd.concat(out)


def session_frame(d, family):
    d = d.copy()
    d["eta"] = d[EFF1].mean(axis=1, skipna=False)
    d["corr"] = corr_scores(d) if family is None else family.loc[d.index]
    return d


SES1 = {"T2+T3 calibration":  session_frame(cal, None),
        "T3 only":         session_frame(res1[res1.condition == "T3"], None)}
# the pooled datasets: this handoff at the selected configuration
# T3 only and Pooled, T3 share eta^2_H and differ in the BH family behind corr.
_pool = res1.copy(); _pool_corr = corr_scores(_pool)
SES1.update({"Pooled, overall": session_frame(_pool, _pool_corr),
             "Pooled, T2":      session_frame(_pool[_pool.condition == "T2"], _pool_corr),
             "Pooled, T3":      session_frame(_pool[_pool.condition == "T3"], _pool_corr)})
MAT1 = {ds: {v: d.set_index(["id", "session", "method"])[v].unstack("method") for v in ("eta", "corr", "n_blocks")}
        for ds, d in SES1.items()}          # rows = the dataset's real sessions only
for ds in DS:
    MAT1[ds]["corr"] = MAT1[ds]["corr"].fillna(0.0)
_rec = 100 * MAT1["T2+T3 calibration"]["corr"].mean()
h1_check("T2+T3 calibration: reconstructed Corr% = sweep avg_corr_sig% at each detector's η²_H-optimal config",
         np.allclose(_rec[METHODS1].values, _best_eta.loc[METHODS1, "Corr%"].values, atol=1e-9),
         f"max |diff| {np.abs(_rec[METHODS1].values - _best_eta.loc[METHODS1, 'Corr%'].values).max():.1e} pp")
h1_check("T2+T3 calibration: reconstructed η²_H mean = sweep avg_effsize at each detector's optimum",
         np.allclose(MAT1["T2+T3 calibration"]["eta"].mean()[METHODS1].values, _best_eta.loc[METHODS1, "η²_H"].values,
                     atol=1e-9))
_n_ses = {ds: len(MAT1[ds]["eta"]) for ds in DS}
_n_handoff = {c: len(res1.loc[res1.condition == c, ["id", "session"]].drop_duplicates()) for c in ("T2", "T3")}
h1_check("sessions: T2+T3 calibration = the handoff's T2 and T3 sessions; T3 only = its T3 sessions",
         _n_ses["T2+T3 calibration"] == _n_handoff["T2"] + _n_handoff["T3"] and _n_ses["T3 only"] == _n_handoff["T3"],
         f"{_n_ses}; handoff {_n_handoff}")
# the pooled datasets = the handoff's T2 and T3 sessions
h1_check("sessions: Pooled, T2 / Pooled, T3 = the handoff's T2 / T3 sessions; Pooled, overall = both",
         _n_ses["Pooled, T2"] == _n_handoff["T2"] and _n_ses["Pooled, T3"] == _n_handoff["T3"]
         and _n_ses["Pooled, overall"] == _n_handoff["T2"] + _n_handoff["T3"], str(_n_ses))
_best_corr = pd.read_csv(LVL_OUTS / f"{LVL1_SWEEP}_report_best_corr.csv").set_index("method")
cal_c = []
for m in METHODS1:
    _s = pd.read_csv(H1_SWEEP_DIR / f"sessions_{_best_corr.loc[m, 'cfg_file']}")
    cal_c.append(_s[_s.method == m])
cal_c = pd.concat(cal_c, ignore_index=True)
WE = session_frame(cal_c, None).set_index(["id", "session", "method"])["corr"].unstack("method").fillna(0.0)
_rec = 100 * WE.mean()
h1_check("T2+T3 calibration, Corr%-optimal: reconstructed Corr% = sweep avg_corr_sig% at each detector's best-Corr% config",
         np.allclose(_rec[METHODS1].values, _best_corr.loc[METHODS1, "avg_corr_sig%"].values, atol=1e-9),
         f"max |diff| {np.abs(_rec[METHODS1].values - _best_corr.loc[METHODS1, 'avg_corr_sig%'].values).max():.1e} pp")


In [ ]:
# [notebook only] H1: DESCRIPTIVES
# =============================================================================
# Overall, selected detector vs the remaining detectors, per detector. Units: session (detector x session values pooled
# for multi-detector rows). eta^2_H over segmented sessions only; Corr% = 100 x mean session score over all sessions;
# blocks over all sessions (1 = unsplit). M (SD) with n - 1. † = the selected detector.
mark = lambda m: m + (" †" if m == WINNER_LVL1 else "")


def h1_desc_rows(ds, label, cols):
    E, S, NB = (MAT1[ds][v][cols] for v in ("eta", "corr", "n_blocks"))
    nb = NB.to_numpy(float).ravel(); seg = nb >= 2
    return {"Dataset": ds, "Detector(s)": label, "Units (session × detector)": int(NB.notna().to_numpy().sum()),
            "Segmented n (%)": f"{int(seg.sum())} ({100 * seg.sum() / np.isfinite(nb).sum():.1f})",
            "η²_H Mdn [Q1, Q3] (sMAD)": summ_str(E.to_numpy().ravel()),
            "η²_H M (SD)": f"{np.nanmean(E.to_numpy()):.3f} ({np.nanstd(E.to_numpy(), ddof=1):.3f})",
            "Corr% (all sessions)": 100 * np.nanmean(S.to_numpy()),
            "Corr% session score Mdn [Q1, Q3] (sMAD)": summ_str(S.to_numpy().ravel(), 2),
            "Corr% M (SD)": f"{100 * np.nanmean(S.to_numpy()):.1f} ({100 * np.nanstd(S.to_numpy(), ddof=1):.1f})",
            "Blocks per session Mdn [Q1, Q3] (sMAD)": summ_str(nb, 1),
            "Blocks per session M (SD)": f"{np.nanmean(nb):.2f} ({np.nanstd(nb, ddof=1):.2f})"}


H1_DESC = {"overall": [], "winner": [], "method": []}
for ds in DS:
    allm = list(MAT1[ds]["eta"].columns); rest = [m for m in allm if m != WINNER_LVL1]
    H1_DESC["overall"].append(h1_desc_rows(ds, f"all {len(allm)} detectors", allm))
    H1_DESC["winner"] += [h1_desc_rows(ds, mark(WINNER_LVL1), [WINNER_LVL1]),
                          h1_desc_rows(ds, f"remaining {len(rest)} detectors", rest)]
    H1_DESC["method"] += [h1_desc_rows(ds, mark(m), [m]) for m in sorted(allm, key=lambda m: -MAT1[ds]["eta"][m].mean())]
h1_export("desc_overall", pd.DataFrame(H1_DESC["overall"]))
h1_export("desc_winner", pd.DataFrame(H1_DESC["winner"]))
h1_export("desc_detector", pd.DataFrame(H1_DESC["method"]))

In [ ]:
# [notebook only] H1: TESTS
# =============================================================================
# Four tests, each of every pair of the detectors (120 for 16): eta^2_H and Corr% in T2+T3 calibration, each detector at
# its own eta^2_H-optimal and Corr%-optimal sweep configuration respectively, and eta^2_H and Corr% in T3 only, every
# detector at the selected configuration. Per pair: two-sided participant-cluster paired bootstrap of the difference in
# means. Participants are resampled with replacement with all their sessions (multinomial weights; one set of H1_BOOT
# resamples per dataset, shared by every detector and both criteria), and each detector's mean is over its own sessions
# with a value (eta^2_H: the sessions it split; Corr%: all sessions). Detectors leave different sessions unsplit, so
# each mean keeps every session it has while the shared resamples keep the pairing of the same participants. Corr%
# enters as each session's number of significant features (0-3), so equal resampled sums tie exactly (a tie counts in
# both tails), and is shown in percentage points (x 100/3). 95% percentile interval; p = 2 x the smaller tail,
# (count + 1) / (B + 1), at most 1; BH-FDR over the pairs of each test. A = the detector with the higher mean.
import itertools
import zlib

H1_BOOT = 10_000
H1_TESTS = [   # dataset, criterion, configurations, session x detector values, scale shown
    ("T2+T3 calibration", "η²_H", "each detector's η²_H-optimal configuration (T2+T3 sweep)", MAT1["T2+T3 calibration"]["eta"], 1.0),
    ("T2+T3 calibration", "Corr%", "each detector's Corr%-optimal configuration (T2+T3 sweep)", np.rint(3 * WE), 100 / 3),
    ("T3 only", "η²_H", "the selected configuration (LVL_1 handoff)", MAT1["T3 only"]["eta"], 1.0),
    ("T3 only", "Corr%", "the selected configuration (LVL_1 handoff)", np.rint(3 * MAT1["T3 only"]["corr"]), 100 / 3)]


def boot_weights(ds):
    """The dataset's participants (sorted ids) and their H1_BOOT x P multinomial resampling weights."""
    ids = np.array(sorted(MAT1[ds]["eta"].index.get_level_values("id").unique()))
    rng = np.random.default_rng([26, zlib.crc32(f"H1 all pairs, {ds}".encode())])
    return ids, rng.multinomial(len(ids), np.full(len(ids), 1 / len(ids)), size=H1_BOOT).astype(float)


def pairwise_bootstrap(X, ids, W, scale):
    """Every pair of the columns of X (session x detector, NaN = no value): the difference in the detectors' means (each
    over its own sessions), its 95% percentile interval and the two-sided bootstrap p, shown x scale; A = the higher
    mean. Returns the pairs and the means, best first."""
    pidx = np.searchsorted(ids, X.index.get_level_values("id"))
    v = X.to_numpy(float); ok = np.isfinite(v)
    S, N = np.zeros((len(ids), v.shape[1])), np.zeros((len(ids), v.shape[1]))
    np.add.at(S, pidx, np.where(ok, v, 0.0)); np.add.at(N, pidx, ok.astype(float))
    obs = (scale * pd.Series(S.sum(0) / N.sum(0), index=X.columns)).sort_values(ascending=False, kind="stable")
    with np.errstate(invalid="ignore", divide="ignore"):
        star = pd.DataFrame((W @ S) / (W @ N), columns=X.columns)      # H1_BOOT x detectors
    rows = []
    for a, b in itertools.combinations(obs.index, 2):
        d = (star[a] - star[b]).to_numpy(); d = d[np.isfinite(d)]
        rows.append({"A": a, "B": b, "Mean A": obs[a], "Mean B": obs[b], "Difference (A − B)": obs[a] - obs[b],
                     "CI 2.5%": scale * np.percentile(d, 2.5), "CI 97.5%": scale * np.percentile(d, 97.5),
                     "p": min(1.0, 2 * min(((d <= 0).sum() + 1) / (len(d) + 1), ((d >= 0).sum() + 1) / (len(d) + 1))),
                     "Replicates": len(d)})
    t = pd.DataFrame(rows)
    t["q"] = bh(t.p.to_numpy())
    return t, obs


H1W = {ds: boot_weights(ds) for ds in dict.fromkeys(ds for ds, *_ in H1_TESTS)}
H1T, H1M = [], {}            # the pairs; each test's detector means, best first
for ds, lab, cfg, X, sc in H1_TESTS:
    h1_check(f"{ds}, {lab}: one row per session of the dataset", X.index.equals(MAT1[ds]["eta"].index))
    _t, H1M[ds, lab] = pairwise_bootstrap(X, *H1W[ds], sc)
    H1T.append(_t.assign(Dataset=ds, Criterion=lab, Configurations=cfg))
H1T = pd.concat(H1T, ignore_index=True)[["Dataset", "Criterion", "Configurations", "A", "B", "Mean A", "Mean B",
                                         "Difference (A − B)", "CI 2.5%", "CI 97.5%", "p", "q", "Replicates"]]
h1_export("tests", H1T)


In [ ]:
# [notebook only] H1: SUMMARY ACROSS THE DATASETS AND METHOD-LEVEL ASSOCIATION
# =============================================================================
# Per test (H1: TESTS) and detector: rank by mean, mean and SD over its own sessions (in the test's units), the number
# of detectors it is significantly above and below (q <= ALPHA), those it is not separable from (q > ALPHA), and its
# group: best, not separable from the best, or lower (the calibration figure's colours). Per test: the numerically
# best detector, the selected detector's rank, the detectors not separable from the best and from the selected
# detector, and those significantly above or below it. Method level: the detectors' mean eta^2_H vs mean Corr%
# (Pearson), and the Mahalanobis distance of (mean eta^2_H, mean blocks per session) from the detectors' centroid
# (sample covariance; with n detectors D^2 cannot exceed (n - 1)^2 / n, so it is descriptive).
def standing(t, m):
    """Detector m in the pairs t of one test: each other detector, m's mean minus the other's, and the pair's q."""
    s = t[(t.A == m) | (t.B == m)]
    return pd.DataFrame({"other": np.where(s.A == m, s.B, s.A), "q": s.q.to_numpy(),
                         "diff": np.where(s.A == m, s["Difference (A − B)"], -s["Difference (A − B)"])})


H1K, H1R = [], []
for ds, lab, cfg, X, sc in H1_TESTS:
    t = H1T[(H1T.Dataset == ds) & (H1T.Criterion == lab)]
    mu = H1M[ds, lab]
    best = mu.index[0]
    for rank, m in enumerate(mu.index, 1):
        s = standing(t, m)
        H1K.append({"Dataset": ds, "Criterion": lab, "Configurations": cfg, "Rank": rank, "Detector": m, "Mean": mu[m],
                    "SD": sc * X[m].std(ddof=1), "Sessions": int(X[m].notna().sum()),
                    "Significantly above (n)": int(((s.q <= ALPHA) & (s["diff"] > 0)).sum()),
                    "Significantly below (n)": int(((s.q <= ALPHA) & (s["diff"] < 0)).sum()),
                    "Not separable from (q > .05)": ", ".join(s.other[s.q > ALPHA]) or "—",
                    "Group": ("best" if m == best else "not separable from the best"
                              if best in set(s.other[s.q > ALPHA]) else "lower")})
    sb, sw = standing(t, best), standing(t, WINNER_LVL1)
    H1R.append({"Dataset": ds, "Criterion": lab, "Configurations": cfg, "Participants": len(H1W[ds][0]),
                "Sessions": len(X), "Numerical best": mark(best), "Best mean": mu.iloc[0],
                f"{WINNER_LVL1} † mean": mu[WINNER_LVL1], "† rank": int(list(mu.index).index(WINNER_LVL1) + 1),
                "Not separable from the best (q > .05)": ", ".join(sb.other[sb.q > ALPHA]) or "—",
                "Not separable from † (q > .05)": ", ".join(sw.other[sw.q > ALPHA]) or "—",
                "Significantly below †": int(((sw.q <= ALPHA) & (sw["diff"] > 0)).sum()),
                "Significantly above † (q ≤ .05)": ", ".join(sw.other[(sw.q <= ALPHA) & (sw["diff"] < 0)]) or "—"})
h1_export("ranks", pd.DataFrame(H1K))
h1_export("summary", pd.DataFrame(H1R))

rows, _d2 = [], {}
for ds in DS:
    M_ = pd.DataFrame({"eta": MAT1[ds]["eta"].mean(), "corr": MAT1[ds]["corr"].mean(), "blocks": MAT1[ds]["n_blocks"].mean()})
    r_, p_ = stats.pearsonr(M_["eta"], M_["corr"])
    X_ = M_[["eta", "blocks"]].to_numpy(float); dX = X_ - X_.mean(axis=0)
    D2 = np.einsum("ij,jk,ik->i", dX, np.linalg.inv(np.cov(X_.T)), dX)
    n_ = len(M_)
    rows.append({"Dataset": ds, "Detectors": n_, "r (mean η²_H, mean Corr%)": r_, "df": n_ - 2, "p": p_,
                 "Max D² (mean η²_H, mean blocks)": D2.max(), "Detector with max D²": mark(M_.index[int(np.argmax(D2))]),
                 "χ²(2) critical value, p = .001": stats.chi2.ppf(0.999, 2), "Largest attainable D²": (n_ - 1) ** 2 / n_})
    _d2[ds] = pd.Series(D2, index=[mark(m) for m in M_.index])
h1_export("method_association", pd.DataFrame(rows))
h1_export("mahalanobis", pd.DataFrame(_d2).rename_axis("Detector").reset_index())

In [ ]:
# [notebook only] H1: FIGURE -- EFFECT SIZE VS CORRECTED SIGNIFICANCE AT EACH DETECTOR'S OPTIMAL CONFIGURATION (T2 + T3)
# =============================================================================
# Each detector at the configuration that maximises its mean effect size (top; the LVL1 sweep report's best_eta) and its
# Corr% (bottom; best_corr), mean effect size against avg_corr_sig%. Colours from this notebook's T2+T3 calibration tests
# (H1: TESTS, every pair; the groups of H1: SUMMARY ACROSS THE DATASETS): top, the eta^2_H test; bottom, the Corr% test at the
# Corr%-optimal configurations; red = the test's best detector, orange = not separable from it (BH q > .05), blue =
# significantly lower. Draws only.
import matplotlib.transforms as mtransforms
from matplotlib.lines import Line2D

EFF, CORR = "avg_effsize", "avg_corr_sig%"
EFF_LABEL_FIG = "avg " + (r"$\eta^2_H$" if str(LVL_1_CONFIG["validate_test"]).strip("'\"").lower() == "kruskal" else "ω²")
SEQ, INK, MUTED = "#2a78d6", "#0b0b0b", "#52514e"
_RC_SWEEP = {"figure.dpi": 110, "savefig.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
             "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK, "text.color": INK, "xtick.color": MUTED,
             "ytick.color": MUTED, "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.6,
             "axes.axisbelow": True, "legend.frameon": False, "font.size": 9}
chk = pd.read_csv(LVL_OUTS / f"{LVL1_SWEEP}_report_best_eta.csv").set_index("method").sort_values(EFF, ascending=False)
chk_sig = pd.read_csv(LVL_OUTS / f"{LVL1_SWEEP}_report_best_corr.csv").set_index("method").sort_values(CORR, ascending=False)
_grp_rows = pd.DataFrame(H1K)
_grp_rows = _grp_rows[(_grp_rows.Dataset == "T2+T3 calibration") & (_grp_rows.Group != "lower")]
H1_GROUP = _grp_rows.loc[_grp_rows.Criterion == "η²_H", "Detector"].tolist()      # best first
E_GROUP = _grp_rows.loc[_grp_rows.Criterion == "Corr%", "Detector"].tolist()
WINNER, E_WINNER = H1_GROUP[0], E_GROUP[0]


def _bbox_overlap(a, b):
    dx = min(a.x1, b.x1) - max(a.x0, b.x0)
    dy = min(a.y1, b.y1) - max(a.y0, b.y0)
    return dx * dy if dx > 0 and dy > 0 else 0.0


def place_labels(ax, xs, ys, labels, fontsize=7, color="#666666", inside=False, obstacles=()):
    """Annotate without overlaps: try candidate offsets, keep the first that
    collides with no marker, no label already placed and no artist in `obstacles`
    (e.g. a legend). Falls back to the least-overlapping candidate when every position
    collides. inside=True also counts any part of a label outside the axes as overlap."""
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    cands = [(6, 3, "left", "bottom"), (6, -4, "left", "top"),
             (-6, 3, "right", "bottom"), (-6, -4, "right", "top"),
             (0, 9, "center", "bottom"), (0, -9, "center", "top"),
             (12, 3, "left", "bottom"), (-12, 3, "right", "bottom")]
    taken = [o.get_window_extent(rend) for o in obstacles]
    _ax_bb = ax.get_window_extent(rend)
    for x, y in zip(xs, ys):                      # markers are obstacles too
        px, py = ax.transData.transform((x, y))
        taken.append(mtransforms.Bbox.from_bounds(px - 5, py - 5, 10, 10))
    for i in np.argsort(ys)[::-1]:                # densest region first
        best = None
        for dx, dy, ha, va in cands:
            t = ax.annotate(labels[i], (xs[i], ys[i]), textcoords="offset points",
                            xytext=(dx, dy), fontsize=fontsize, color=color,
                            ha=ha, va=va, annotation_clip=False)
            bb = t.get_window_extent(rend).expanded(1.06, 1.25)
            ov = sum(_bbox_overlap(bb, o) for o in taken)
            if inside:                                # area of the label outside the axes
                ov += bb.width * bb.height - _bbox_overlap(bb, _ax_bb)
            if best is None or ov < best[1]:
                if best is not None:
                    best[0].remove()
                best = (t, ov, bb)
            else:
                t.remove()
            if ov == 0:
                break
        taken.append(best[2])


PANELS = [(chk, f"optimized for {EFF_LABEL_FIG}"),
          (chk_sig, f"optimized for {CORR}")]

with plt.rc_context(_RC_SWEEP):
    fig, axes = plt.subplots(
        2, 1,
        figsize=(7.5, 10.0),
        sharex=True,
        sharey=True
    )

    # red = best, orange = not separable from it (BH): top panel from the effect-size test,
    # bottom from the Corr% test -- each panel's own metric at each method's own best config
    _GROUPS = [(WINNER, H1_GROUP), (E_WINNER, E_GROUP)]
    for pos, (ax, (tbl, sub)) in enumerate(zip(axes, PANELS)):
        _win, _grp = _GROUPS[pos]
        ax.scatter(
            tbl[CORR], tbl[EFF],
            s=44,
            color=["#d62728" if m == _win else "#f0932b" if m in _grp else SEQ for m in tbl.index],
            zorder=3,
            edgecolor="white",
            linewidth=1.4
        )

        ax.margins(x=0.14, y=0.10)

        place_labels(
            ax,
            tbl[CORR].values,
            tbl[EFF].values,
            list(tbl.index),
            color=MUTED
        )

        ax.set_title(sub, fontweight="bold", loc="left")
        if pos == 0:   # key once, in the top panel
            ax.legend(handles=[Line2D([], [], ls="", marker="o", color="#d62728", label="highest; sig. > blue (BH)"),
                               Line2D([], [], ls="", marker="o", color="#f0932b", label="lower, n.s. vs red (BH)"),
                               Line2D([], [], ls="", marker="o", color=SEQ, label="sig. lower than red (BH)")],
                      fontsize=8, loc="lower right", frameon=False)

    # shared labels
    fig.supylabel(EFF_LABEL_FIG)

    axes[-1].set_xlabel(
        f"{CORR}   (significant / ALL sessions - the honest rate)"
    )

    fig.suptitle(
        f"{EFF_LABEL_FIG} vs corrected significance ratio ({CORR}) - methods at optimal config",
        fontweight="bold",
        x=0.01,
        ha="left"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


In [ ]:
# [notebook only] DESCRIPTIVES: INPUTS AND PARTICIPANT METRICS
# =============================================================================
# The behavioural features by condition and session (the final report's Table 0.3), the age check behind the age
# adjustment, and the data of the supplementary feature figures, exported as <handoff prefix>_report_<table>.csv.
# Inputs: this handoff (written by [cell 25]) and the LVL_0 handoff it was built from, with LVL_0's intake gates from
# that handoff's run log. Per participant-session:
#   slide_var   rolling variance of the raw RT (seconds) along each scale's stimulus sequence, with LVL_1's window
#               (derivative_filter_size, rolling_center, min_periods), averaged over the session; the "Adj RT Var
#               (win)" flavour, not the cross-scale within-stimulus variance
#   entropy     the session mean of LVL_1's rolling normalised response entropy, per scale and scale-averaged
#   blocks      mean block count across detectors and its SD across detectors
TABLE_ALPHA   = 0.05      # FDR level
TABLE_RT_COL  = "rt_sec"  # the sliding-window variance is computed in SECONDS
L1_PREFIX = LVL_OUTS / f"lv1_out_{LVL1_EXPORT_TS}"
TAB_LV0 = Path(LVL0_HANDOFF)
TABLE_OUT_DIR, TABLE_STEM = LVL_OUTS, f"{L1_PREFIX.name}_report"   # outputs: <handoff prefix>_report_*
_l1cfg = json.load(open(f"{L1_PREFIX}_meta.json"))["lvl1_config"]
SLIDE_WINDOW      = int(_l1cfg["derivative_filter_size"])
SLIDE_CENTER      = str(_l1cfg["rolling_center"]) == "True"
SLIDE_MIN_PERIODS = int(_l1cfg["min_periods"])
_lv0_par = LVL0_RECORD["parameters"]
PR_STIM_DROP, MIN_RT_THRESH, MAX_RT_THRESH = (_lv0_par[k] for k in ("pr_stim_drop", "min_rt_thresh", "max_rt_thresh"))

# ---- LVL_0 handoff: raw RT, after LVL_0's intake gates (the handoff is UNFILTERED) ------------
lv0 = pd.read_csv(TAB_LV0, usecols=["id", "condition", "session", "scale", "stim_order", TABLE_RT_COL])
_n0 = len(lv0)
clean = lv0[lv0["stim_order"] > PR_STIM_DROP].copy()
if MIN_RT_THRESH > 0:
    clean = clean[clean[TABLE_RT_COL] >= MIN_RT_THRESH]
if MAX_RT_THRESH > 0:
    clean = clean[clean[TABLE_RT_COL] <= MAX_RT_THRESH]
clean = clean.reset_index(drop=True)
SCALES = sorted(clean["scale"].dropna().unique())

# ---- LVL_1 handoff: entropy (features) + block counts (results) ------------
_ent_cols = ["rating_entropy_norm"] + [f"rating_entropy_norm_{s}" for s in SCALES]
l1_feat = pd.read_csv(f"{L1_PREFIX}_features.csv",
                      usecols=lambda c: c in set(["id", "session", "condition", "stim_order"] + _ent_cols))
l1_res = pd.read_csv(f"{L1_PREFIX}_results.csv")
print(f"LVL_1 handoff : {L1_PREFIX.name}  (window {SLIDE_WINDOW}, centred {SLIDE_CENTER}, min_periods {SLIDE_MIN_PERIODS})")
print(f"LVL_0 handoff : {TAB_LV0.name}  (pr_stim_drop={PR_STIM_DROP}, min_rt_thresh={MIN_RT_THRESH}, "
      f"max_rt_thresh={MAX_RT_THRESH}); rows {_n0:,} -> {len(clean):,} after the intake gates")

g_all = ["id", "condition", "session"]
_c = clean.sort_values(g_all + ["scale", "stim_order"])
_c["_slide"] = (_c.groupby(g_all + ["scale"])[TABLE_RT_COL]
                  .transform(lambda s: s.rolling(SLIDE_WINDOW, center=SLIDE_CENTER,
                                                 min_periods=SLIDE_MIN_PERIODS).var()))
slide_w = (_c.groupby(g_all + ["scale"])["_slide"].mean().reset_index()
             .pivot_table(index=g_all, columns="scale", values="_slide"))
slide_w.columns = [f"slide_var::{c}" for c in slide_w.columns]
slide_w["slide_var::ALL"] = slide_w.mean(axis=1)      # scale-averaged
_ent_key = ["id", "session"]
ent = l1_feat.groupby(_ent_key)[[c for c in _ent_cols if c in l1_feat.columns]].mean()
ent.columns = [("entropy::ALL" if c == "rating_entropy_norm"
                else f"entropy::{c.replace('rating_entropy_norm_', '')}") for c in ent.columns]
blk = l1_res.groupby(_ent_key)["n_blocks"].agg(**{"blocks::mean": "mean", "blocks::sd": "std"})
_units = clean.groupby(g_all).size().rename("_n_trials")
pmet = (pd.concat([_units, slide_w], axis=1).drop(columns="_n_trials")
          .reset_index()
          .merge(ent.reset_index(), on=_ent_key, how="left")
          .merge(blk.reset_index(), on=_ent_key, how="left"))
pmet.to_csv(TABLE_OUT_DIR / f"{TABLE_STEM}_participant_metrics.csv", index=False)
print(f"participant-level matrix: {pmet.shape[0]} id x session rows -> {TABLE_STEM}_participant_metrics.csv")

CELLS = {                                   # name -> (condition filter, session filter)
    "T2-overall":   ("T2", None),
    "T3-overall":   ("T3", None),
    "T2-S1":        ("T2", 1),
    "T2-S2":        ("T2", 2),
    "T3-S1":        ("T3", 1),
    "T3-S2":        ("T3", 2),
    "S1-overall":   (None, 1),
    "S2-overall":   (None, 2),
    "task-overall": (None, None),
}


def cell_mask(df, cond, ses):
    m = pd.Series(True, index=df.index)
    if cond is not None:
        m &= df["condition"] == cond
    if ses is not None:
        m &= df["session"] == ses
    return m


def bh_fdr(pvals):
    """Benjamini-Hochberg q-values; NaNs pass through untouched."""
    p = np.asarray(pvals, dtype=float)
    ok = ~np.isnan(p)
    q = np.full_like(p, np.nan)
    if ok.sum() == 0:
        return q
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    qv = np.empty_like(ranked)
    qv[order] = np.clip(ranked, 0, 1)
    q[ok] = qv
    return q

In [ ]:
# [notebook only] DESCRIPTIVES: AGE CHECK (WHY THE DRIFT RESIDUALS ARE AGE-ADJUSTED)
# =============================================================================
# The three stimulus-level features, per scale and as the stimulus composite (mean over the five scales), BEFORE the
# age adjustment, against age:
#   AdjRT  - drift-adjusted log RT before the age adjustment: the LVL_1 per-scale residual ({scale}_residual) plus that
#            participant x scale's age term, recovered from the LVL_0 handoff as drift_residual_<tag>_<base> -
#            drift_residual_<tag>_<base>_age (a constant per participant x scale)
#   AdjVar - rolling variance of each scale's residual (LVL_1's window); the age term is constant within a participant x
#            scale, so this is the same before and after the age adjustment
#   Ent    - normalised rolling response entropy per scale (rating_entropy_norm_<scale>), untouched by the adjustment
# Unit: one value per participant (mean over both sessions and all stimuli); T2 and T3 pooled, as LVL_0 pools them to
# fit the age slopes. Tests: OLS / Pearson r on age, and Mann-Whitney U adolescents (< 18) vs adults (rank-biserial r);
# BH-FDR across the five scales within each family; the composite rows are supplementary (uncorrected p).
from scipy.stats import linregress, mannwhitneyu, t as _tdist

AGE_ADULT_MIN = 18                  # adolescents: age < 18; adults: age >= 18
_RT_TAG = {"raw": "raw", "log": "log", "boxcox": "bc"}
_tag = _RT_TAG[str(_l1cfg["feature_rt_transform"]).strip("'\"")]
_base = "per_ses" if str(_l1cfg["ses_wise_drift"]) == "True" else "overall"
_res_pre, _res_age = f"drift_residual_{_tag}_{_base}", f"drift_residual_{_tag}_{_base}_age"
if str(_l1cfg["age_adjust_drift_residuals"]) != "True":
    print("note: the LVL_1 handoff was built WITHOUT age-adjusted residuals; the age term is 0")
_a0 = pd.read_csv(TAB_LV0, usecols=["id", "scale", "age", _res_pre, _res_age])
_a0["_term"] = _a0[_res_pre] - _a0[_res_age]
_term = _a0.groupby(["id", "scale"])["_term"].agg(["mean", "std"])
print(f"age term = {_res_pre} - {_res_age}: constant within participant x scale "
      f"(max within-sd {_term['std'].max():.2g})")
AGE_TERM = _term["mean"].unstack("scale")                                 # id x scale
AGE = _a0.groupby("id")["age"].first()
_fcols = (["id", "session", "condition", "stim_order", "aggregated_residual", "local_residual_var",
           "rating_entropy_norm"] + [f"{s}_residual" for s in SCALES]
          + [f"rating_entropy_norm_{s}" for s in SCALES])
_f = (pd.read_csv(f"{L1_PREFIX}_features.csv", usecols=_fcols)
        .sort_values(["id", "session", "stim_order"]).reset_index(drop=True))
_stim = _f[["id", "session", "condition"]].copy()
for s in SCALES:
    _post = _f[f"{s}_residual"]
    _stim[f"AdjRT::{s}"] = _post + _f["id"].map(AGE_TERM[s]).fillna(0.0)
    _stim[f"AdjVar::{s}"] = (_post.groupby([_f["id"], _f["session"]])
                               .transform(lambda v: v.rolling(SLIDE_WINDOW, center=SLIDE_CENTER,
                                                              min_periods=SLIDE_MIN_PERIODS).var()))
    _stim[f"Ent::{s}"] = _f[f"rating_entropy_norm_{s}"]
FAMILIES = ["AdjRT", "AdjVar", "Ent"]
for fam in FAMILIES:
    _stim[f"{fam}::comp."] = _stim[[f"{fam}::{s}" for s in SCALES]].mean(axis=1)
_chk = {"AdjRT (age-adjusted)": (_f[[f"{s}_residual" for s in SCALES]].mean(axis=1), _f["aggregated_residual"]),
        "AdjVar": (_stim["AdjVar::comp."], _f["local_residual_var"]),
        "Ent": (_stim["Ent::comp."], _f["rating_entropy_norm"])}
for _k, (_a, _b) in _chk.items():
    _ok = _a.notna() & _b.notna()
    print(f"composite check {_k:22s}: r = {np.corrcoef(_a[_ok], _b[_ok])[0, 1]:.5f}, "
          f"median |diff| = {np.median(np.abs(_a[_ok] - _b[_ok])):.2g}  ({_ok.sum():,} stimuli)")
AGE_P = (_stim.drop(columns=["session", "condition"]).groupby("id").mean()
              .join(AGE.rename("age"), how="inner"))
AGE_P["group"] = np.where(AGE_P["age"] < AGE_ADULT_MIN, "adolescent", "adult")
ROWS = list(SCALES) + ["comp."]
print(f"{len(AGE_P)} participants ({(AGE_P.group == 'adolescent').sum()} adolescents, "
      f"{(AGE_P.group == 'adult').sum()} adults); age {AGE_P.age.min():.0f}-{AGE_P.age.max():.0f}")
_cont = []
for fam in FAMILIES:
    for s in ROWS:
        d = AGE_P[["age", f"{fam}::{s}"]].dropna()
        lr = linregress(d["age"], d[f"{fam}::{s}"])
        _q = _tdist.ppf(0.975, len(d) - 2)
        _cont.append({"family": fam, "scale": s, "n": len(d), "intercept": lr.intercept,
                      "slope": lr.slope, "slope_lo": lr.slope - _q * lr.stderr,
                      "slope_hi": lr.slope + _q * lr.stderr, "r": lr.rvalue, "p": lr.pvalue})
AGE_CONT = pd.DataFrame(_cont)
_grp = []
for fam in FAMILIES:
    for s in ROWS:
        d = AGE_P[["group", f"{fam}::{s}"]].dropna()
        a = d.loc[d.group == "adolescent", f"{fam}::{s}"]
        b = d.loc[d.group == "adult", f"{fam}::{s}"]
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        _grp.append({"family": fam, "scale": s, "n_adolescent": len(a), "n_adult": len(b),
                     "median_adolescent": a.median(), "median_adult": b.median(),
                     # MAD = median absolute deviation from the group median (unscaled)
                     "mad_adolescent": (a - a.median()).abs().median(),
                     "mad_adult": (b - b.median()).abs().median(), "U": u,
                     # rank-biserial r = P(adolescent > adult) - P(adolescent < adult)
                     "r_rb": 2 * u / (len(a) * len(b)) - 1, "p": p})
AGE_GRP = pd.DataFrame(_grp)
for _t in (AGE_CONT, AGE_GRP):
    _t["p_FDR"] = np.nan
    for fam in FAMILIES:
        _m = (_t.family == fam) & (_t.scale != "comp.")
        _t.loc[_m, "p_FDR"] = multipletests(_t.loc[_m, "p"], method="fdr_bh")[1]
    _t["row"] = np.where(_t.scale == "comp.", "supplementary (not in FDR family)", "")
pd.set_option("display.width", 200)
print("\nCONTINUOUS AGE (OLS slope per year of age; Pearson r; BH-FDR over the five scales per family)")
print(AGE_CONT[["family", "scale", "n", "slope", "r", "p", "p_FDR", "row"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("\nADOLESCENTS (< 18) vs ADULTS (Mann-Whitney U, two-sided; r_rb > 0 = adolescents higher)")
print(AGE_GRP[["family", "scale", "n_adolescent", "n_adult", "median_adolescent", "median_adult",
               "mad_adolescent", "mad_adult", "U", "r_rb", "p", "p_FDR", "row"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4g}"))
AGE_CONT.to_csv(TABLE_OUT_DIR / f"{TABLE_STEM}_age_continuous.csv", index=False)
AGE_GRP.to_csv(TABLE_OUT_DIR / f"{TABLE_STEM}_age_groups.csv", index=False)
AGE_P.reset_index().to_csv(TABLE_OUT_DIR / f"{TABLE_STEM}_age_participants.csv", index=False)
print(f"saved -> {TABLE_STEM}_age_continuous.csv, _age_groups.csv, _age_participants.csv")

In [ ]:
# [notebook only] DESCRIPTIVES: SESSION FEATURES AND THE NONPARAMETRIC TABLE
# =============================================================================
# Per participant-session x scale (the supplementary feature-distribution figure; the table below):
#   Session-level RT variance   RT variance per scale over the whole session (s^2)
#   Sliding-window RT variance  the session mean of the rolling RT variance (slide_var above)
#   Session-level entropy       normalised response entropy per scale over the whole session (in-range responses)
#   Sliding-window entropy      the session mean of LVL_1's rolling normalised entropy (entropy above)
# The table: RT (each participant-session's mean RT) and the four features, per scale and over all scales ("All
# scales": RT over all trials of the session; the others the mean of the five scales), in the nine cells; one value per
# participant (a two-session cell averages the participant's sessions first), as median (IQR) across participants.
# T2 vs T3: Mann-Whitney U; S1 vs S2 (T2, T3): Wilcoxon signed-rank on the ids with both sessions; BH-FDR across all
# tests of one feature (6 rows x 3 comparisons).
import ast
_K = {s: hi - lo + 1 for s, (lo, hi) in ast.literal_eval(str(_l1cfg["scale_ranges"])).items()}
_r = pd.read_csv(TAB_LV0, usecols=["id", "session", "scale", "stim_order", TABLE_RT_COL, "response"])
_r = _r[_r["stim_order"] > PR_STIM_DROP]
if MIN_RT_THRESH > 0:
    _r = _r[_r[TABLE_RT_COL] >= MIN_RT_THRESH]
if MAX_RT_THRESH > 0:
    _r = _r[_r[TABLE_RT_COL] <= MAX_RT_THRESH]


def _norm_entropy(resp, k):
    resp = resp[(resp >= 1) & (resp <= k)]                # in-range responses only
    if len(resp) == 0:
        return np.nan
    p = resp.value_counts(normalize=True).to_numpy()
    return float(-(p * np.log(p)).sum() / np.log(k))


def _long(wide, prefix):
    """wide metrics (prefix::scale) -> id, session, scale, value"""
    w = wide.reset_index() if "id" not in wide.columns else wide
    cols = [f"{prefix}::{s}" for s in SCALES if f"{prefix}::{s}" in w.columns]
    out = w[["id", "session"] + cols].melt(id_vars=["id", "session"], var_name="scale", value_name="value")
    out["scale"] = out["scale"].str.replace(f"{prefix}::", "", regex=False)
    return out


_gk = ["id", "session", "scale"]
SESS_FEATS = {
    "Session-level RT variance": clean.groupby(_gk)[TABLE_RT_COL].var().rename("value").reset_index(),
    "Sliding-window RT variance": _long(slide_w, "slide_var"),
    "Session-level entropy": (_r.groupby(_gk)["response"]
                              .apply(lambda v: _norm_entropy(v, _K[v.name[2]])).rename("value").reset_index()),
    "Sliding-window entropy": _long(ent, "entropy"),
}
pd.concat([d.assign(feature=k) for k, d in SESS_FEATS.items()], ignore_index=True)[
    ["feature", "id", "session", "scale", "value"]].to_csv(TABLE_OUT_DIR / f"{TABLE_STEM}_session_features.csv",
                                                           index=False)
NP_FEATS = ["RT (s)", "Session-level RT variance", "Sliding-window RT variance",
            "Session-level entropy", "Sliding-window entropy"]
_ids = pmet[["id", "condition", "session"]].drop_duplicates()


def _wide(long_df, prefix):
    w = long_df.pivot_table(index=["id", "session"], columns="scale", values="value")
    w["ALL"] = w[SCALES].mean(axis=1)
    w.columns = [f"{prefix}::{c}" for c in w.columns]
    return w.reset_index()


_rt = clean.groupby(["id", "session", "scale"])[TABLE_RT_COL].mean().rename("value").reset_index()
_rt_w = _wide(_rt, "RT (s)")
_rt_w["RT (s)::ALL"] = (clean.groupby(["id", "session"])[TABLE_RT_COL].mean()
                         .reindex(pd.MultiIndex.from_frame(_rt_w[["id", "session"]])).to_numpy())
NP_MET = _ids.merge(_rt_w, on=["id", "session"], how="left")
for _feat in NP_FEATS[1:]:
    NP_MET = NP_MET.merge(_wide(SESS_FEATS[_feat], _feat), on=["id", "session"], how="left")
_NP_ROWS = list(SCALES) + ["ALL"]


def _np_values(metric, cond, ses):
    sub = NP_MET[cell_mask(NP_MET, cond, ses)]
    return sub.groupby("id")[metric].mean().dropna()


def _med_iqr(v):
    if not len(v):
        return "-"
    q1, m, q3 = v.quantile([.25, .5, .75])
    d = 3 if abs(m) < 1 else 2
    return f"{m:.{d}f} ({q1:.{d}f}-{q3:.{d}f})"


NP_COMPARISONS = [("T2 vs T3", ("T2-overall", "T3-overall"), "mannwhitney"),
                  ("S1 vs S2 (T2)", ("T2-S1", "T2-S2"), "wilcoxon"),
                  ("S1 vs S2 (T3)", ("T3-S1", "T3-S2"), "wilcoxon")]
_recs = []
for feat in NP_FEATS:
    for s in _NP_ROWS:
        metric = f"{feat}::{s}"
        rec = {"feature": feat, "scale": "All scales" if s == "ALL" else s}
        for cname, (cond, ses) in CELLS.items():
            rec[cname] = _med_iqr(_np_values(metric, cond, ses))
        for name, (ca, cb), kind in NP_COMPARISONS:
            a, b = _np_values(metric, *CELLS[ca]), _np_values(metric, *CELLS[cb])
            if kind == "mannwhitney":
                stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
            else:
                _i = a.index.intersection(b.index)
                stat, p = stats.wilcoxon(a.loc[_i], b.loc[_i])
            rec[f"{name} stat"], rec[f"{name} p"] = float(stat), float(p)
        _recs.append(rec)
NP_TABLE = pd.DataFrame(_recs)
_pcols = [f"{n} p" for n, _, _ in NP_COMPARISONS]
for feat in NP_FEATS:                                   # BH-FDR over all tests of one feature
    _m = NP_TABLE["feature"] == feat
    _q = bh_fdr(NP_TABLE.loc[_m, _pcols].to_numpy().ravel()).reshape(_m.sum(), len(_pcols))
    for k, c in enumerate(_pcols):
        NP_TABLE.loc[_m, c.replace(" p", " p_FDR")] = _q[:, k]
NP_TABLE.to_csv(TABLE_OUT_DIR / f"{TABLE_STEM}_table_nonparametric.csv", index=False)
print(f"saved -> {TABLE_STEM}_session_features.csv, {TABLE_STEM}_table_nonparametric.csv")
display(NP_TABLE)